# ENVIRONMENT SETUP AND REPRODUCIBILITY

In [ ]:



import os
import random
import numpy as np
import torch

# Fixed seed for reproducible results

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Using device:", device)
print("Random seed set to:", SEED)

 DOWNLOAD ACDC DATASET

In [ ]:


from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="yscript/ACDC-PNG",
    repo_type="dataset",
    local_dir="/content/ACDC-PNG"
)

print("ACDC download complete!")

EXTRACT THE DATASET

In [ ]:

import zipfile

zip_path = "/content/ACDC-PNG/ACDC_png_ysc.zip"
extract_path = "/content/ACDC-PNG/ACDC_Extracted"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete!")
print("Extracted contents:", os.listdir(extract_path))

# DATASET PATHS

In [ ]:


base_path = "/content/ACDC-PNG/ACDC_Extracted/ACDC_png_ysc/ACDC_png_split_10percent"

train_image_folder = f"{base_path}/train-label/image"
train_mask_folder = f"{base_path}/train-label/mask"

val_image_folder = f"{base_path}/val/image"
val_mask_folder = f"{base_path}/val/mask"

test_image_folder = f"{base_path}/test/image"
test_mask_folder = f"{base_path}/test/mask"

print("Original dataset paths defined.")

# PATIENT-LEVEL 70/10/20 SPLIT

In [ ]:

# PATIENT-LEVEL 70/10/20 SPLIT


import random

source_pairs = {}

source_sets = [
    (train_image_folder, train_mask_folder),
    (val_image_folder, val_mask_folder),
    (test_image_folder, test_mask_folder)
]

for image_folder, mask_folder in source_sets:

    for filename in os.listdir(image_folder):

        if not filename.endswith(".png"):
            continue

        image_path = os.path.join(image_folder, filename)
        mask_name = filename.replace(".png", ".npy")
        mask_path = os.path.join(mask_folder, mask_name)

        if os.path.exists(mask_path):

            if filename not in source_pairs:
                source_pairs[filename] = (image_path, mask_path)

patients = sorted(
    list(set(filename.split("_")[0] for filename in source_pairs.keys()))
)

print("Total unique slices:", len(source_pairs))
print("Total unique patients:", len(patients))

random.shuffle(patients)

num_patients = len(patients)
num_train = int(num_patients * 0.7)
num_val = int(num_patients * 0.1)

train_patients = set(patients[:num_train])
val_patients = set(patients[num_train:num_train + num_val])
test_patients = set(patients[num_train + num_val:])

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Testing patients:", len(test_patients))

train_files = []
val_files = []
test_files = []

for filename, (image_path, mask_path) in source_pairs.items():

    patient_id = filename.split("_")[0]

    if patient_id in train_patients:
        train_files.append((image_path, mask_path))

    elif patient_id in val_patients:
        val_files.append((image_path, mask_path))

    elif patient_id in test_patients:
        test_files.append((image_path, mask_path))

total = len(train_files) + len(val_files) + len(test_files)

print("Training slices:", len(train_files), f"({len(train_files) / total * 100:.1f}%)")
print("Validation slices:", len(val_files), f"({len(val_files) / total * 100:.1f}%)")
print("Testing slices:", len(test_files), f"({len(test_files) / total * 100:.1f}%)")

## ACDC DATASET

In [ ]:
# ============================================================
# ACDC DATASET
# ============================================================

import numpy as np
from PIL import Image
from torch.utils.data import Dataset
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode


class ACDCDataset(Dataset):

    def __init__(self, file_list, image_size=256, augment=False):

        self.file_list = file_list
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.file_list)


    def __getitem__(self, idx):

        image_path, mask_path = self.file_list[idx]

        image = np.array(
            Image.open(image_path),
            dtype=np.float32
        )

        mask = np.load(mask_path).astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.5:
                dx = int(random.uniform(-0.05, 0.05) * self.image_size)
                dy = int(random.uniform(-0.05, 0.05) * self.image_size)
                scale = random.uniform(0.95, 1.05)
                image = TF.affine(image, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.affine(mask, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.3:
                mean_val = image.mean()
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp((image - mean_val) * factor + mean_val, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        image = image.repeat(3, 1, 1)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=image.dtype).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=image.dtype).view(3, 1, 1)
        image = (image - mean) / std

        mask = mask.squeeze(0)

        return image, mask


train_dataset = ACDCDataset(train_files, image_size=256, augment=True)
val_dataset = ACDCDataset(val_files, image_size=256, augment=False)
test_dataset = ACDCDataset(test_files, image_size=256, augment=False)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

# SHARED MODEL COMPONENTS

In [ ]:
# ============================================================
# SHARED MODEL COMPONENTS
# ============================================================

import torch.nn as nn
import torch.nn.functional as F


class ResidualConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm1 = nn.GroupNorm(8, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.norm2 = nn.GroupNorm(8, out_channels)

        self.activation = nn.GELU()

        if in_channels != out_channels:
            self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            self.skip = nn.Identity()


    def forward(self, x):

        residual = self.skip(x)

        x = self.conv1(x)
        x = self.norm1(x)
        x = self.activation(x)

        x = self.conv2(x)
        x = self.norm2(x)

        x = x + residual
        x = self.activation(x)

        return x


class TransformerBlock(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, num_tokens=256):
        super().__init__()

        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)


    def forward(self, x):

        x = x + self.pos_embedding[:, :x.size(1), :]
        x = self.transformer(x)
        x = self.norm(x)

        return x


class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.conv = ResidualConvBlock(in_channels + skip_channels, out_channels)


    def forward(self, x, skip):

        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)

        return x


print("Shared model components ready!")

# TRANSUNET MODEL

In [ ]:


class TransUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)

        self.transformer = TransformerBlock(
            embed_dim=512,
            num_heads=8,
            num_layers=4,
            num_tokens=256
        )

        self.transformer_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape

        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.transformer_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)

        output = self.output_layer(d1)

        return output


model = TransUNet(in_channels=3, out_channels=4).to(device)

print("TransUNet created!")
print("Total parameters:", sum(p.numel() for p in model.parameters()))

# MULTICLASS DICE LOSS

In [ ]:


class MulticlassDiceLoss(nn.Module):

    def __init__(self, num_classes=4, smooth=1e-6):
        super().__init__()

        self.num_classes = num_classes
        self.smooth = smooth


    def forward(self, logits, targets):

        probabilities = torch.softmax(logits, dim=1)

        total_dice_loss = 0.0
        foreground_classes = 0

        for class_idx in range(1, self.num_classes):

            pred_class = probabilities[:, class_idx]
            target_class = (targets == class_idx).float()

            intersection = (pred_class * target_class).sum()
            denominator = pred_class.sum() + target_class.sum()

            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)

            total_dice_loss += (1.0 - dice)
            foreground_classes += 1

        return total_dice_loss / foreground_classes


# ============================================================
# ACDC LOSS
# Weighted Focal + Dice, with extra emphasis on myocardium
# ============================================================

class ACDCLoss(nn.Module):

    def __init__(self, num_classes=4, gamma=2.0, smooth=1e-5):
        super().__init__()

        self.num_classes = num_classes
        self.gamma = gamma
        self.smooth = smooth

        # 0 = Background, 1 = RV, 2 = Myocardium, 3 = LV

        self.register_buffer(
            "class_weights",
            torch.tensor([0.20, 1.00, 1.50, 1.00], dtype=torch.float32)
        )

        self.dice_loss = MulticlassDiceLoss(num_classes=num_classes)


    def myocardium_dice_loss(self, logits, targets):

        probabilities = torch.softmax(logits, dim=1)

        myo_prediction = probabilities[:, 2]
        myo_target = (targets == 2).float()

        intersection = (myo_prediction * myo_target).sum(dim=(1, 2))
        prediction_sum = myo_prediction.sum(dim=(1, 2))
        target_sum = myo_target.sum(dim=(1, 2))

        dice = (2.0 * intersection + self.smooth) / (prediction_sum + target_sum + self.smooth)

        return 1.0 - dice.mean()


    def forward(self, logits, targets):

        ce_loss = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")

        pt = torch.exp(-ce_loss)
        focal_loss = (((1 - pt) ** self.gamma) * ce_loss).mean()

        dice_loss = self.dice_loss(logits, targets)

        myo_loss = self.myocardium_dice_loss(logits, targets)

        total_loss = (0.35 * focal_loss) + (1.00 * dice_loss) + (0.30 * myo_loss)

        return total_loss


loss_fn = ACDCLoss(num_classes=4).to(device)

print("ACDC Loss ready!")
print("Class weights: Background 0.20, RV 1.00, Myocardium 1.50, LV 1.00")

# DATALOADERS

In [ ]:
# ============================================================
# DATALOADERS
# ============================================================

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("DataLoaders created!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))


# ============================================================
# VALIDATION / TEST FUNCTION
# ============================================================

def evaluate(model, loader, loss_fn, device, num_classes=4):

    model.eval()

    total_loss = 0.0

    intersections = torch.zeros(num_classes - 1, device=device)
    pred_sums = torch.zeros(num_classes - 1, device=device)
    target_sums = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = loss_fn(outputs, masks)

            total_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            for class_id in range(1, num_classes):

                pred_class = (predictions == class_id).float()
                target_class = (masks == class_id).float()

                intersections[class_id - 1] += (pred_class * target_class).sum()
                pred_sums[class_id - 1] += pred_class.sum()
                target_sums[class_id - 1] += target_class.sum()

    dice_scores = (2.0 * intersections + 1e-5) / (pred_sums + target_sums + 1e-5)
    mean_dice = dice_scores.mean()
    average_loss = total_loss / len(loader.dataset)

    return average_loss, mean_dice, dice_scores


# ============================================================
# OPTIMIZER AND SCHEDULER
# ============================================================

import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer and scheduler ready!")

# TRAIN TRANSUNET

In [ ]:
# ============================================================
# TRAIN TRANSUNET
# ============================================================

num_epochs = 150

best_val_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_model_path = "/content/best_transunet.pth"

print("Starting TransUNet training...")
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print()

for epoch in range(num_epochs):

    model.train()

    total_train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(model, val_loader, loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_val_dice:

        best_val_dice = current_dice
        epochs_without_improvement = 0

        torch.save(model.state_dict(), best_model_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    scheduler.step(current_dice)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"RV: {val_dice[0].item():.4f} | "
        f"Myocardium: {val_dice[1].item():.4f} | "
        f"LV: {val_dice[2].item():.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_val_dice:.4f}")
print("Best model saved to:", best_model_path)

# TRANSUNET WITH PRETRAINED RESNET34 ENCODER

In [ ]:
# ============================================================
# RECREATE + TRAIN ACDC TRANSUNET (RESNET34) - FULL CODE
# ============================================================

resnet_model = ResNetTransUNet(out_channels=4, pretrained=True).to(device)

print("ResNet34 TransUNet created!")
print("Total parameters:", sum(p.numel() for p in resnet_model.parameters()))

resnet_loss_fn = ACDCLoss(num_classes=4).to(device)

encoder_params = list(resnet_model.stem.parameters()) + \
    list(resnet_model.layer1.parameters()) + \
    list(resnet_model.layer2.parameters()) + \
    list(resnet_model.layer3.parameters()) + \
    list(resnet_model.layer4.parameters())

new_params = list(resnet_model.bottleneck.parameters()) + \
    list(resnet_model.transformer.parameters()) + \
    list(resnet_model.transformer_refine.parameters()) + \
    list(resnet_model.decoder4.parameters()) + \
    list(resnet_model.decoder3.parameters()) + \
    list(resnet_model.decoder2.parameters()) + \
    list(resnet_model.decoder1.parameters()) + \
    list(resnet_model.final_refine.parameters()) + \
    list(resnet_model.output_layer.parameters())

resnet_optimizer = optim.AdamW([
    {"params": encoder_params, "lr": 2e-5},
    {"params": new_params, "lr": 1e-4}
], weight_decay=1e-4)

resnet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    resnet_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")
print("Encoder LR:", resnet_optimizer.param_groups[0]["lr"])
print("New layers LR:", resnet_optimizer.param_groups[1]["lr"])


# TRAIN RESNET34 TRANSUNET

num_epochs = 150

best_resnet_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_resnet_path = "/content/best_resnet_transunet.pth"

print()
print("Starting ResNet34 TransUNet training...")
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print()

for epoch in range(num_epochs):

    resnet_model.train()

    total_train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        resnet_optimizer.zero_grad(set_to_none=True)

        outputs = resnet_model(images)
        loss = resnet_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(resnet_model.parameters(), max_norm=1.0)

        resnet_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(resnet_model, val_loader, resnet_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_resnet_dice:

        best_resnet_dice = current_dice
        epochs_without_improvement = 0

        torch.save(resnet_model.state_dict(), best_resnet_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    resnet_scheduler.step(current_dice)

    encoder_lr = resnet_optimizer.param_groups[0]["lr"]
    new_lr = resnet_optimizer.param_groups[1]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"RV: {val_dice[0].item():.4f} | "
        f"Myocardium: {val_dice[1].item():.4f} | "
        f"LV: {val_dice[2].item():.4f} | "
        f"Encoder LR: {encoder_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("RESNET34 TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_resnet_dice:.4f}")
print("Best model saved to:", best_resnet_path)


# ============================================================
# BACKUP IMMEDIATELY
# ============================================================

shutil.copy(best_resnet_path, "/content/drive/MyDrive/model_checkpoints_backup/best_resnet_transunet.pth")
print("Backed up to Drive!")

# LOSS AND OPTIMIZER FOR RESNET34 TRANSUNET

In [ ]:


resnet_loss_fn = ACDCLoss(num_classes=4).to(device)

encoder_params = list(resnet_model.stem.parameters()) + \
    list(resnet_model.layer1.parameters()) + \
    list(resnet_model.layer2.parameters()) + \
    list(resnet_model.layer3.parameters()) + \
    list(resnet_model.layer4.parameters())

new_params = list(resnet_model.bottleneck.parameters()) + \
    list(resnet_model.transformer.parameters()) + \
    list(resnet_model.transformer_refine.parameters()) + \
    list(resnet_model.decoder4.parameters()) + \
    list(resnet_model.decoder3.parameters()) + \
    list(resnet_model.decoder2.parameters()) + \
    list(resnet_model.decoder1.parameters()) + \
    list(resnet_model.final_refine.parameters()) + \
    list(resnet_model.output_layer.parameters())

resnet_optimizer = optim.AdamW([
    {"params": encoder_params, "lr": 2e-5},
    {"params": new_params, "lr": 1e-4}
], weight_decay=1e-4)

resnet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    resnet_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")
print("Encoder LR:", resnet_optimizer.param_groups[0]["lr"])
print("New layers LR:", resnet_optimizer.param_groups[1]["lr"])


# TRAIN RESNET34 TRANSUNET


num_epochs = 150

best_resnet_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_resnet_path = "/content/best_resnet_transunet.pth"

print()
print("Starting ResNet34 TransUNet training...")
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print()

for epoch in range(num_epochs):

    resnet_model.train()

    total_train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        resnet_optimizer.zero_grad(set_to_none=True)

        outputs = resnet_model(images)
        loss = resnet_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(resnet_model.parameters(), max_norm=1.0)

        resnet_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(resnet_model, val_loader, resnet_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_resnet_dice:

        best_resnet_dice = current_dice
        epochs_without_improvement = 0

        torch.save(resnet_model.state_dict(), best_resnet_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    resnet_scheduler.step(current_dice)

    encoder_lr = resnet_optimizer.param_groups[0]["lr"]
    new_lr = resnet_optimizer.param_groups[1]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"RV: {val_dice[0].item():.4f} | "
        f"Myocardium: {val_dice[1].item():.4f} | "
        f"LV: {val_dice[2].item():.4f} | "
        f"Encoder LR: {encoder_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("RESNET34 TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_resnet_dice:.4f}")
print("Best model saved to:", best_resnet_path)

# FINAL TEST - RESNET34 TRANSUNET

In [ ]:


resnet_model.load_state_dict(torch.load(best_resnet_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(resnet_model, test_loader, resnet_loss_fn, device)

print("RESNET34 TRANSUNET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"RV Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Myocardium Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"LV Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")



```
# GRID GRAPH EDGES FOR THE BOTTLENECK
```



In [ ]:


def create_grid_edges(height, width):

    edges = []

    for row in range(height):
        for col in range(width):

            node = row * width + col

            if col < width - 1:
                right = node + 1
                edges.append([node, right])
                edges.append([right, node])

            if row < height - 1:
                down = node + width
                edges.append([node, down])
                edges.append([down, node])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return edge_index


print("Grid edge function ready!")

# GRAPH BOTTLENECK BLOCK

In [ ]:
!pip install -q torch-geometric

from torch_geometric.nn import SAGEConv


class GraphBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, hidden_dim=512, num_layers=4):
        super().__init__()

        self.num_layers = num_layers

        self.gnn_layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()

        for _ in range(num_layers):
            self.gnn_layers.append(SAGEConv(feature_dim, hidden_dim))
            self.norm_layers.append(nn.LayerNorm(hidden_dim))


    def forward(self, x, edge_index):

        # x shape: [batch, num_nodes, feature_dim]
        # SAGEConv only accepts [num_nodes, feature_dim], so each
        # image in the batch is processed one at a time

        batch_size = x.size(0)

        output_list = []

        for batch_index in range(batch_size):

            node_features = x[batch_index]

            for layer_index in range(self.num_layers):

                residual = node_features

                node_features = self.gnn_layers[layer_index](node_features, edge_index)
                node_features = self.norm_layers[layer_index](node_features)
                node_features = F.gelu(node_features)
                node_features = node_features + residual

            output_list.append(node_features)

        x = torch.stack(output_list, dim=0)

        return x


# ============================================================
# GRAPH U-NET WITH RESNET34 ENCODER
# Same encoder and decoder as ResNetTransUNet,
# transformer bottleneck replaced with graph reasoning
# ============================================================

class ResNetGraphUNet(nn.Module):

    def __init__(self, out_channels=4, pretrained=True):
        super().__init__()

        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = models.resnet34(weights=weights)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)

        self.graph_reasoning = GraphBottleneckBlock(
            feature_dim=512,
            hidden_dim=512,
            num_layers=4
        )

        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)

        self.graph_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 32)

        self.output_layer = nn.Conv2d(32, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device

        edge_index = self.bottleneck_edge_index.to(device)

        s = self.stem(x)
        p = self.pool(s)

        e1 = self.layer1(p)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape

        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.graph_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, s)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        d1 = self.final_refine(d1)

        output = self.output_layer(d1)

        return output


graph_model = ResNetGraphUNet(out_channels=4, pretrained=True).to(device)

print("ResNet34 Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in graph_model.parameters()))

ACDC Standalone GNN

In [ ]:
# ============================================================
# RECREATE + TRAIN ACDC GRAPH U-NET (RESNET34) - FULL CODE
# ============================================================

class ResNetGraphUNet(nn.Module):

    def __init__(self, out_channels=4, pretrained=True):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)

        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)

        self.graph_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape

        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.graph_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)

        output = self.output_layer(d1)

        return output


graph_model = ResNetGraphUNet(out_channels=4, pretrained=True).to(device)

print("ACDC Graph U-Net (ResNet34) created!")
print("Total parameters:", sum(p.numel() for p in graph_model.parameters()))


# ============================================================
# LOSS AND OPTIMIZER FOR RESNET34 GRAPH U-NET
# ============================================================

graph_loss_fn = ACDCLoss(num_classes=4).to(device)

graph_encoder_params = list(graph_model.stem.parameters()) + \
    list(graph_model.layer1.parameters()) + \
    list(graph_model.layer2.parameters()) + \
    list(graph_model.layer3.parameters()) + \
    list(graph_model.layer4.parameters())

graph_new_params = list(graph_model.bottleneck.parameters()) + \
    list(graph_model.graph_reasoning.parameters()) + \
    list(graph_model.graph_refine.parameters()) + \
    list(graph_model.decoder4.parameters()) + \
    list(graph_model.decoder3.parameters()) + \
    list(graph_model.decoder2.parameters()) + \
    list(graph_model.decoder1.parameters()) + \
    list(graph_model.final_refine.parameters()) + \
    list(graph_model.output_layer.parameters())

graph_optimizer = optim.AdamW([
    {"params": graph_encoder_params, "lr": 2e-5},
    {"params": graph_new_params, "lr": 1e-4}
], weight_decay=1e-4)

graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")
print("Encoder LR:", graph_optimizer.param_groups[0]["lr"])
print("New layers LR:", graph_optimizer.param_groups[1]["lr"])


# ============================================================
# TRAIN RESNET34 GRAPH U-NET
# ============================================================

num_epochs = 150

best_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_graph_path = "/content/best_resnet_graph_unet.pth"

print()
print("Starting ResNet34 Graph U-Net training...")
print("Training images:", len(train_loader.dataset))
print("Validation images:", len(val_loader.dataset))
print()

for epoch in range(num_epochs):

    graph_model.train()

    total_train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        graph_optimizer.zero_grad(set_to_none=True)

        outputs = graph_model(images)
        loss = graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(graph_model.parameters(), max_norm=1.0)

        graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(graph_model, val_loader, graph_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_graph_dice:

        best_graph_dice = current_dice
        epochs_without_improvement = 0

        torch.save(graph_model.state_dict(), best_graph_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    graph_scheduler.step(current_dice)

    encoder_lr = graph_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"RV: {val_dice[0].item():.4f} | "
        f"Myocardium: {val_dice[1].item():.4f} | "
        f"LV: {val_dice[2].item():.4f} | "
        f"Encoder LR: {encoder_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("RESNET34 GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_graph_dice:.4f}")
print("Best model saved to:", best_graph_path)


# ============================================================
# BACKUP IMMEDIATELY
# ============================================================

shutil.copy(best_graph_path, "/content/drive/MyDrive/model_checkpoints_backup/best_resnet_graph_unet.pth")
print("Backed up to Drive!")

Test Numbers

In [ ]:
graph_model.load_state_dict(torch.load(best_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(graph_model, test_loader, graph_loss_fn, device)

print("RESNET34 GRAPH U-NET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"RV Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Myocardium Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"LV Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# COMBINED GNN + TRANSFORMER BOTTLENECK

In [ ]:


class CombinedBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=64):
        super().__init__()

        self.graph_reasoning = GraphBottleneckBlock(
            feature_dim=feature_dim,
            hidden_dim=feature_dim,
            num_layers=num_graph_layers
        )

        self.transformer = TransformerBlock(
            embed_dim=feature_dim,
            num_heads=num_heads,
            num_layers=num_transformer_layers,
            num_tokens=num_tokens
        )


    def forward(self, x, edge_index):

        x = self.graph_reasoning(x, edge_index)
        x = self.transformer(x)

        return x


# ============================================================
# COMBINED GNN + TRANSFORMER MODEL FOR ACDC
# Same ResNet34 encoder/decoder as the winning ACDC models
# ============================================================

class ResNetCombinedUNet(nn.Module):

    def __init__(self, out_channels=4, pretrained=True):
        super().__init__()

        weights = models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = models.resnet34(weights=weights)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool

        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)

        self.combined_reasoning = CombinedBottleneckBlock(
            feature_dim=512,
            num_graph_layers=4,
            num_heads=8,
            num_transformer_layers=4,
            num_tokens=64
        )

        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)

        self.combined_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 32)

        self.output_layer = nn.Conv2d(32, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device

        edge_index = self.bottleneck_edge_index.to(device)

        s = self.stem(x)
        p = self.pool(s)

        e1 = self.layer1(p)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape

        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.combined_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, s)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        d1 = self.final_refine(d1)

        output = self.output_layer(d1)

        return output


acdc_combined_model = ResNetCombinedUNet(out_channels=4, pretrained=True).to(device)

print("ACDC Combined GNN+Transformer U-Net created!")
print("Total parameters:", sum(p.numel() for p in acdc_combined_model.parameters()))

# LOSS AND OPTIMIZER FOR ACDC COMBINED MODEL

In [ ]:


class ResNetCombinedUNet(nn.Module):

    def __init__(self, out_channels=4, pretrained=True):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)

        self.combined_reasoning = CombinedBottleneckBlock(
            feature_dim=512,
            num_graph_layers=4,
            num_heads=8,
            num_transformer_layers=4,
            num_tokens=64
        )

        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)

        self.combined_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape

        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.combined_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)

        output = self.output_layer(d1)

        return output


acdc_combined_model = ResNetCombinedUNet(out_channels=4, pretrained=True).to(device)

print("ACDC Combined GNN+Transformer (ResNet34) created!")
print("Total parameters:", sum(p.numel() for p in acdc_combined_model.parameters()))


acdc_combined_loss_fn = ACDCLoss(num_classes=4).to(device)

acdc_combined_encoder_params = list(acdc_combined_model.stem.parameters()) + \
    list(acdc_combined_model.layer1.parameters()) + \
    list(acdc_combined_model.layer2.parameters()) + \
    list(acdc_combined_model.layer3.parameters()) + \
    list(acdc_combined_model.layer4.parameters())

acdc_combined_new_params = list(acdc_combined_model.bottleneck.parameters()) + \
    list(acdc_combined_model.combined_reasoning.parameters()) + \
    list(acdc_combined_model.combined_refine.parameters()) + \
    list(acdc_combined_model.decoder4.parameters()) + \
    list(acdc_combined_model.decoder3.parameters()) + \
    list(acdc_combined_model.decoder2.parameters()) + \
    list(acdc_combined_model.decoder1.parameters()) + \
    list(acdc_combined_model.final_refine.parameters()) + \
    list(acdc_combined_model.output_layer.parameters())

acdc_combined_optimizer = optim.AdamW([
    {"params": acdc_combined_encoder_params, "lr": 2e-5},
    {"params": acdc_combined_new_params, "lr": 1e-4}
], weight_decay=1e-4)

acdc_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    acdc_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_acdc_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_acdc_combined_path = "/content/best_acdc_combined_unet.pth"

print()
print("Starting ACDC Combined model training...")
print()

for epoch in range(num_epochs):

    acdc_combined_model.train()

    total_train_loss = 0.0

    for images, masks in train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        acdc_combined_optimizer.zero_grad(set_to_none=True)

        outputs = acdc_combined_model(images)
        loss = acdc_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(acdc_combined_model.parameters(), max_norm=1.0)

        acdc_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(acdc_combined_model, val_loader, acdc_combined_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_acdc_combined_dice:
        best_acdc_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(acdc_combined_model.state_dict(), best_acdc_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    acdc_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("ACDC COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_acdc_combined_dice:.4f}")


shutil.copy(best_acdc_combined_path, "/content/drive/MyDrive/model_checkpoints_backup/best_acdc_combined_unet.pth")
print("Backed up to Drive!")

Results

In [ ]:
acdc_combined_model.load_state_dict(torch.load(best_acdc_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(acdc_combined_model, test_loader, acdc_combined_loss_fn, device)

print("ACDC COMBINED MODEL FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"RV Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Myocardium Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"LV Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# VISUALIZATIONS

In [ ]:


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def compute_single_image_dice(prediction, target, num_classes=4):

    dice_scores = []

    for class_id in range(1, num_classes):

        pred_class = (prediction == class_id)
        target_class = (target == class_id)

        intersection = np.logical_and(pred_class, target_class).sum()
        denom = pred_class.sum() + target_class.sum()

        if denom == 0:
            continue

        dice = (2.0 * intersection) / denom
        dice_scores.append(dice)

    if len(dice_scores) == 0:
        return 0.0

    return np.mean(dice_scores)


def visualize_predictions(model, dataset, device, num_images=40, num_classes=4, class_colors=None, dataset_name="ACDC"):

    model.eval()

    if class_colors is None:
        class_colors = {1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1]}

    num_images = min(num_images, len(dataset))

    cols = 5
    rows = (num_images * 3 + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    plot_idx = 0

    with torch.no_grad():

        for idx in range(num_images):

            image, true_mask = dataset[idx]

            input_image = image.unsqueeze(0).to(device)

            output = model(input_image)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            true_mask_np = true_mask.numpy()

            dice = compute_single_image_dice(prediction, true_mask_np, num_classes)

            display_image = image[0].cpu().numpy()
            display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min() + 1e-8)

            gt_overlay = np.stack([display_image] * 3, axis=-1)
            pred_overlay = np.stack([display_image] * 3, axis=-1)

            for class_id, color in class_colors.items():

                gt_mask = (true_mask_np == class_id)
                pred_mask = (prediction == class_id)

                for c in range(3):
                    gt_overlay[..., c] = np.where(gt_mask, color[c], gt_overlay[..., c])
                    pred_overlay[..., c] = np.where(pred_mask, color[c], pred_overlay[..., c])

            axes[plot_idx].imshow(display_image, cmap="gray")
            axes[plot_idx].set_title("Input", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(gt_overlay)
            axes[plot_idx].set_title("Ground Truth", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(pred_overlay)
            axes[plot_idx].set_title(f"Prediction (Dice: {dice:.3f})", fontsize=9, fontweight="bold")
            axes[plot_idx].axis("off")
            plot_idx += 1

    for remaining_idx in range(plot_idx, len(axes)):
        axes[remaining_idx].axis("off")

    legend_patches = [
        mpatches.Patch(color=class_colors[1], label="RV"),
        mpatches.Patch(color=class_colors[2], label="Myocardium"),
        mpatches.Patch(color=class_colors[3], label="LV")
    ]

    fig.legend(handles=legend_patches, loc="upper center", ncol=3, fontsize=11, bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{dataset_name} - Predictions with Per-Image Dice Score", fontsize=13, y=1.03)

    plt.tight_layout()
    plt.show()


print("Visualization function ready!")

In [ ]:
visualize_predictions(
    acdc_combined_model,
    test_dataset,
    device,
    num_images=40,
    num_classes=4,
    dataset_name="ACDC Combined GNN+Transformer"
)

# SSIM ACDC

In [ ]:


resnet_model.load_state_dict(torch.load(best_resnet_path, map_location=device))
mean_ssim_acdc_transunet = evaluate_ssim(resnet_model, test_loader, device, num_classes=4)

print("ACDC TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_acdc_transunet:.4f}")
print()

graph_model.load_state_dict(torch.load(best_graph_path, map_location=device))
mean_ssim_acdc_graph = evaluate_ssim(graph_model, test_loader, device, num_classes=4)

print("ACDC GRAPH U-NET - SSIM")
print(f"Mean SSIM: {mean_ssim_acdc_graph:.4f}")
print()

acdc_combined_model.load_state_dict(torch.load(best_acdc_combined_path, map_location=device))
mean_ssim_acdc_combined = evaluate_ssim(acdc_combined_model, test_loader, device, num_classes=4)

print("ACDC COMBINED MODEL - SSIM")
print(f"Mean SSIM: {mean_ssim_acdc_combined:.4f}")

DOWNLOAD MS-CMRSEG DATASET

In [ ]:
!pip install -q kagglehub

import kagglehub

mnms_path = kagglehub.dataset_download(
    "anhoangvo/mscmrseg"
)

print("Dataset downloaded!")
print("Dataset path:", mnms_path)

# EXPLORE MS-CMRSEG DATASET STRUCTURE

In [ ]:


for root, dirs, files in os.walk(mnms_path):

    level = root.replace(mnms_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
slices_folder = os.path.join(mnms_path, "MSCMR_preprocessed", "MSCMR_training_slices")

print("Folder exists:", os.path.exists(slices_folder))

if os.path.exists(slices_folder):
    all_files = os.listdir(slices_folder)
    print("Number of files:", len(all_files))
    print("First 5 files:", all_files[:5])

# INSPECT ONE MS-CMRSEG H5 FILE

In [ ]:
# ============================================================
# INSPECT ONE MS-CMRSEG H5 FILE
# ============================================================

import h5py

sample_path = os.path.join(slices_folder, "subject34_DE_slice_0.h5")

with h5py.File(sample_path, "r") as f:

    print("Keys in file:", list(f.keys()))

    for key in f.keys():
        print(f"{key} shape:", f[key].shape)
        print(f"{key} dtype:", f[key].dtype)

# CHECK LABEL CLASSES ACROSS A FEW FILES

In [ ]:


import numpy as np

all_files = os.listdir(slices_folder)

unique_values = set()

for filename in all_files[:20]:

    file_path = os.path.join(slices_folder, filename)

    with h5py.File(file_path, "r") as f:
        label = f["label"][:]
        unique_values.update(np.unique(label).tolist())

print("Unique label values found across 20 sample files:", sorted(unique_values))

# MS-CMRSEG SUBJECT-LEVEL TRAIN/VAL

In [ ]:


all_slice_files = [f for f in os.listdir(slices_folder) if f.endswith(".h5")]

subjects = sorted(list(set(
    filename.split("_DE_slice_")[0] for filename in all_slice_files
)))

print("Total training slices:", len(all_slice_files))
print("Total unique subjects:", len(subjects))

random.shuffle(subjects)

num_subjects = len(subjects)
num_val_subjects = max(1, int(num_subjects * 0.15))

val_subjects = set(subjects[:num_val_subjects])
train_subjects = set(subjects[num_val_subjects:])

print("Training subjects:", len(train_subjects))
print("Validation subjects:", len(val_subjects))

mscmr_train_files = []
mscmr_val_files = []

for filename in all_slice_files:

    subject_id = filename.split("_DE_slice_")[0]
    file_path = os.path.join(slices_folder, filename)

    if subject_id in train_subjects:
        mscmr_train_files.append(file_path)

    elif subject_id in val_subjects:
        mscmr_val_files.append(file_path)

print()
print("Training slices:", len(mscmr_train_files))
print("Validation slices:", len(mscmr_val_files))

# MS-CMRSEG DATASET

In [ ]:


import h5py


class MSCMRDataset(Dataset):

    def __init__(self, file_list, image_size=256, augment=False):

        self.file_list = file_list
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.file_list)


    def __getitem__(self, idx):

        file_path = self.file_list[idx]

        with h5py.File(file_path, "r") as f:
            image = f["image"][:].astype(np.float32)
            label = f["label"][:].astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(label, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:
            image = (image - image_min) / (image_max - image_min)

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.5:
                dx = int(random.uniform(-0.05, 0.05) * self.image_size)
                dy = int(random.uniform(-0.05, 0.05) * self.image_size)
                scale = random.uniform(0.95, 1.05)
                image = TF.affine(image, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.affine(mask, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.3:
                mean_val = image.mean()
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp((image - mean_val) * factor + mean_val, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        image = image.repeat(3, 1, 1)
        mask = mask.squeeze(0)

        return image, mask


mscmr_train_dataset = MSCMRDataset(mscmr_train_files, image_size=256, augment=True)
mscmr_val_dataset = MSCMRDataset(mscmr_val_files, image_size=256, augment=False)

print("MS-CMRSeg datasets created!")
print("Training samples:", len(mscmr_train_dataset))
print("Validation samples:", len(mscmr_val_dataset))

# INSPECT ONE MS-CMRSEG TESTING VOLUME

In [ ]:


testing_volumes_folder = os.path.join(mnms_path, "MSCMR_preprocessed", "MSCMR_testing_volumes")

sample_volume_files = os.listdir(testing_volumes_folder)

print("Number of testing volume files:", len(sample_volume_files))
print("First file:", sample_volume_files[0])

sample_volume_path = os.path.join(testing_volumes_folder, sample_volume_files[0])

with h5py.File(sample_volume_path, "r") as f:

    print("Keys in file:", list(f.keys()))

    for key in f.keys():
        print(f"{key} shape:", f[key].shape)
        print(f"{key} dtype:", f[key].dtype)

# SLICE MS-CMRSEG TESTING VOLUMES INTO 2D TEST SET

In [ ]:


testing_volumes_folder = os.path.join(mnms_path, "MSCMR_preprocessed", "MSCMR_testing_volumes")

volume_files = [f for f in os.listdir(testing_volumes_folder) if f.endswith(".h5")]

print("Number of testing volumes:", len(volume_files))

mscmr_test_slices = []

for filename in volume_files:

    file_path = os.path.join(testing_volumes_folder, filename)

    with h5py.File(file_path, "r") as f:

        images = f["image"][:]
        labels = f["label"][:]

    num_slices = images.shape[0]

    for slice_idx in range(num_slices):

        mscmr_test_slices.append((
            images[slice_idx],
            labels[slice_idx]
        ))

print("Total test slices after slicing all volumes:", len(mscmr_test_slices))

# MS-CMRSEG TEST DATASET

In [ ]:


class MSCMRTestDataset(Dataset):

    def __init__(self, slice_list, image_size=256):

        self.slice_list = slice_list
        self.image_size = image_size


    def __len__(self):
        return len(self.slice_list)


    def __getitem__(self, idx):

        image, label = self.slice_list[idx]

        image = image.astype(np.float32)
        label = label.astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(label, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:
            image = (image - image_min) / (image_max - image_min)

        image = image.repeat(3, 1, 1)
        mask = mask.squeeze(0)

        return image, mask


mscmr_test_dataset = MSCMRTestDataset(mscmr_test_slices, image_size=256)

print("MS-CMRSeg test dataset created!")
print("Test samples:", len(mscmr_test_dataset))


# ============================================================
# MS-CMRSEG DATALOADERS
# ============================================================

mscmr_train_loader = DataLoader(mscmr_train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
mscmr_val_loader = DataLoader(mscmr_val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
mscmr_test_loader = DataLoader(mscmr_test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("MS-CMRSeg DataLoaders created!")
print("Training batches:", len(mscmr_train_loader))
print("Validation batches:", len(mscmr_val_loader))
print("Testing batches:", len(mscmr_test_loader))

# TRANSUNET FOR MS-CMRSEG

In [ ]:
mscmr_model = TransUNet(in_channels=3, out_channels=4).to(device)

mscmr_loss_fn = ACDCLoss(num_classes=4).to(device)

mscmr_optimizer = optim.AdamW(mscmr_model.parameters(), lr=1e-4, weight_decay=1e-4)

mscmr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mscmr_optimizer, mode="max", factor=0.5, patience=8
)

print("MS-CMRSeg TransUNet created!")
print("Total parameters:", sum(p.numel() for p in mscmr_model.parameters()))


num_epochs = 150

best_mscmr_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_mscmr_path = "/content/best_mscmr_transunet.pth"

print()
print("Starting MS-CMRSeg TransUNet training...")
print("Training images:", len(mscmr_train_loader.dataset))
print("Validation images:", len(mscmr_val_loader.dataset))
print()

for epoch in range(num_epochs):

    mscmr_model.train()

    total_train_loss = 0.0

    for images, masks in mscmr_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        mscmr_optimizer.zero_grad(set_to_none=True)

        outputs = mscmr_model(images)
        loss = mscmr_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(mscmr_model.parameters(), max_norm=1.0)

        mscmr_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(mscmr_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(mscmr_model, mscmr_val_loader, mscmr_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_mscmr_dice:

        best_mscmr_dice = current_dice
        epochs_without_improvement = 0

        torch.save(mscmr_model.state_dict(), best_mscmr_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    mscmr_scheduler.step(current_dice)

    current_lr = mscmr_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("MS-CMRSEG TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_mscmr_dice:.4f}")
print("Best model saved to:", best_mscmr_path)


shutil.copy(best_mscmr_path, "/content/drive/MyDrive/model_checkpoints_backup/best_mscmr_transunet.pth")
print("Backed up to Drive!")

Test Results


In [ ]:


mscmr_model.load_state_dict(torch.load(best_mscmr_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(mscmr_model, mscmr_test_loader, mscmr_loss_fn, device)

print("MS-CMRSEG TRANSUNET FINAL TEST RESULTS (RETRAINED)")
print(f"Test Loss: {test_loss:.4f}")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

 For use with pretrained ResNet34 encoder

In [ ]:
# ============================================================
# MS-CMRSEG DATASET WITH IMAGENET NORMALIZATION
# For use with pretrained ResNet34 encoder
# ============================================================

class MSCMRDatasetImageNet(Dataset):

    def __init__(self, file_list, image_size=256, augment=False):

        self.file_list = file_list
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.file_list)


    def __getitem__(self, idx):

        file_path = self.file_list[idx]

        with h5py.File(file_path, "r") as f:
            image = f["image"][:].astype(np.float32)
            label = f["label"][:].astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(label, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:
            image = (image - image_min) / (image_max - image_min)

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.5:
                dx = int(random.uniform(-0.05, 0.05) * self.image_size)
                dy = int(random.uniform(-0.05, 0.05) * self.image_size)
                scale = random.uniform(0.95, 1.05)
                image = TF.affine(image, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.affine(mask, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.3:
                mean_val = image.mean()
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp((image - mean_val) * factor + mean_val, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        image = image.repeat(3, 1, 1)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=image.dtype).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=image.dtype).view(3, 1, 1)
        image = (image - mean) / std

        mask = mask.squeeze(0)

        return image, mask


class MSCMRTestDatasetImageNet(Dataset):

    def __init__(self, slice_list, image_size=256):

        self.slice_list = slice_list
        self.image_size = image_size


    def __len__(self):
        return len(self.slice_list)


    def __getitem__(self, idx):

        image, label = self.slice_list[idx]

        image = image.astype(np.float32)
        label = label.astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(label, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image_min = image.min()
        image_max = image.max()

        if image_max > image_min:
            image = (image - image_min) / (image_max - image_min)

        image = image.repeat(3, 1, 1)

        mean = torch.tensor([0.485, 0.456, 0.406], dtype=image.dtype).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=image.dtype).view(3, 1, 1)
        image = (image - mean) / std

        mask = mask.squeeze(0)

        return image, mask


mscmr_train_dataset_in = MSCMRDatasetImageNet(mscmr_train_files, image_size=256, augment=True)
mscmr_val_dataset_in = MSCMRDatasetImageNet(mscmr_val_files, image_size=256, augment=False)
mscmr_test_dataset_in = MSCMRTestDatasetImageNet(mscmr_test_slices, image_size=256)

mscmr_train_loader_in = DataLoader(mscmr_train_dataset_in, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
mscmr_val_loader_in = DataLoader(mscmr_val_dataset_in, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
mscmr_test_loader_in = DataLoader(mscmr_test_dataset_in, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("MS-CMRSeg ImageNet-normalized DataLoaders created!")
print("Training batches:", len(mscmr_train_loader_in))
print("Validation batches:", len(mscmr_val_loader_in))
print("Testing batches:", len(mscmr_test_loader_in))

# RESNET34 TRANSUNET FOR MS-CMRSEG

In [ ]:


mscmr_resnet_model = ResNetTransUNet(out_channels=4, pretrained=True).to(device)

mscmr_resnet_loss_fn = ACDCLoss(num_classes=4).to(device)

mscmr_encoder_params = list(mscmr_resnet_model.stem.parameters()) + \
    list(mscmr_resnet_model.layer1.parameters()) + \
    list(mscmr_resnet_model.layer2.parameters()) + \
    list(mscmr_resnet_model.layer3.parameters()) + \
    list(mscmr_resnet_model.layer4.parameters())

mscmr_new_params = list(mscmr_resnet_model.bottleneck.parameters()) + \
    list(mscmr_resnet_model.transformer.parameters()) + \
    list(mscmr_resnet_model.transformer_refine.parameters()) + \
    list(mscmr_resnet_model.decoder4.parameters()) + \
    list(mscmr_resnet_model.decoder3.parameters()) + \
    list(mscmr_resnet_model.decoder2.parameters()) + \
    list(mscmr_resnet_model.decoder1.parameters()) + \
    list(mscmr_resnet_model.final_refine.parameters()) + \
    list(mscmr_resnet_model.output_layer.parameters())

mscmr_resnet_optimizer = optim.AdamW([
    {"params": mscmr_encoder_params, "lr": 2e-5},
    {"params": mscmr_new_params, "lr": 1e-4}
], weight_decay=1e-4)

mscmr_resnet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mscmr_resnet_optimizer, mode="max", factor=0.5, patience=8
)

print("MS-CMRSeg ResNet34 TransUNet created!")
print("Total parameters:", sum(p.numel() for p in mscmr_resnet_model.parameters()))


# ============================================================
# TRAIN RESNET34 TRANSUNET ON MS-CMRSEG
# ============================================================

num_epochs = 150

best_mscmr_resnet_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_mscmr_resnet_path = "/content/best_mscmr_resnet_transunet.pth"

print()
print("Starting MS-CMRSeg ResNet34 TransUNet training...")
print("Training images:", len(mscmr_train_loader_in.dataset))
print("Validation images:", len(mscmr_val_loader_in.dataset))
print()

for epoch in range(num_epochs):

    mscmr_resnet_model.train()

    total_train_loss = 0.0

    for images, masks in mscmr_train_loader_in:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        mscmr_resnet_optimizer.zero_grad(set_to_none=True)

        outputs = mscmr_resnet_model(images)
        loss = mscmr_resnet_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(mscmr_resnet_model.parameters(), max_norm=1.0)

        mscmr_resnet_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(mscmr_train_loader_in.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(mscmr_resnet_model, mscmr_val_loader_in, mscmr_resnet_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_mscmr_resnet_dice:

        best_mscmr_resnet_dice = current_dice
        epochs_without_improvement = 0

        torch.save(mscmr_resnet_model.state_dict(), best_mscmr_resnet_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    mscmr_resnet_scheduler.step(current_dice)

    encoder_lr = mscmr_resnet_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f} | "
        f"Encoder LR: {encoder_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("MS-CMRSEG RESNET34 TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_mscmr_resnet_dice:.4f}")
print("Best model saved to:", best_mscmr_resnet_path)

Results


In [ ]:
mscmr_resnet_model.load_state_dict(torch.load(best_mscmr_resnet_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(mscmr_resnet_model, mscmr_test_loader_in, mscmr_resnet_loss_fn, device)

print("MS-CMRSEG RESNET34 TRANSUNET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

 GRAPH U-NET FOR MS-CMRSEG

In [ ]:


class MSCMRGraphUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)

        self.graph_reasoning = GraphBottleneckBlock(
            feature_dim=512,
            hidden_dim=512,
            num_layers=4
        )

        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)

        self.graph_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device

        edge_index = self.bottleneck_edge_index.to(device)

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape

        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.graph_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)

        output = self.output_layer(d1)

        return output


mscmr_graph_model = MSCMRGraphUNet(in_channels=3, out_channels=4).to(device)

print("MS-CMRSeg Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in mscmr_graph_model.parameters()))

# GNN FOR MS-CMRSEG DATASET
# LOSS AND OPTIMIZER

In [ ]:


mscmr_graph_model = MSCMRGraphUNet(in_channels=3, out_channels=4).to(device)

print("MS-CMRSeg Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in mscmr_graph_model.parameters()))

mscmr_graph_loss_fn = ACDCLoss(num_classes=4).to(device)

mscmr_graph_optimizer = optim.AdamW(mscmr_graph_model.parameters(), lr=1e-4, weight_decay=1e-4)

mscmr_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mscmr_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_mscmr_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_mscmr_graph_path = "/content/best_mscmr_graph_unet.pth"

print()
print("Starting MS-CMRSeg Graph U-Net training...")
print("Training images:", len(mscmr_train_loader.dataset))
print("Validation images:", len(mscmr_val_loader.dataset))
print()

for epoch in range(num_epochs):

    mscmr_graph_model.train()

    total_train_loss = 0.0

    for images, masks in mscmr_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        mscmr_graph_optimizer.zero_grad(set_to_none=True)

        outputs = mscmr_graph_model(images)
        loss = mscmr_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(mscmr_graph_model.parameters(), max_norm=1.0)

        mscmr_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(mscmr_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(mscmr_graph_model, mscmr_val_loader, mscmr_graph_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_mscmr_graph_dice:
        best_mscmr_graph_dice = current_dice
        epochs_without_improvement = 0
        torch.save(mscmr_graph_model.state_dict(), best_mscmr_graph_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    mscmr_graph_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("MS-CMRSEG GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_mscmr_graph_dice:.4f}")


shutil.copy(best_mscmr_graph_path, "/content/drive/MyDrive/model_checkpoints_backup/best_mscmr_graph_unet.pth")
print("Backed up to Drive!")


Results


In [ ]:
mscmr_graph_model.load_state_dict(torch.load(best_mscmr_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(mscmr_graph_model, mscmr_test_loader, mscmr_graph_loss_fn, device)

print("MS-CMRSEG GRAPH U-NET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# COMBINED GNN + TRANSFORMER MODEL FOR MS-CMRSEG

In [ ]:


class MSCMRCombinedUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)

        self.combined_reasoning = CombinedBottleneckBlock(
            feature_dim=512,
            num_graph_layers=4,
            num_heads=8,
            num_transformer_layers=4,
            num_tokens=256
        )

        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)

        self.combined_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):

        input_size = x.shape[2:]
        device = x.device

        edge_index = self.bottleneck_edge_index.to(device)

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape

        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)

        b = self.combined_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)

        output = self.output_layer(d1)

        return output


mscmr_combined_model = MSCMRCombinedUNet(in_channels=3, out_channels=4).to(device)

print("MS-CMRSeg Combined GNN+Transformer U-Net created!")
print("Total parameters:", sum(p.numel() for p in mscmr_combined_model.parameters()))

# LOSS AND OPTIMIZER FOR MS-CMRSEG GNN + Transformer


In [ ]:


mscmr_combined_model = MSCMRCombinedUNet(in_channels=3, out_channels=4).to(device)

print("MS-CMRSeg Combined GNN+Transformer created!")
print("Total parameters:", sum(p.numel() for p in mscmr_combined_model.parameters()))

mscmr_combined_loss_fn = ACDCLoss(num_classes=4).to(device)

mscmr_combined_optimizer = optim.AdamW(mscmr_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

mscmr_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    mscmr_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_mscmr_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_mscmr_combined_path = "/content/best_mscmr_combined_unet.pth"

print()
print("Starting MS-CMRSeg Combined model training...")
print("Training images:", len(mscmr_train_loader.dataset))
print("Validation images:", len(mscmr_val_loader.dataset))
print()

for epoch in range(num_epochs):

    mscmr_combined_model.train()

    total_train_loss = 0.0

    for images, masks in mscmr_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        mscmr_combined_optimizer.zero_grad(set_to_none=True)

        outputs = mscmr_combined_model(images)
        loss = mscmr_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(mscmr_combined_model.parameters(), max_norm=1.0)

        mscmr_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(mscmr_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(mscmr_combined_model, mscmr_val_loader, mscmr_combined_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_mscmr_combined_dice:
        best_mscmr_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(mscmr_combined_model.state_dict(), best_mscmr_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    mscmr_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("MS-CMRSEG COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_mscmr_combined_dice:.4f}")


shutil.copy(best_mscmr_combined_path, "/content/drive/MyDrive/model_checkpoints_backup/best_mscmr_combined_unet.pth")
print("Backed up to Drive!")

In [ ]:
mscmr_combined_model.load_state_dict(torch.load(best_mscmr_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(mscmr_combined_model, mscmr_test_loader, mscmr_combined_loss_fn, device)

print("MS-CMRSEG COMBINED MODEL FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# ============================================================
# FINAL TEST - ALL 6 RETRAINED MODELS
# ============================================================

# ACDC TransUNet
resnet_model.load_state_dict(torch.load(best_resnet_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(resnet_model, test_loader, resnet_loss_fn, device)
print("ACDC TRANSUNET TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

# ACDC Graph U-Net
graph_model.load_state_dict(torch.load(best_graph_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(graph_model, test_loader, graph_loss_fn, device)
print("ACDC GRAPH U-NET TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

# ACDC Combined
acdc_combined_model.load_state_dict(torch.load(best_acdc_combined_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(acdc_combined_model, test_loader, acdc_combined_loss_fn, device)
print("ACDC COMBINED TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

print()

# MS-CMRSeg TransUNet
mscmr_model.load_state_dict(torch.load(best_mscmr_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(mscmr_model, mscmr_test_loader, mscmr_loss_fn, device)
print("MSCMR TRANSUNET TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

# MS-CMRSeg Graph U-Net
mscmr_graph_model.load_state_dict(torch.load(best_mscmr_graph_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(mscmr_graph_model, mscmr_test_loader, mscmr_graph_loss_fn, device)
print("MSCMR GRAPH U-NET TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

# MS-CMRSeg Combined
mscmr_combined_model.load_state_dict(torch.load(best_mscmr_combined_path, map_location=device))
test_loss, test_mean_dice, test_dice = evaluate(mscmr_combined_model, mscmr_test_loader, mscmr_combined_loss_fn, device)
print("MSCMR COMBINED TEST - Mean Dice:", f"{test_mean_dice.item():.4f} ({test_mean_dice.item()*100:.2f}%)")

# Visualizations


In [ ]:
visualize_predictions(
    mscmr_combined_model,
    mscmr_test_dataset,
    device,
    num_images=40,
    num_classes=4,
    class_colors={1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1]},
    dataset_name="MS-CMRSeg Combined GNN+Transformer"
)

# Metrics


In [ ]:
# ============================================================
# IOU METRIC FUNCTION
# ============================================================

def evaluate_iou(model, loader, device, num_classes=4):

    model.eval()

    intersections = torch.zeros(num_classes - 1, device=device)
    unions = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            for class_id in range(1, num_classes):

                pred_class = (predictions == class_id)
                target_class = (masks == class_id)

                intersection = (pred_class & target_class).sum().float()
                union = (pred_class | target_class).sum().float()

                intersections[class_id - 1] += intersection
                unions[class_id - 1] += union

    iou_scores = (intersections + 1e-6) / (unions + 1e-6)
    mean_iou = iou_scores.mean()

    return mean_iou, iou_scores


print("IoU metric function ready!")

# IOU FOR ACDC COMBINED MODE

In [ ]:
# ============================================================
# IOU FOR ACDC TRANSUNET
# ============================================================

resnet_model.load_state_dict(torch.load(best_resnet_path, map_location=device))

mean_iou, iou_scores = evaluate_iou(resnet_model, test_loader, device)

print("ACDC TRANSUNET - IOU RESULTS")
print(f"RV IoU: {iou_scores[0].item():.4f} ({iou_scores[0].item() * 100:.2f}%)")
print(f"Myocardium IoU: {iou_scores[1].item():.4f} ({iou_scores[1].item() * 100:.2f}%)")
print(f"LV IoU: {iou_scores[2].item():.4f} ({iou_scores[2].item() * 100:.2f}%)")
print(f"Mean IoU: {mean_iou.item():.4f} ({mean_iou.item() * 100:.2f}%)")

Distance

In [ ]:
# ============================================================
# HD95 METRIC FUNCTION
# ============================================================

from scipy.ndimage import binary_erosion, distance_transform_edt


def hd95_single_class(pred_mask, true_mask):

    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)

    if not pred_mask.any() and not true_mask.any():
        return 0.0

    if not pred_mask.any() or not true_mask.any():
        return np.nan

    pred_boundary = pred_mask ^ binary_erosion(pred_mask)
    true_boundary = true_mask ^ binary_erosion(true_mask)

    distance_to_true = distance_transform_edt(~true_boundary)
    distance_to_pred = distance_transform_edt(~pred_boundary)

    pred_to_true = distance_to_true[pred_boundary]
    true_to_pred = distance_to_pred[true_boundary]

    all_distances = np.concatenate([pred_to_true, true_to_pred])

    return np.percentile(all_distances, 95)


def evaluate_hd95(model, loader, device, num_classes=4):

    model.eval()

    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):

                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]

                for class_id in range(1, num_classes):

                    pred_class = (pred == class_id)
                    true_class = (true == class_id)

                    hd = hd95_single_class(pred_class, true_class)

                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }

    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])

    return overall_mean_hd95, mean_hd95_per_class


print("HD95 metric function ready!")

# HD95 FOR ACDC TRANSUNET

In [ ]:


overall_hd95, hd95_per_class = evaluate_hd95(resnet_model, test_loader, device)

print("ACDC TRANSUNET - HD95 RESULTS (in pixels)")
print(f"RV HD95: {hd95_per_class[1]:.4f} pixels")
print(f"Myocardium HD95: {hd95_per_class[2]:.4f} pixels")
print(f"LV HD95: {hd95_per_class[3]:.4f} pixels")
print(f"Mean HD95: {overall_hd95:.4f} pixels")

In [ ]:
# ============================================================
# LARGEST CONNECTED COMPONENT POST-PROCESSING
# ============================================================

from scipy.ndimage import label


def keep_largest_component(binary_mask):

    labeled_array, num_features = label(binary_mask)

    if num_features == 0:
        return binary_mask

    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0

    largest_component_label = component_sizes.argmax()

    return labeled_array == largest_component_label


def evaluate_hd95_with_postprocessing(model, loader, device, num_classes=4):

    model.eval()

    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):

                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]

                for class_id in range(1, num_classes):

                    pred_class = (pred == class_id)
                    true_class = (true == class_id)

                    pred_class_cleaned = keep_largest_component(pred_class)

                    hd = hd95_single_class(pred_class_cleaned, true_class)

                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }

    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])

    return overall_mean_hd95, mean_hd95_per_class


overall_hd95_clean, hd95_per_class_clean = evaluate_hd95_with_postprocessing(resnet_model, test_loader, device)

print("ACDC TRANSUNET - HD95 WITH POST-PROCESSING (in pixels)")
print(f"RV HD95: {hd95_per_class_clean[1]:.4f} pixels")
print(f"Myocardium HD95: {hd95_per_class_clean[2]:.4f} pixels")
print(f"LV HD95: {hd95_per_class_clean[3]:.4f} pixels")
print(f"Mean HD95: {overall_hd95_clean:.4f} pixels")

In [ ]:
# ============================================================
# ROC-AUC CURVE PLOTTING
# ============================================================

from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt


def plot_roc_curve(model, loader, device, num_classes=4, class_names=None, title="ROC Curve"):

    model.eval()

    if class_names is None:
        class_names = [f"Class {i}" for i in range(1, num_classes)]

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)

            for class_id in range(1, num_classes):

                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()

                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    plt.figure(figsize=(7, 6))

    for i, class_id in enumerate(range(1, num_classes)):

        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])

        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        auc_value = roc_auc_score(true_labels, probabilities)

        plt.plot(fpr, tpr, linewidth=2, label=f"{class_names[i]} (AUC = {auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random guess")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


print("ROC curve plotting function ready!")

In [ ]:
plot_roc_curve(
    resnet_model,
    test_loader,
    device,
    num_classes=4,
    class_names=["RV", "Myocardium", "LV"],
    title="ACDC TransUNet - ROC Curve"
)

# ROC-AUC - MS-CMRSEG TRANSUNET

In [ ]:


mscmr_model.load_state_dict(torch.load(best_mscmr_path, map_location=device))

mean_auc, auc_scores = evaluate_roc_auc(mscmr_model, mscmr_test_loader, device)

print("MS-CMRSEG TRANSUNET - ROC-AUC RESULTS")
print(f"Class1 AUC: {auc_scores[1]:.4f} ({auc_scores[1] * 100:.2f}%)")
print(f"Class2 AUC: {auc_scores[2]:.4f} ({auc_scores[2] * 100:.2f}%)")
print(f"Class3 AUC: {auc_scores[3]:.4f} ({auc_scores[3] * 100:.2f}%)")
print(f"Mean AUC: {mean_auc:.4f} ({mean_auc * 100:.2f}%)")

plot_roc_curve(
    mscmr_model,
    mscmr_test_loader,
    device,
    num_classes=4,
    class_names=["Class1", "Class2", "Class3"],
    title="MS-CMRSeg TransUNet - ROC Curve"
)

# COMBINED ACDC + MS-CMRSEG DATASET

In [ ]:


class CombinedCardiacDataset(Dataset):

    def __init__(self, items, image_size=256, augment=False):

        # items is a list of tuples:
        # ("acdc", image_path, mask_path) or ("mscmr", file_path, None)

        self.items = items
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.items)


    def __getitem__(self, idx):

        source, path_a, path_b = self.items[idx]

        if source == "acdc":

            image = np.array(Image.open(path_a), dtype=np.float32)
            mask = np.load(path_b).astype(np.int64)

        else:

            with h5py.File(path_a, "r") as f:
                image = f["image"][:].astype(np.float32)
                mask = f["label"][:].astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0), size=(self.image_size, self.image_size),
            mode="bilinear", align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0), size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        if source == "acdc":
            image = image / 255.0
        else:
            image_min = image.min()
            image_max = image.max()
            if image_max > image_min:
                image = (image - image_min) / (image_max - image_min)

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

        image = image.repeat(3, 1, 1)
        mask = mask.squeeze(0)

        return image, mask


# Build combined item lists
combined_train_items = [("acdc", p[0], p[1]) for p in train_files] + \
                        [("mscmr", p, None) for p in mscmr_train_files]

combined_val_items = [("acdc", p[0], p[1]) for p in val_files] + \
                      [("mscmr", p, None) for p in mscmr_val_files]

combined_test_items = [("acdc", p[0], p[1]) for p in test_files]

random.shuffle(combined_train_items)

combined_train_dataset = CombinedCardiacDataset(combined_train_items, image_size=256, augment=True)
combined_val_dataset = CombinedCardiacDataset(combined_val_items, image_size=256, augment=False)

print("Combined dataset created!")
print("Combined training samples:", len(combined_train_dataset))
print("Combined validation samples:", len(combined_val_dataset))

In [ ]:


combined_train_loader = DataLoader(combined_train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
combined_val_loader = DataLoader(combined_val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Combined DataLoaders created!")
print("Training batches:", len(combined_train_loader))
print("Validation batches:", len(combined_val_loader))



# COMBINED DATASET MODEL, LOSS, OPTIMIZER


joint_model = MSCMRCombinedUNet(in_channels=3, out_channels=4).to(device)

joint_loss_fn = ACDCLoss(num_classes=4).to(device)

joint_optimizer = optim.AdamW(joint_model.parameters(), lr=1e-4, weight_decay=1e-4)

joint_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    joint_optimizer, mode="max", factor=0.5, patience=8
)

print("Joint model created!")
print("Total parameters:", sum(p.numel() for p in joint_model.parameters()))


# TRAIN ON COMBINED DATASET


num_epochs = 150

best_joint_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_joint_path = "/content/best_joint_combined_dataset_model.pth"

print()
print("Starting joint dataset training...")
print("Training images:", len(combined_train_loader.dataset))
print("Validation images:", len(combined_val_loader.dataset))
print()

for epoch in range(num_epochs):

    joint_model.train()

    total_train_loss = 0.0

    for images, masks in combined_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        joint_optimizer.zero_grad(set_to_none=True)

        outputs = joint_model(images)
        loss = joint_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(joint_model.parameters(), max_norm=1.0)

        joint_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(combined_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(joint_model, combined_val_loader, joint_loss_fn, device)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_joint_dice:

        best_joint_dice = current_dice
        epochs_without_improvement = 0

        torch.save(joint_model.state_dict(), best_joint_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    joint_scheduler.step(current_dice)

    current_lr = joint_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("JOINT DATASET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_joint_dice:.4f}")
print("Best model saved to:", best_joint_path)

Test Results

In [ ]:
combined_test_dataset = CombinedCardiacDataset(combined_test_items, image_size=256, augment=False)
combined_test_loader = DataLoader(combined_test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

joint_model.load_state_dict(torch.load(best_joint_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(joint_model, combined_test_loader, joint_loss_fn, device)

print("JOINT MODEL FINAL TEST RESULTS (ACDC TEST SET)")
print(f"Test Loss: {test_loss:.4f}")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
visualize_predictions(
    joint_model,
    test_dataset,
    device,
    num_images=40,
    num_classes=4,
    dataset_name="Joint ACDC+MS-CMRSeg Model (tested on ACDC)"
)

 Metrics

In [ ]:
# ============================================================
# IOU WITH LARGEST CONNECTED COMPONENT POST-PROCESSING
# ============================================================

from scipy.ndimage import label


def keep_largest_component(binary_mask):

    labeled_array, num_features = label(binary_mask)

    if num_features == 0:
        return binary_mask

    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0

    largest_component_label = component_sizes.argmax()

    return labeled_array == largest_component_label


def evaluate_iou_with_postprocessing(model, loader, device, num_classes=4):

    model.eval()

    intersections = torch.zeros(num_classes - 1, device=device)
    unions = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            predictions_np = predictions.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for batch_idx in range(predictions_np.shape[0]):

                for class_id in range(1, num_classes):

                    pred_class = (predictions_np[batch_idx] == class_id)
                    target_class = (masks_np[batch_idx] == class_id)

                    pred_class_cleaned = keep_largest_component(pred_class)

                    intersection = np.logical_and(pred_class_cleaned, target_class).sum()
                    union = np.logical_or(pred_class_cleaned, target_class).sum()

                    intersections[class_id - 1] += intersection
                    unions[class_id - 1] += union

    iou_scores = (intersections + 1e-6) / (unions + 1e-6)
    mean_iou = iou_scores.mean()

    return mean_iou, iou_scores


print("IoU with post-processing function ready!")

HD95

In [ ]:
# ============================================================
# HD95 WITH LARGEST CONNECTED COMPONENT POST-PROCESSING
# ============================================================

from scipy.ndimage import binary_erosion, distance_transform_edt


def hd95_single_class(pred_mask, true_mask):

    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)

    if not pred_mask.any() and not true_mask.any():
        return 0.0

    if not pred_mask.any() or not true_mask.any():
        return np.nan

    pred_boundary = pred_mask ^ binary_erosion(pred_mask)
    true_boundary = true_mask ^ binary_erosion(true_mask)

    distance_to_true = distance_transform_edt(~true_boundary)
    distance_to_pred = distance_transform_edt(~pred_boundary)

    pred_to_true = distance_to_true[pred_boundary]
    true_to_pred = distance_to_pred[true_boundary]

    all_distances = np.concatenate([pred_to_true, true_to_pred])

    return np.percentile(all_distances, 95)


def evaluate_hd95_with_postprocessing(model, loader, device, num_classes=4):

    model.eval()

    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):

                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]

                for class_id in range(1, num_classes):

                    pred_class = (pred == class_id)
                    true_class = (true == class_id)

                    pred_class_cleaned = keep_largest_component(pred_class)

                    hd = hd95_single_class(pred_class_cleaned, true_class)

                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }

    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])

    return overall_mean_hd95, mean_hd95_per_class


print("HD95 with post-processing function ready!")

In [ ]:
# ROC-AUC METRIC FUNCTION

In [ ]:


from sklearn.metrics import roc_auc_score


def evaluate_roc_auc(model, loader, device, num_classes=4):

    model.eval()

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)

            for class_id in range(1, num_classes):

                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()

                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    auc_scores = {}

    for class_id in range(1, num_classes):

        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])

        auc_scores[class_id] = roc_auc_score(true_labels, probabilities)

    mean_auc = np.mean(list(auc_scores.values()))

    return mean_auc, auc_scores


print("ROC-AUC metric function ready!")

In [ ]:


from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt


def plot_roc_curve(model, loader, device, num_classes=4, class_names=None, title="ROC Curve"):

    model.eval()

    if class_names is None:
        class_names = [f"Class {i}" for i in range(1, num_classes)]

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)

            for class_id in range(1, num_classes):

                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()

                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    plt.figure(figsize=(7, 6))

    for i, class_id in enumerate(range(1, num_classes)):

        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])

        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        auc_value = roc_auc_score(true_labels, probabilities)

        plt.plot(fpr, tpr, linewidth=2, label=f"{class_names[i]} (AUC = {auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random guess")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


print("ROC curve plotting function ready!")

# FULL METRICS - JOINT MODEL (ACDC TEST SET)

In [ ]:
# ============================================================
# FULL METRICS - JOINT MODEL (ACDC TEST SET)
# ============================================================

mean_iou_clean, iou_scores_clean = evaluate_iou_with_postprocessing(joint_model, combined_test_loader, device)

print("JOINT MODEL - IOU WITH POST-PROCESSING")
print(f"Class1 IoU: {iou_scores_clean[0].item():.4f} ({iou_scores_clean[0].item() * 100:.2f}%)")
print(f"Class2 IoU: {iou_scores_clean[1].item():.4f} ({iou_scores_clean[1].item() * 100:.2f}%)")
print(f"Class3 IoU: {iou_scores_clean[2].item():.4f} ({iou_scores_clean[2].item() * 100:.2f}%)")
print(f"Mean IoU: {mean_iou_clean.item():.4f} ({mean_iou_clean.item() * 100:.2f}%)")
print()

overall_hd95_clean, hd95_per_class_clean = evaluate_hd95_with_postprocessing(joint_model, combined_test_loader, device)

print("JOINT MODEL - HD95 WITH POST-PROCESSING (in pixels)")
print(f"Class1 HD95: {hd95_per_class_clean[1]:.4f} pixels")
print(f"Class2 HD95: {hd95_per_class_clean[2]:.4f} pixels")
print(f"Class3 HD95: {hd95_per_class_clean[3]:.4f} pixels")
print(f"Mean HD95: {overall_hd95_clean:.4f} pixels")
print()

mean_auc, auc_scores = evaluate_roc_auc(joint_model, combined_test_loader, device)

print("JOINT MODEL - ROC-AUC RESULTS")
print(f"Class1 AUC: {auc_scores[1]:.4f} ({auc_scores[1] * 100:.2f}%)")
print(f"Class2 AUC: {auc_scores[2]:.4f} ({auc_scores[2] * 100:.2f}%)")
print(f"Class3 AUC: {auc_scores[3]:.4f} ({auc_scores[3] * 100:.2f}%)")
print(f"Mean AUC: {mean_auc:.4f} ({mean_auc * 100:.2f}%)")

plot_roc_curve(
    joint_model,
    combined_test_loader,
    device,
    num_classes=4,
    class_names=["RV", "Myocardium", "LV"],
    title="Joint ACDC+MS-CMRSeg Model - ROC Curve"
)

In [ ]:
# ============================================================
# SSIM - ALL THREE MS-CMRSEG MODELS
# ============================================================

mscmr_model.load_state_dict(torch.load(best_mscmr_path, map_location=device))
mean_ssim_mscmr_transunet = evaluate_ssim(mscmr_model, mscmr_test_loader, device, num_classes=4)

print("MS-CMRSEG TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_mscmr_transunet:.4f}")
print()

mscmr_graph_model.load_state_dict(torch.load(best_mscmr_graph_path, map_location=device))
mean_ssim_mscmr_graph = evaluate_ssim(mscmr_graph_model, mscmr_test_loader, device, num_classes=4)

print("MS-CMRSEG GRAPH U-NET - SSIM")
print(f"Mean SSIM: {mean_ssim_mscmr_graph:.4f}")
print()

mscmr_combined_model.load_state_dict(torch.load(best_mscmr_combined_path, map_location=device))
mean_ssim_mscmr_combined = evaluate_ssim(mscmr_combined_model, mscmr_test_loader, device, num_classes=4)

print("MS-CMRSEG COMBINED MODEL - SSIM")
print(f"Mean SSIM: {mean_ssim_mscmr_combined:.4f}")

OCT Dataset

In [ ]:
# ============================================================
# DOWNLOAD OCT DATASET
# ============================================================

oct_path = kagglehub.dataset_download(
    "zeeshanahmed13/intraretinal-cystoid-fluid"
)

print("OCT dataset downloaded!")
print("Dataset path:", oct_path)

# EXPLORE OCT DATASET STRUCTURE

In [ ]:


for root, dirs, files in os.walk(oct_path):

    level = root.replace(oct_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

# BUILD OCT IMAGE/MASK PAIR LIST

In [ ]:


oct_data_root = os.path.join(oct_path, "2021-training-data-ZA", "2021-training-data-ZA")

oct_pairs = []

for case_folder in os.listdir(oct_data_root):

    case_path = os.path.join(oct_data_root, case_folder)

    images_folder = os.path.join(case_path, "images")
    masks_folder = os.path.join(case_path, "masks")

    if not (os.path.isdir(images_folder) and os.path.isdir(masks_folder)):
        continue

    image_files = os.listdir(images_folder)
    mask_files = os.listdir(masks_folder)

    if len(image_files) == 0 or len(mask_files) == 0:
        continue

    image_path = os.path.join(images_folder, image_files[0])
    mask_path = os.path.join(masks_folder, mask_files[0])

    oct_pairs.append((image_path, mask_path))

print("Total OCT image/mask pairs found:", len(oct_pairs))
print("First pair:", oct_pairs[0])

# INSPECT ONE OCT IMAGE/MASK PAI

In [ ]:

sample_image_path, sample_mask_path = oct_pairs[0]

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print("Image path:", sample_image_path)
print("Image size (W, H):", sample_image.size)
print("Image mode:", sample_image.mode)
print()

print("Mask path:", sample_mask_path)
print("Mask size (W, H):", sample_mask.size)
print("Mask mode:", sample_mask.mode)

mask_array = np.array(sample_mask)
print("Mask array shape:", mask_array.shape)
print("Mask array dtype:", mask_array.dtype)
print("Unique values in mask:", np.unique(mask_array))

# VERIFY MASK VALUES ACROSS MANY SAMPLES

In [ ]:


all_mask_values = set()
inconsistent_sizes = []

for image_path, mask_path in oct_pairs[:100]:

    mask = Image.open(mask_path)
    mask_array = np.array(mask)

    all_mask_values.update(np.unique(mask_array).tolist())

    image = Image.open(image_path)

    if image.size != mask.size:
        inconsistent_sizes.append((image_path, image.size, mask.size))

print("Unique mask values across first 100 samples:", sorted(all_mask_values))
print("Number of image/mask size mismatches:", len(inconsistent_sizes))

if inconsistent_sizes:
    print("First mismatch:", inconsistent_sizes[0])

# FIND THE INCONSISTENT MASK FILE

In [ ]:


for image_path, mask_path in oct_pairs[:100]:

    mask = Image.open(mask_path)

    if mask.mode != "L":
        print("Mask path:", mask_path)
        print("Mask mode:", mask.mode)
        print()

# VERIFY THE FIX HANDLES BOTH MASK FORMATS

In [ ]:


test_paths = [
    "/root/.cache/kagglehub/datasets/zeeshanahmed13/intraretinal-cystoid-fluid/versions/3/2021-training-data-ZA/2021-training-data-ZA/DME327/masks/MASK-DME327.PNG",
    "/root/.cache/kagglehub/datasets/zeeshanahmed13/intraretinal-cystoid-fluid/versions/3/2021-training-data-ZA/2021-training-data-ZA/942DME_F/masks/942MASK_DME_F.PNG"
]

for path in test_paths:

    raw_mask = Image.open(path)
    converted_mask = raw_mask.convert("L")

    converted_array = np.array(converted_mask)

    print("Path:", path)
    print("Original mode:", raw_mask.mode)
    print("Converted mode:", converted_mask.mode)
    print("Unique values after convert('L'):", np.unique(converted_array))
    print()

# OCT DATASET

In [ ]:


class OCTDataset(Dataset):

    def __init__(self, pairs, image_size=256, augment=False):

        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.pairs)


    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask, dtype=np.float32)
        mask = (mask > 127).astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.5:
                dx = int(random.uniform(-0.05, 0.05) * self.image_size)
                dy = int(random.uniform(-0.05, 0.05) * self.image_size)
                scale = random.uniform(0.95, 1.05)
                image = TF.affine(image, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.affine(mask, angle=0, translate=[dx, dy], scale=scale, shear=[0.0, 0.0], interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


# ============================================================
# 70/15/15 SPLIT
# ============================================================

random.shuffle(oct_pairs)

num_pairs = len(oct_pairs)
num_train = int(num_pairs * 0.70)
num_val = int(num_pairs * 0.15)

oct_train_pairs = oct_pairs[:num_train]
oct_val_pairs = oct_pairs[num_train:num_train + num_val]
oct_test_pairs = oct_pairs[num_train + num_val:]

oct_train_dataset = OCTDataset(oct_train_pairs, image_size=256, augment=True)
oct_val_dataset = OCTDataset(oct_val_pairs, image_size=256, augment=False)
oct_test_dataset = OCTDataset(oct_test_pairs, image_size=256, augment=False)

print("OCT datasets created!")
print("Training samples:", len(oct_train_dataset))
print("Validation samples:", len(oct_val_dataset))
print("Testing samples:", len(oct_test_dataset))

# OCT LOSS (binary version, no myocardium-style term)

In [ ]:


class OCTLoss(nn.Module):

    def __init__(self, num_classes=2, gamma=2.0, smooth=1e-5):
        super().__init__()

        self.num_classes = num_classes
        self.gamma = gamma
        self.smooth = smooth

        # 0 = Background, 1 = Fluid

        self.register_buffer(
            "class_weights",
            torch.tensor([0.30, 1.00], dtype=torch.float32)
        )

        self.dice_loss = MulticlassDiceLoss(num_classes=num_classes)


    def forward(self, logits, targets):

        ce_loss = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")

        pt = torch.exp(-ce_loss)
        focal_loss = (((1 - pt) ** self.gamma) * ce_loss).mean()

        dice_loss = self.dice_loss(logits, targets)

        total_loss = (0.40 * focal_loss) + (1.00 * dice_loss)

        return total_loss


oct_loss_fn = OCTLoss(num_classes=2).to(device)

print("OCT Loss ready!")
print("Class weights: Background 0.30, Fluid 1.00")


# ============================================================
# OCT DATALOADERS
# ============================================================

oct_train_loader = DataLoader(oct_train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
oct_val_loader = DataLoader(oct_val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
oct_test_loader = DataLoader(oct_test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("OCT DataLoaders created!")
print("Training batches:", len(oct_train_loader))
print("Validation batches:", len(oct_val_loader))
print("Testing batches:", len(oct_test_loader))

OCT DATASET — TRANSUNET (BINARY FLUID SEGMENTATION)

In [ ]:


oct_model = TransUNet(in_channels=3, out_channels=2).to(device)

print("OCT TransUNet created!")
print("Total parameters:", sum(p.numel() for p in oct_model.parameters()))


# ============================================================
# OPTIMIZER FOR OCT TRANSUNET
# ============================================================

oct_optimizer = optim.AdamW(oct_model.parameters(), lr=1e-4, weight_decay=1e-4)

oct_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    oct_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


# ============================================================
# TRAIN OCT TRANSUNET
# ============================================================

num_epochs = 150

best_oct_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_oct_path = "/content/best_oct_transunet.pth"

print()
print("Starting OCT TransUNet training...")
print("Training images:", len(oct_train_loader.dataset))
print("Validation images:", len(oct_val_loader.dataset))
print()

for epoch in range(num_epochs):

    oct_model.train()

    total_train_loss = 0.0

    for images, masks in oct_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        oct_optimizer.zero_grad(set_to_none=True)

        outputs = oct_model(images)
        loss = oct_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(oct_model.parameters(), max_norm=1.0)

        oct_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(oct_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(oct_model, oct_val_loader, oct_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_oct_dice:

        best_oct_dice = current_dice
        epochs_without_improvement = 0

        torch.save(oct_model.state_dict(), best_oct_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    oct_scheduler.step(current_dice)

    current_lr = oct_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Fluid Dice: {current_dice:.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("OCT TRANSUNET TRAINING COMPLETE")
print("Best Validation Fluid Dice:", f"{best_oct_dice:.4f}")
print("Best model saved to:", best_oct_path)

# FINAL TEST - OCT TRANSUNET

In [ ]:


oct_model.load_state_dict(torch.load(best_oct_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(oct_model, oct_test_loader, oct_loss_fn, device, num_classes=2)

print("OCT TRANSUNET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Fluid Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

# GRAPH U GNN-NET FOR OCT

In [ ]:
# ============================================================
# GRAPH U-NET FOR OCT
# ============================================================

oct_graph_model = MSCMRGraphUNet(in_channels=3, out_channels=2).to(device)

print("OCT Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in oct_graph_model.parameters()))


# ============================================================
# LOSS AND OPTIMIZER FOR OCT GRAPH U-NET
# ============================================================

oct_graph_loss_fn = OCTLoss(num_classes=2).to(device)

oct_graph_optimizer = optim.AdamW(oct_graph_model.parameters(), lr=1e-4, weight_decay=1e-4)

oct_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    oct_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


# ============================================================
# TRAIN OCT GRAPH U-NET
# ============================================================

num_epochs = 150

best_oct_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_oct_graph_path = "/content/best_oct_graph_unet.pth"

print()
print("Starting OCT Graph U-Net training...")
print("Training images:", len(oct_train_loader.dataset))
print("Validation images:", len(oct_val_loader.dataset))
print()

for epoch in range(num_epochs):

    oct_graph_model.train()

    total_train_loss = 0.0

    for images, masks in oct_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        oct_graph_optimizer.zero_grad(set_to_none=True)

        outputs = oct_graph_model(images)
        loss = oct_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(oct_graph_model.parameters(), max_norm=1.0)

        oct_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(oct_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(oct_graph_model, oct_val_loader, oct_graph_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_oct_graph_dice:

        best_oct_graph_dice = current_dice
        epochs_without_improvement = 0

        torch.save(oct_graph_model.state_dict(), best_oct_graph_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    oct_graph_scheduler.step(current_dice)

    current_lr = oct_graph_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Fluid Dice: {current_dice:.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("OCT GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Fluid Dice:", f"{best_oct_graph_dice:.4f}")
print("Best model saved to:", best_oct_graph_path)

Test Results

In [ ]:


oct_graph_model.load_state_dict(torch.load(best_oct_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(oct_graph_model, oct_test_loader, oct_graph_loss_fn, device, num_classes=2)

print("OCT GRAPH U-NET FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Fluid Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

GNN + Transformer

In [ ]:


oct_combined_model = MSCMRCombinedUNet(in_channels=3, out_channels=2).to(device)

print("OCT Combined GNN+Transformer created!")
print("Total parameters:", sum(p.numel() for p in oct_combined_model.parameters()))



# LOSS AND OPTIMIZER FOR OCT COMBINED MODEL


oct_combined_loss_fn = OCTLoss(num_classes=2).to(device)

oct_combined_optimizer = optim.AdamW(oct_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

oct_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    oct_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


# TRAIN OCT COMBINED MODEL


num_epochs = 150

best_oct_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_oct_combined_path = "/content/best_oct_combined_unet.pth"

print()
print("Starting OCT Combined model training...")
print("Training images:", len(oct_train_loader.dataset))
print("Validation images:", len(oct_val_loader.dataset))
print()

for epoch in range(num_epochs):

    oct_combined_model.train()

    total_train_loss = 0.0

    for images, masks in oct_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        oct_combined_optimizer.zero_grad(set_to_none=True)

        outputs = oct_combined_model(images)
        loss = oct_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(oct_combined_model.parameters(), max_norm=1.0)

        oct_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(oct_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(oct_combined_model, oct_val_loader, oct_combined_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_oct_combined_dice:

        best_oct_combined_dice = current_dice
        epochs_without_improvement = 0

        torch.save(oct_combined_model.state_dict(), best_oct_combined_path)

        new_best = True

    else:
        epochs_without_improvement += 1

    oct_combined_scheduler.step(current_dice)

    current_lr = oct_combined_optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Fluid Dice: {current_dice:.4f} | "
        f"LR: {current_lr:.8f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print()
        print("Early stopping triggered!")
        break

print()
print("OCT COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Fluid Dice:", f"{best_oct_combined_dice:.4f}")
print("Best model saved to:", best_oct_combined_path)

# FINAL TEST - OCT COMBINED MODEL

In [ ]:


oct_combined_model.load_state_dict(torch.load(best_oct_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(oct_combined_model, oct_test_loader, oct_combined_loss_fn, device, num_classes=2)

print("OCT COMBINED MODEL FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Fluid Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

In [ ]:
visualize_predictions(
    oct_combined_model,
    oct_test_dataset,
    device,
    num_images=40,
    num_classes=2,
    class_colors={1: [1, 0, 0]},
    class_names={1: "Fluid"},
    dataset_name="OCT Combined GNN+Transformer"
)

In [ ]:
# ============================================================
# VISUALIZATION FUNCTION - UPDATED WITH FLEXIBLE LEGEND
# ============================================================

def visualize_predictions(model, dataset, device, num_images=40, num_classes=4, class_colors=None, class_names=None, dataset_name="Dataset"):

    model.eval()

    if class_colors is None:
        class_colors = {1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1]}

    if class_names is None:
        class_names = {1: "Class 1", 2: "Class 2", 3: "Class 3"}

    num_images = min(num_images, len(dataset))

    cols = 5
    rows = (num_images * 3 + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    plot_idx = 0

    with torch.no_grad():

        for idx in range(num_images):

            image, true_mask = dataset[idx]

            input_image = image.unsqueeze(0).to(device)

            output = model(input_image)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            true_mask_np = true_mask.numpy()

            dice = compute_single_image_dice(prediction, true_mask_np, num_classes)

            display_image = image[0].cpu().numpy()
            display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min() + 1e-8)

            gt_overlay = np.stack([display_image] * 3, axis=-1)
            pred_overlay = np.stack([display_image] * 3, axis=-1)

            for class_id, color in class_colors.items():

                gt_mask = (true_mask_np == class_id)
                pred_mask = (prediction == class_id)

                for c in range(3):
                    gt_overlay[..., c] = np.where(gt_mask, color[c], gt_overlay[..., c])
                    pred_overlay[..., c] = np.where(pred_mask, color[c], pred_overlay[..., c])

            axes[plot_idx].imshow(display_image, cmap="gray")
            axes[plot_idx].set_title("Input", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(gt_overlay)
            axes[plot_idx].set_title("Ground Truth", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(pred_overlay)
            axes[plot_idx].set_title(f"Prediction (Dice: {dice:.3f})", fontsize=9, fontweight="bold")
            axes[plot_idx].axis("off")
            plot_idx += 1

    for remaining_idx in range(plot_idx, len(axes)):
        axes[remaining_idx].axis("off")

    legend_patches = [
        mpatches.Patch(color=color, label=class_names[class_id])
        for class_id, color in class_colors.items()
    ]

    fig.legend(handles=legend_patches, loc="upper center", ncol=len(legend_patches), fontsize=11, bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{dataset_name} - Predictions with Per-Image Dice Score", fontsize=13, y=1.03)

    plt.tight_layout()
    plt.show()


print("Visualization function updated!")

SSIM Metrics

In [ ]:


from skimage.metrics import structural_similarity as ssim


def evaluate_ssim(model, loader, device, num_classes=4):

    model.eval()

    ssim_scores = []

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):

                pred = predictions[batch_idx].astype(np.float32)
                true = masks_cpu[batch_idx].astype(np.float32)

                score = ssim(pred, true, data_range=num_classes - 1)

                ssim_scores.append(score)

    mean_ssim = np.mean(ssim_scores)

    return mean_ssim


print("SSIM metric function ready!")

# SSIM - ALL THREE OCT MODELS

In [ ]:


oct_model.load_state_dict(torch.load(best_oct_path, map_location=device))
mean_ssim_transunet = evaluate_ssim(oct_model, oct_test_loader, device, num_classes=2)

print("OCT TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_transunet:.4f}")
print()

oct_graph_model.load_state_dict(torch.load(best_oct_graph_path, map_location=device))
mean_ssim_graph = evaluate_ssim(oct_graph_model, oct_test_loader, device, num_classes=2)

print("OCT GRAPH U-NET - SSIM")
print(f"Mean SSIM: {mean_ssim_graph:.4f}")
print()

oct_combined_model.load_state_dict(torch.load(best_oct_combined_path, map_location=device))
mean_ssim_combined = evaluate_ssim(oct_combined_model, oct_test_loader, device, num_classes=2)

print("OCT COMBINED MODEL - SSIM")
print(f"Mean SSIM: {mean_ssim_combined:.4f}")

# FUNDUS IMAGES — DRISHTI-GS1 DATASET (OPTIC DISC + OPTIC CUP)

Downloading the Dataset


In [ ]:


!pip install -q kagglehub

import kagglehub

drishti_path = kagglehub.dataset_download(
    "lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation"
)

print("DRISHTI-GS1 dataset downloaded!")
print("Dataset path:", drishti_path)

In [ ]:


for root, dirs, files in os.walk(drishti_path):

    level = root.replace(drishti_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

# BUILD DRISHTI IMAGE/MASK TRIOS AND INSPECT ONE SAMPLE

In [ ]:


training_images_root = os.path.join(drishti_path, "Training-20211018T055246Z-001", "Training", "Images")
training_gt_root = os.path.join(drishti_path, "Training-20211018T055246Z-001", "Training", "GT")

drishti_pairs = []

for category in ["GLAUCOMA", "NORMAL"]:

    category_folder = os.path.join(training_images_root, category)

    if not os.path.isdir(category_folder):
        continue

    for filename in os.listdir(category_folder):

        if not filename.endswith(".png"):
            continue

        image_id = filename.replace(".png", "")
        image_path = os.path.join(category_folder, filename)

        od_mask_path = os.path.join(training_gt_root, image_id, "SoftMap", f"{image_id}_ODsegSoftmap.png")
        cup_mask_path = os.path.join(training_gt_root, image_id, "SoftMap", f"{image_id}_cupsegSoftmap.png")

        if os.path.exists(od_mask_path) and os.path.exists(cup_mask_path):
            drishti_pairs.append((image_path, od_mask_path, cup_mask_path))

print("Total DRISHTI training trios found:", len(drishti_pairs))
print("First trio:", drishti_pairs[0])

sample_image_path, sample_od_path, sample_cup_path = drishti_pairs[0]

sample_image = Image.open(sample_image_path)
sample_od = Image.open(sample_od_path)
sample_cup = Image.open(sample_cup_path)

print()
print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("OD mask size:", sample_od.size, "mode:", sample_od.mode)
print("Cup mask size:", sample_cup.size, "mode:", sample_cup.mode)

od_array = np.array(sample_od)
cup_array = np.array(sample_cup)

print("OD mask unique values (sample):", np.unique(od_array)[:10], "... total unique:", len(np.unique(od_array)))
print("Cup mask unique values (sample):", np.unique(cup_array)[:10], "... total unique:", len(np.unique(cup_array)))

ADDITIONAL IMPORTS

In [ ]:
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader

# BUILD DRISHTI TRAIN AND TEST TRIO LISTS

In [ ]:


def build_drishti_pairs(images_root, gt_root, categories):

    pairs = []

    for category in categories:

        category_folder = os.path.join(images_root, category)

        if not os.path.isdir(category_folder):
            continue

        for filename in os.listdir(category_folder):

            if not filename.endswith(".png"):
                continue

            image_id = filename.replace(".png", "")
            image_path = os.path.join(category_folder, filename)

            od_mask_path = os.path.join(gt_root, image_id, "SoftMap", f"{image_id}_ODsegSoftmap.png")
            cup_mask_path = os.path.join(gt_root, image_id, "SoftMap", f"{image_id}_cupsegSoftmap.png")

            if os.path.exists(od_mask_path) and os.path.exists(cup_mask_path):
                pairs.append((image_path, od_mask_path, cup_mask_path))

    return pairs


training_images_root = os.path.join(drishti_path, "Training-20211018T055246Z-001", "Training", "Images")
training_gt_root = os.path.join(drishti_path, "Training-20211018T055246Z-001", "Training", "GT")

test_images_root = os.path.join(drishti_path, "Test-20211018T060000Z-001", "Test", "Images")
test_gt_root = os.path.join(drishti_path, "Test-20211018T060000Z-001", "Test", "Test_GT")

drishti_train_val_pairs = build_drishti_pairs(training_images_root, training_gt_root, ["GLAUCOMA", "NORMAL"])
drishti_test_pairs = build_drishti_pairs(test_images_root, test_gt_root, ["glaucoma", "normal"])

random.shuffle(drishti_train_val_pairs)

drishti_train_pairs = drishti_train_val_pairs[:40]
drishti_val_pairs = drishti_train_val_pairs[40:]

print("DRISHTI training pairs:", len(drishti_train_pairs))
print("DRISHTI validation pairs:", len(drishti_val_pairs))
print("DRISHTI test pairs (official):", len(drishti_test_pairs))


# ============================================================
# DRISHTI DATASET
# ============================================================

class DrishtiDataset(Dataset):

    def __init__(self, pairs, image_size=256, augment=False):

        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.pairs)


    def __getitem__(self, idx):

        image_path, od_path, cup_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        od_mask = Image.open(od_path).convert("L")
        od_mask = np.array(od_mask, dtype=np.float32)
        od_mask = (od_mask > 127).astype(np.int64)

        cup_mask = Image.open(cup_path).convert("L")
        cup_mask = np.array(cup_mask, dtype=np.float32)
        cup_mask = (cup_mask > 127).astype(np.int64)

        # Combine into one 3-class mask: 0=background, 1=disc, 2=cup
        # Cup overrides disc wherever they overlap

        combined_mask = np.zeros_like(od_mask, dtype=np.int64)
        combined_mask[od_mask == 1] = 1
        combined_mask[cup_mask == 1] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


drishti_train_dataset = DrishtiDataset(drishti_train_pairs, image_size=256, augment=True)
drishti_val_dataset = DrishtiDataset(drishti_val_pairs, image_size=256, augment=False)
drishti_test_dataset = DrishtiDataset(drishti_test_pairs, image_size=256, augment=False)

drishti_train_loader = DataLoader(drishti_train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
drishti_val_loader = DataLoader(drishti_val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
drishti_test_loader = DataLoader(drishti_test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("DRISHTI DataLoaders created!")
print("Training batches:", len(drishti_train_loader))
print("Validation batches:", len(drishti_val_loader))
print("Testing batches:", len(drishti_test_loader))

In [ ]:
# ============================================================
# REBUILD ALL SHARED MODEL COMPONENTS (FRESH RUNTIME)
# ============================================================

import torch.nn as nn
import torch.optim as optim

!pip install -q torch-geometric
from torch_geometric.nn import SAGEConv


class ResidualConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act = nn.GELU()

        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)


class TransformerBlock(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, num_tokens=256):
        super().__init__()

        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=0.1, activation="gelu", norm_first=True, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.shape[1], :]
        x = self.transformer(x)
        return self.norm(x)


class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = ResidualConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class TransUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=256)
        self.transformer_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


def create_grid_edges(height, width):

    edges = []
    for row in range(height):
        for col in range(width):
            node = row * width + col
            if col < width - 1:
                right = node + 1
                edges.append([node, right])
                edges.append([right, node])
            if row < height - 1:
                down = node + width
                edges.append([node, down])
                edges.append([down, node])

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index


class GraphBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, hidden_dim=512, num_layers=4):
        super().__init__()

        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()

        in_dim = feature_dim
        for _ in range(num_layers):
            self.layers.append(SAGEConv(in_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
            in_dim = hidden_dim

        self.act = nn.GELU()

    def forward(self, x, edge_index):

        batch_size = x.shape[0]
        outputs = []

        for b in range(batch_size):
            h = x[b]
            for layer, norm in zip(self.layers, self.norms):
                residual = h
                h = layer(h, edge_index)
                h = norm(h)
                h = self.act(h)
                if h.shape == residual.shape:
                    h = h + residual
            outputs.append(h)

        return torch.stack(outputs, dim=0)


class CombinedBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=64):
        super().__init__()

        self.graph_reasoning = GraphBottleneckBlock(feature_dim=feature_dim, hidden_dim=feature_dim, num_layers=num_graph_layers)
        self.transformer_reasoning = TransformerBlock(embed_dim=feature_dim, num_heads=num_heads, num_layers=num_transformer_layers, num_tokens=num_tokens)

    def forward(self, x, edge_index):
        x = self.graph_reasoning(x, edge_index)
        x = self.transformer_reasoning(x)
        return x


class MSCMRGraphUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)
        self.graph_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape
        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.graph_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


class MSCMRCombinedUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=4):
        super().__init__()

        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)

        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=256)
        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)
        self.combined_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)

        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))

        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))

        batch_size, channels, height, width = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.combined_refine(b)

        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)

        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


class MulticlassDiceLoss(nn.Module):

    def __init__(self, num_classes=4, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)
        total_dice_loss = 0.0
        foreground_classes = 0

        for class_idx in range(1, self.num_classes):
            pred_class = probabilities[:, class_idx]
            target_class = (targets == class_idx).float()
            intersection = (pred_class * target_class).sum()
            denominator = pred_class.sum() + target_class.sum()
            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
            total_dice_loss += (1.0 - dice)
            foreground_classes += 1

        return total_dice_loss / foreground_classes


class ACDCLoss(nn.Module):

    def __init__(self, num_classes=4, gamma=2.0, smooth=1e-5, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smooth = smooth

        if class_weights is None:
            class_weights = [0.20] + [1.00] * (num_classes - 1)

        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.dice_loss = MulticlassDiceLoss(num_classes=num_classes)

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = (((1 - pt) ** self.gamma) * ce_loss).mean()
        dice_loss = self.dice_loss(logits, targets)
        return (0.40 * focal_loss) + (1.00 * dice_loss)


def evaluate(model, loader, loss_fn, device, num_classes=4):

    model.eval()
    total_loss = 0.0

    intersections = torch.zeros(num_classes - 1, device=device)
    pred_sums = torch.zeros(num_classes - 1, device=device)
    target_sums = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            for class_id in range(1, num_classes):
                pred_class = (predictions == class_id).float()
                target_class = (masks == class_id).float()
                intersections[class_id - 1] += (pred_class * target_class).sum()
                pred_sums[class_id - 1] += pred_class.sum()
                target_sums[class_id - 1] += target_class.sum()

    dice_scores = (2.0 * intersections + 1e-5) / (pred_sums + target_sums + 1e-5)
    mean_dice = dice_scores.mean()
    average_loss = total_loss / len(loader.dataset)

    return average_loss, mean_dice, dice_scores


print("All shared model components rebuilt!")

In [ ]:
# ============================================================
# DRISHTI TRANSUNET - RETRAIN WITH BALANCED CLASS WEIGHTS
# ============================================================

drishti_model = TransUNet(in_channels=3, out_channels=3).to(device)

print("DRISHTI TransUNet (v2, balanced weights) created!")

drishti_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.20, 1.00, 1.00]).to(device)

drishti_optimizer = optim.AdamW(drishti_model.parameters(), lr=1e-4, weight_decay=1e-4)

drishti_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drishti_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drishti_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drishti_path = "/content/best_drishti_transunet_v2.pth"

print()
print("Starting DRISHTI TransUNet (balanced weights) training...")
print()

for epoch in range(num_epochs):

    drishti_model.train()

    total_train_loss = 0.0

    for images, masks in drishti_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drishti_optimizer.zero_grad(set_to_none=True)

        outputs = drishti_model(images)
        loss = drishti_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drishti_model.parameters(), max_norm=1.0)

        drishti_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drishti_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drishti_model, drishti_val_loader, drishti_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drishti_dice:
        best_drishti_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drishti_model.state_dict(), best_drishti_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drishti_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Disc: {val_dice[0].item():.4f} | "
        f"Cup: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRISHTI TRANSUNET (V2) TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_drishti_dice:.4f}")

Test Results



In [ ]:


drishti_model.load_state_dict(torch.load(best_drishti_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drishti_model, drishti_test_loader, drishti_loss_fn, device, num_classes=3)

print("DRISHTI TRANSUNET V2 FINAL TEST RESULTS")
print(f"Test Loss: {test_loss:.4f}")
print(f"Disc Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Cup Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# ============================================================
# REBUILD SPLIT: 45 TRAIN / 5 VALIDATION
# ============================================================

drishti_train_pairs = drishti_train_val_pairs[:45]
drishti_val_pairs = drishti_train_val_pairs[45:]

drishti_train_dataset = DrishtiDataset(drishti_train_pairs, image_size=256, augment=True)
drishti_val_dataset = DrishtiDataset(drishti_val_pairs, image_size=256, augment=False)

drishti_train_loader = DataLoader(drishti_train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
drishti_val_loader = DataLoader(drishti_val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("New split — Training:", len(drishti_train_dataset), "Validation:", len(drishti_val_dataset))


# ============================================================
# RESNET34 TRANSUNET FOR DRISHTI
# ============================================================

import torchvision.models as models


class ResNetTransUNet(nn.Module):

    def __init__(self, out_channels=3, pretrained=True):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=64)
        self.transformer_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]

        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drishti_resnet_model = ResNetTransUNet(out_channels=3, pretrained=True).to(device)

print("DRISHTI ResNet34 TransUNet created!")
print("Total parameters:", sum(p.numel() for p in drishti_resnet_model.parameters()))

drishti_resnet_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.20, 1.00, 1.00]).to(device)

encoder_params = list(drishti_resnet_model.stem.parameters()) + \
    list(drishti_resnet_model.layer1.parameters()) + \
    list(drishti_resnet_model.layer2.parameters()) + \
    list(drishti_resnet_model.layer3.parameters()) + \
    list(drishti_resnet_model.layer4.parameters())

new_params = list(drishti_resnet_model.bottleneck.parameters()) + \
    list(drishti_resnet_model.transformer.parameters()) + \
    list(drishti_resnet_model.transformer_refine.parameters()) + \
    list(drishti_resnet_model.decoder4.parameters()) + \
    list(drishti_resnet_model.decoder3.parameters()) + \
    list(drishti_resnet_model.decoder2.parameters()) + \
    list(drishti_resnet_model.decoder1.parameters()) + \
    list(drishti_resnet_model.final_refine.parameters()) + \
    list(drishti_resnet_model.output_layer.parameters())

drishti_resnet_optimizer = optim.AdamW([
    {"params": encoder_params, "lr": 2e-5},
    {"params": new_params, "lr": 1e-4}
], weight_decay=1e-4)

drishti_resnet_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drishti_resnet_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drishti_resnet_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drishti_resnet_path = "/content/best_drishti_resnet_transunet.pth"

print()
print("Starting DRISHTI ResNet34 TransUNet training...")
print()

for epoch in range(num_epochs):

    drishti_resnet_model.train()

    total_train_loss = 0.0

    for images, masks in drishti_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drishti_resnet_optimizer.zero_grad(set_to_none=True)

        outputs = drishti_resnet_model(images)
        loss = drishti_resnet_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drishti_resnet_model.parameters(), max_norm=1.0)

        drishti_resnet_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drishti_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drishti_resnet_model, drishti_val_loader, drishti_resnet_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drishti_resnet_dice:
        best_drishti_resnet_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drishti_resnet_model.state_dict(), best_drishti_resnet_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drishti_resnet_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Disc: {val_dice[0].item():.4f} | "
        f"Cup: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRISHTI RESNET34 TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_drishti_resnet_dice:.4f}")


# ============================================================
# FINAL TEST
# ============================================================

drishti_resnet_model.load_state_dict(torch.load(best_drishti_resnet_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drishti_resnet_model, drishti_test_loader, drishti_resnet_loss_fn, device, num_classes=3)

print()
print("DRISHTI RESNET34 TRANSUNET FINAL TEST RESULTS")
print(f"Disc Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Cup Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

Standalone GNN

In [ ]:
# ============================================================
# DRISHTI GRAPH U-NET (RESNET34)
# ============================================================

class ResNetGraphUNet(nn.Module):

    def __init__(self, out_channels=3, pretrained=True):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)
        self.graph_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape
        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.graph_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drishti_graph_model = ResNetGraphUNet(out_channels=3, pretrained=True).to(device)

print("DRISHTI Graph U-Net (ResNet34) created!")
print("Total parameters:", sum(p.numel() for p in drishti_graph_model.parameters()))

drishti_graph_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.20, 1.00, 1.00]).to(device)

graph_encoder_params = list(drishti_graph_model.stem.parameters()) + \
    list(drishti_graph_model.layer1.parameters()) + \
    list(drishti_graph_model.layer2.parameters()) + \
    list(drishti_graph_model.layer3.parameters()) + \
    list(drishti_graph_model.layer4.parameters())

graph_new_params = list(drishti_graph_model.bottleneck.parameters()) + \
    list(drishti_graph_model.graph_reasoning.parameters()) + \
    list(drishti_graph_model.graph_refine.parameters()) + \
    list(drishti_graph_model.decoder4.parameters()) + \
    list(drishti_graph_model.decoder3.parameters()) + \
    list(drishti_graph_model.decoder2.parameters()) + \
    list(drishti_graph_model.decoder1.parameters()) + \
    list(drishti_graph_model.final_refine.parameters()) + \
    list(drishti_graph_model.output_layer.parameters())

drishti_graph_optimizer = optim.AdamW([
    {"params": graph_encoder_params, "lr": 2e-5},
    {"params": graph_new_params, "lr": 1e-4}
], weight_decay=1e-4)

drishti_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drishti_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drishti_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drishti_graph_path = "/content/best_drishti_graph_unet.pth"

print()
print("Starting DRISHTI Graph U-Net training...")
print()

for epoch in range(num_epochs):

    drishti_graph_model.train()

    total_train_loss = 0.0

    for images, masks in drishti_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drishti_graph_optimizer.zero_grad(set_to_none=True)

        outputs = drishti_graph_model(images)
        loss = drishti_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drishti_graph_model.parameters(), max_norm=1.0)

        drishti_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drishti_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drishti_graph_model, drishti_val_loader, drishti_graph_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drishti_graph_dice:
        best_drishti_graph_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drishti_graph_model.state_dict(), best_drishti_graph_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drishti_graph_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Disc: {val_dice[0].item():.4f} | "
        f"Cup: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRISHTI GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_drishti_graph_dice:.4f}")


drishti_graph_model.load_state_dict(torch.load(best_drishti_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drishti_graph_model, drishti_test_loader, drishti_graph_loss_fn, device, num_classes=3)

print()
print("DRISHTI GRAPH U-NET FINAL TEST RESULTS")
print(f"Disc Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Cup Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# DRISHTI COMBINED GNN+TRANSFORMER (RESNET34)

In [ ]:


class ResNetCombinedUNet(nn.Module):

    def __init__(self, out_channels=3, pretrained=True):
        super().__init__()

        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(
            feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=64
        )
        self.bottleneck_edge_index = create_grid_edges(height=8, width=8)
        self.combined_refine = ResidualConvBlock(512, 512)

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)

        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)

        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)

        b = self.bottleneck(e4)

        batch_size, channels, height, width = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.combined_refine(b)

        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drishti_combined_model = ResNetCombinedUNet(out_channels=3, pretrained=True).to(device)

print("DRISHTI Combined GNN+Transformer (ResNet34) created!")
print("Total parameters:", sum(p.numel() for p in drishti_combined_model.parameters()))

drishti_combined_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.20, 1.00, 1.00]).to(device)

combined_encoder_params = list(drishti_combined_model.stem.parameters()) + \
    list(drishti_combined_model.layer1.parameters()) + \
    list(drishti_combined_model.layer2.parameters()) + \
    list(drishti_combined_model.layer3.parameters()) + \
    list(drishti_combined_model.layer4.parameters())

combined_new_params = list(drishti_combined_model.bottleneck.parameters()) + \
    list(drishti_combined_model.combined_reasoning.parameters()) + \
    list(drishti_combined_model.combined_refine.parameters()) + \
    list(drishti_combined_model.decoder4.parameters()) + \
    list(drishti_combined_model.decoder3.parameters()) + \
    list(drishti_combined_model.decoder2.parameters()) + \
    list(drishti_combined_model.decoder1.parameters()) + \
    list(drishti_combined_model.final_refine.parameters()) + \
    list(drishti_combined_model.output_layer.parameters())

drishti_combined_optimizer = optim.AdamW([
    {"params": combined_encoder_params, "lr": 2e-5},
    {"params": combined_new_params, "lr": 1e-4}
], weight_decay=1e-4)

drishti_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drishti_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drishti_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drishti_combined_path = "/content/best_drishti_combined_unet.pth"

print()
print("Starting DRISHTI Combined model training...")
print()

for epoch in range(num_epochs):

    drishti_combined_model.train()

    total_train_loss = 0.0

    for images, masks in drishti_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drishti_combined_optimizer.zero_grad(set_to_none=True)

        outputs = drishti_combined_model(images)
        loss = drishti_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drishti_combined_model.parameters(), max_norm=1.0)

        drishti_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drishti_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drishti_combined_model, drishti_val_loader, drishti_combined_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drishti_combined_dice:
        best_drishti_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drishti_combined_model.state_dict(), best_drishti_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drishti_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Disc: {val_dice[0].item():.4f} | "
        f"Cup: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRISHTI COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_drishti_combined_dice:.4f}")

Test Results


In [ ]:
drishti_combined_model.load_state_dict(torch.load(best_drishti_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(
    drishti_combined_model, drishti_test_loader, drishti_combined_loss_fn, device, num_classes=3
)

print("DRISHTI COMBINED MODEL FINAL TEST RESULTS")
print(f"Disc Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Cup Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# VISUALIZATION FUNCTION WITH PER-IMAGE DICE SCORE

In [ ]:


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def compute_single_image_dice(prediction, target, num_classes=4):

    dice_scores = []

    for class_id in range(1, num_classes):

        pred_class = (prediction == class_id)
        target_class = (target == class_id)

        intersection = np.logical_and(pred_class, target_class).sum()
        denom = pred_class.sum() + target_class.sum()

        if denom == 0:
            continue

        dice = (2.0 * intersection) / denom
        dice_scores.append(dice)

    if len(dice_scores) == 0:
        return 0.0

    return np.mean(dice_scores)


def visualize_predictions(model, dataset, device, num_images=40, num_classes=4, class_colors=None, class_names=None, dataset_name="Dataset"):

    model.eval()

    if class_colors is None:
        class_colors = {1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1]}

    if class_names is None:
        class_names = {1: "Class 1", 2: "Class 2", 3: "Class 3"}

    num_images = min(num_images, len(dataset))

    cols = 5
    rows = (num_images * 3 + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    plot_idx = 0

    with torch.no_grad():

        for idx in range(num_images):

            image, true_mask = dataset[idx]

            input_image = image.unsqueeze(0).to(device)

            output = model(input_image)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            true_mask_np = true_mask.numpy()

            dice = compute_single_image_dice(prediction, true_mask_np, num_classes)

            display_image = image[0].cpu().numpy()
            display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min() + 1e-8)

            gt_overlay = np.stack([display_image] * 3, axis=-1)
            pred_overlay = np.stack([display_image] * 3, axis=-1)

            for class_id, color in class_colors.items():

                gt_mask = (true_mask_np == class_id)
                pred_mask = (prediction == class_id)

                for c in range(3):
                    gt_overlay[..., c] = np.where(gt_mask, color[c], gt_overlay[..., c])
                    pred_overlay[..., c] = np.where(pred_mask, color[c], pred_overlay[..., c])

            axes[plot_idx].imshow(display_image, cmap="gray")
            axes[plot_idx].set_title("Input", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(gt_overlay)
            axes[plot_idx].set_title("Ground Truth", fontsize=9)
            axes[plot_idx].axis("off")
            plot_idx += 1

            axes[plot_idx].imshow(pred_overlay)
            axes[plot_idx].set_title(f"Prediction (Dice: {dice:.3f})", fontsize=9, fontweight="bold")
            axes[plot_idx].axis("off")
            plot_idx += 1

    for remaining_idx in range(plot_idx, len(axes)):
        axes[remaining_idx].axis("off")

    legend_patches = [
        mpatches.Patch(color=color, label=class_names[class_id])
        for class_id, color in class_colors.items()
    ]

    fig.legend(handles=legend_patches, loc="upper center", ncol=len(legend_patches), fontsize=11, bbox_to_anchor=(0.5, 1.01))

    fig.suptitle(f"{dataset_name} - Predictions with Per-Image Dice Score", fontsize=13, y=1.03)

    plt.tight_layout()
    plt.show()


print("Visualization function ready!")

In [ ]:
visualize_predictions(
    drishti_combined_model,
    drishti_test_dataset,
    device,
    num_images=40,
    num_classes=3,
    class_colors={1: [1, 0, 0], 2: [0, 0, 1]},
    class_names={1: "Optic Disc", 2: "Optic Cup"},
    dataset_name="DRISHTI Combined GNN+Transformer"
)

SSIM Metric

In [ ]:




drishti_resnet_model = ResNetTransUNet(out_channels=3, pretrained=True).to(device)

print("DRISHTI ResNet34 TransUNet recreated!")



# SSIM - ALL THREE DRISHTI MODELS (CORRECTED)


drishti_resnet_model.load_state_dict(torch.load(best_drishti_resnet_path, map_location=device))
mean_ssim_drishti_transunet = evaluate_ssim(drishti_resnet_model, drishti_test_loader, device, num_classes=3)

print("DRISHTI TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_drishti_transunet:.4f}")
print()

drishti_graph_model.load_state_dict(torch.load(best_drishti_graph_path, map_location=device))
mean_ssim_drishti_graph = evaluate_ssim(drishti_graph_model, drishti_test_loader, device, num_classes=3)

print("DRISHTI GRAPH U-NET - SSIM")
print(f"Mean SSIM: {mean_ssim_drishti_graph:.4f}")
print()

drishti_combined_model.load_state_dict(torch.load(best_drishti_combined_path, map_location=device))
mean_ssim_drishti_combined = evaluate_ssim(drishti_combined_model, drishti_test_loader, device, num_classes=3)

print("DRISHTI COMBINED MODEL - SSIM")
print(f"Mean SSIM: {mean_ssim_drishti_combined:.4f}")

In [ ]:
# ============================================================
# BACKUP DRISHTI CHECKPOINTS TO GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)

drishti_checkpoints = [
    best_drishti_resnet_path,
    best_drishti_graph_path,
    best_drishti_combined_path,
]

for path in drishti_checkpoints:
    filename = os.path.basename(path)
    destination = os.path.join(backup_folder, filename)
    shutil.copy(path, destination)
    print(f"Backed up: {filename}")

print()
print("All DRISHTI checkpoints backed up to Google Drive!")

# DOWNLOAD CHEST X-RAY DATASET

In [ ]:
# ============================================================
# DOWNLOAD CHEST X-RAY DATASET (JSRT + PADCHEST)
# ============================================================

chest_path = kagglehub.dataset_download(
    "meyupy/chest-x-ray-segmentation-jsrt-and-padchest"
)

print("Chest X-ray dataset downloaded!")
print("Dataset path:", chest_path)

# EXPLORE CHEST X-RAY DATASET STRUCTURE

In [ ]:
for root, dirs, files in os.walk(chest_path):

    level = root.replace(chest_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

Inspect On file

In [ ]:
images_folder = os.path.join(chest_path, "images")
masks_folder = os.path.join(chest_path, "masks")

image_files = sorted(os.listdir(images_folder))

sample_filename = image_files[0]

sample_image_path = os.path.join(images_folder, sample_filename)
sample_mask_path = os.path.join(masks_folder, sample_filename)

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print("Sample filename:", sample_filename)
print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("Mask size:", sample_mask.size, "mode:", sample_mask.mode)

mask_array = np.array(sample_mask)
print("Mask array shape:", mask_array.shape)
print("Unique values in mask:", np.unique(mask_array))

In [ ]:
mask_array = np.array(sample_mask)

pixels = mask_array.reshape(-1, 3)
unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

print("Unique RGB colors and pixel counts:")
for color, count in zip(unique_colors, counts):
    print(f"Color {tuple(color)}: {count} pixels")

# CHEST X-RAY DATASET

In [ ]:


chest_image_files = sorted(os.listdir(images_folder))

random.shuffle(chest_image_files)

num_files = len(chest_image_files)
num_train = int(num_files * 0.70)
num_val = int(num_files * 0.15)

chest_train_files = chest_image_files[:num_train]
chest_val_files = chest_image_files[num_train:num_train + num_val]
chest_test_files = chest_image_files[num_train + num_val:]

print("Total chest X-ray files:", num_files)
print("Training:", len(chest_train_files))
print("Validation:", len(chest_val_files))
print("Testing:", len(chest_test_files))


class ChestXrayDataset(Dataset):

    def __init__(self, filenames, images_folder, masks_folder, image_size=256, augment=False):

        self.filenames = filenames
        self.images_folder = images_folder
        self.masks_folder = masks_folder
        self.image_size = image_size
        self.augment = augment


    def __len__(self):
        return len(self.filenames)


    def __getitem__(self, idx):

        filename = self.filenames[idx]

        image_path = os.path.join(self.images_folder, filename)
        mask_path = os.path.join(self.masks_folder, filename)

        image = Image.open(image_path).convert("L")
        image = np.array(image, dtype=np.float32)

        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask, dtype=np.float32)

        # Remap raw values [0, 85, 170, 255] to sequential classes [0, 1, 2, 3]

        remapped_mask = np.zeros_like(mask, dtype=np.int64)
        remapped_mask[mask == 85] = 1
        remapped_mask[mask == 170] = 2
        remapped_mask[mask == 255] = 3

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(remapped_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        mask = F.interpolate(
            mask.float().unsqueeze(0),
            size=(self.image_size, self.image_size),
            mode="nearest"
        ).squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        image = image.repeat(3, 1, 1)
        mask = mask.squeeze(0)

        return image, mask


chest_train_dataset = ChestXrayDataset(chest_train_files, images_folder, masks_folder, image_size=256, augment=True)
chest_val_dataset = ChestXrayDataset(chest_val_files, images_folder, masks_folder, image_size=256, augment=False)
chest_test_dataset = ChestXrayDataset(chest_test_files, images_folder, masks_folder, image_size=256, augment=False)

chest_train_loader = DataLoader(chest_train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
chest_val_loader = DataLoader(chest_val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
chest_test_loader = DataLoader(chest_test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Chest X-ray DataLoaders created!")
print("Training batches:", len(chest_train_loader))
print("Validation batches:", len(chest_val_loader))
print("Testing batches:", len(chest_test_loader))

# CHEST X-RAY TRANSUNET

In [ ]:


chest_model = TransUNet(in_channels=3, out_channels=4).to(device)

print("Chest X-ray TransUNet created!")
print("Total parameters:", sum(p.numel() for p in chest_model.parameters()))

chest_loss_fn = ACDCLoss(num_classes=4, class_weights=[0.20, 1.00, 1.00, 1.00]).to(device)

chest_optimizer = optim.AdamW(chest_model.parameters(), lr=1e-4, weight_decay=1e-4)

chest_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    chest_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_chest_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_chest_path = "/content/best_chest_transunet.pth"

print()
print("Starting Chest X-ray TransUNet training...")
print()

for epoch in range(num_epochs):

    chest_model.train()

    total_train_loss = 0.0

    for images, masks in chest_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        chest_optimizer.zero_grad(set_to_none=True)

        outputs = chest_model(images)
        loss = chest_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(chest_model.parameters(), max_norm=1.0)

        chest_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(chest_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(chest_model, chest_val_loader, chest_loss_fn, device, num_classes=4)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_chest_dice:
        best_chest_dice = current_dice
        epochs_without_improvement = 0
        torch.save(chest_model.state_dict(), best_chest_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    chest_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("CHEST X-RAY TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_chest_dice:.4f}")


chest_model.load_state_dict(torch.load(best_chest_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(chest_model, chest_test_loader, chest_loss_fn, device, num_classes=4)

print()
print("CHEST X-RAY TRANSUNET FINAL TEST RESULTS")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# CHEST X-RAY GRAPH U-NET

In [ ]:


chest_graph_model = MSCMRGraphUNet(in_channels=3, out_channels=4).to(device)

print("Chest X-ray Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in chest_graph_model.parameters()))

chest_graph_loss_fn = ACDCLoss(num_classes=4, class_weights=[0.20, 1.00, 1.00, 1.00]).to(device)

chest_graph_optimizer = optim.AdamW(chest_graph_model.parameters(), lr=1e-4, weight_decay=1e-4)

chest_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    chest_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_chest_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_chest_graph_path = "/content/best_chest_graph_unet.pth"

print()
print("Starting Chest X-ray Graph U-Net training...")
print()

for epoch in range(num_epochs):

    chest_graph_model.train()

    total_train_loss = 0.0

    for images, masks in chest_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        chest_graph_optimizer.zero_grad(set_to_none=True)

        outputs = chest_graph_model(images)
        loss = chest_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(chest_graph_model.parameters(), max_norm=1.0)

        chest_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(chest_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(chest_graph_model, chest_val_loader, chest_graph_loss_fn, device, num_classes=4)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_chest_graph_dice:
        best_chest_graph_dice = current_dice
        epochs_without_improvement = 0
        torch.save(chest_graph_model.state_dict(), best_chest_graph_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    chest_graph_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("CHEST X-RAY GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_chest_graph_dice:.4f}")


chest_graph_model.load_state_dict(torch.load(best_chest_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(chest_graph_model, chest_test_loader, chest_graph_loss_fn, device, num_classes=4)

print()
print("CHEST X-RAY GRAPH U-NET FINAL TEST RESULTS")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

# CHEST X-RAY COMBINED GNN+TRANSFORMER

In [ ]:


chest_combined_model = MSCMRCombinedUNet(in_channels=3, out_channels=4).to(device)

print("Chest X-ray Combined GNN+Transformer created!")
print("Total parameters:", sum(p.numel() for p in chest_combined_model.parameters()))

chest_combined_loss_fn = ACDCLoss(num_classes=4, class_weights=[0.20, 1.00, 1.00, 1.00]).to(device)

chest_combined_optimizer = optim.AdamW(chest_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

chest_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    chest_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_chest_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_chest_combined_path = "/content/best_chest_combined_unet.pth"

print()
print("Starting Chest X-ray Combined model training...")
print()

for epoch in range(num_epochs):

    chest_combined_model.train()

    total_train_loss = 0.0

    for images, masks in chest_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        chest_combined_optimizer.zero_grad(set_to_none=True)

        outputs = chest_combined_model(images)
        loss = chest_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(chest_combined_model.parameters(), max_norm=1.0)

        chest_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(chest_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(chest_combined_model, chest_val_loader, chest_combined_loss_fn, device, num_classes=4)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_chest_combined_dice:
        best_chest_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(chest_combined_model.state_dict(), best_chest_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    chest_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Class1: {val_dice[0].item():.4f} | "
        f"Class2: {val_dice[1].item():.4f} | "
        f"Class3: {val_dice[2].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("CHEST X-RAY COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_chest_combined_dice:.4f}")


chest_combined_model.load_state_dict(torch.load(best_chest_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(chest_combined_model, chest_test_loader, chest_combined_loss_fn, device, num_classes=4)

print()
print("CHEST X-RAY COMBINED MODEL FINAL TEST RESULTS")
print(f"Class1 Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Class2 Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Class3 Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)

chest_checkpoints = [
    best_chest_path,
    best_chest_graph_path,
    best_chest_combined_path,
]

for path in chest_checkpoints:
    filename = os.path.basename(path)
    destination = os.path.join(backup_folder, filename)
    shutil.copy(path, destination)
    print(f"Backed up: {filename}")

print()
print("All chest X-ray checkpoints backed up to Google Drive!")

# Visualizations

In [ ]:
visualize_predictions(
    chest_combined_model,
    chest_test_dataset,
    device,
    num_images=40,
    num_classes=4,
    class_colors={1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1]},
    class_names={1: "Right Lung", 2: "Left Lung", 3: "Heart"},
    dataset_name="Chest X-ray Combined GNN+Transformer"
)

# SSIM - ALL THREE CHEST X-RAY MODELS

In [ ]:


chest_model.load_state_dict(torch.load(best_chest_path, map_location=device))
mean_ssim_chest_transunet = evaluate_ssim(chest_model, chest_test_loader, device, num_classes=4)

print("CHEST X-RAY TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_chest_transunet:.4f}")
print()

chest_graph_model.load_state_dict(torch.load(best_chest_graph_path, map_location=device))
mean_ssim_chest_graph = evaluate_ssim(chest_graph_model, chest_test_loader, device, num_classes=4)

print("CHEST X-RAY GRAPH U-NET - SSIM")
print(f"Mean SSIM: {mean_ssim_chest_graph:.4f}")
print()

chest_combined_model.load_state_dict(torch.load(best_chest_combined_path, map_location=device))
mean_ssim_chest_combined = evaluate_ssim(chest_combined_model, chest_test_loader, device, num_classes=4)

print("CHEST X-RAY COMBINED MODEL - SSIM")
print(f"Mean SSIM: {mean_ssim_chest_combined:.4f}")

# DOWNLOAD RAVIR DATASET (ARTERIES, VEINS, VESSELS - IR IMAGING)

In [ ]:
# ============================================================
# UPLOAD AND EXTRACT AVRDB DATASET
# ============================================================

from google.colab import files

uploaded = files.upload()

# This will prompt a file picker - select the zip file you downloaded

# EXTRACT AND EXPLORE AVRDB DATASET STRUCTURE

In [ ]:


import zipfile

avrdb_zip_path = "Data on Fundus Images for Vessels Segmentation, Detection of Hypertensive Retinopathy, Diabetic Retinopathy and Papilledema.zip"

with zipfile.ZipFile(avrdb_zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/avrdb_dataset")

avrdb_path = "/content/avrdb_dataset"

for root, dirs, files in os.walk(avrdb_path):

    level = root.replace(avrdb_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

Extarct Zip

In [ ]:


inner_zip_path = os.path.join(
    avrdb_path,
    "Data on Fundus Images for Vessels Segmentation, Detection of Hypertensive Retinopathy, Diabetic Retinopathy and Papilledema",
    "AV-20191104T162310Z-001.zip"
)

with zipfile.ZipFile(inner_zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/avrdb_extracted")

avrdb_inner_path = "/content/avrdb_extracted"

for root, dirs, files in os.walk(avrdb_inner_path):

    level = root.replace(avrdb_inner_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")
# ============================================================

inner_zip_path = os.path.join(
    avrdb_path,
    "Data on Fundus Images for Vessels Segmentation, Detection of Hypertensive Retinopathy, Diabetic Retinopathy and Papilledema",
    "AV-20191104T162310Z-001.zip"
)

with zipfile.ZipFile(inner_zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/avrdb_extracted")

avrdb_inner_path = "/content/avrdb_extracted"

for root, dirs, files in os.walk(avrdb_inner_path):

    level = root.replace(avrdb_inner_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")
# ============================================================

inner_zip_path = os.path.join(
    avrdb_path,
    "Data on Fundus Images for Vessels Segmentation, Detection of Hypertensive Retinopathy, Diabetic Retinopathy and Papilledema",
    "AV-20191104T162310Z-001.zip"
)

with zipfile.ZipFile(inner_zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/avrdb_extracted")

avrdb_inner_path = "/content/avrdb_extracted"

for root, dirs, files in os.walk(avrdb_inner_path):

    level = root.replace(avrdb_inner_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

# BUILD AVRDB FILE LIST WITH FUZZY TYPO-TOLERANT MATCHING

In [ ]:

avrdb_root = avrdb_inner_path

patient_folders = [
    f for f in os.listdir(avrdb_root)
    if os.path.isdir(os.path.join(avrdb_root, f))
]

print("Total patient folders found:", len(patient_folders))

avrdb_triplets = []
unmatched_files = []

for patient_id in patient_folders:

    patient_path = os.path.join(avrdb_root, patient_id)
    files_in_folder = os.listdir(patient_path)

    main_image_path = None
    artery_path = None
    vein_path = None
    vessel_path = None

    for filename in files_in_folder:

        lower_name = filename.lower()
        full_path = os.path.join(patient_path, filename)

        if "art" in lower_name:
            artery_path = full_path
        elif "vei" in lower_name:
            vein_path = full_path
        elif "vess" in lower_name:
            vessel_path = full_path
        elif filename.lower().endswith((".jpg", ".jpeg", ".png")) and "--" not in filename:
            main_image_path = full_path
        else:
            unmatched_files.append(filename)

    if main_image_path and artery_path and vein_path and vessel_path:
        avrdb_triplets.append((main_image_path, artery_path, vein_path, vessel_path))

print("Complete triplets found (image + artery + vein + vessel):", len(avrdb_triplets))
print()
print("Unmatched files (need manual check):", unmatched_files[:20])
if len(unmatched_files) > 20:
    print("... and", len(unmatched_files) - 20, "more")

# FIND THE ACTUAL NESTED FOLDER LEVEL

In [ ]:


top_level_folder = os.path.join(avrdb_root, patient_folders[0])

print("Top level folder name:", patient_folders[0])
print("Contents:", os.listdir(top_level_folder)[:10])
print("Total items inside:", len(os.listdir(top_level_folder)))

# BUILD AVRDB FILE LIST - CORRECTED PATH

In [ ]:


avrdb_av_root = os.path.join(avrdb_root, "AV")

patient_folders = [
    f for f in os.listdir(avrdb_av_root)
    if os.path.isdir(os.path.join(avrdb_av_root, f))
]

print("Total patient folders found:", len(patient_folders))

avrdb_triplets = []
unmatched_files = []

for patient_id in patient_folders:

    patient_path = os.path.join(avrdb_av_root, patient_id)
    files_in_folder = os.listdir(patient_path)

    main_image_path = None
    artery_path = None
    vein_path = None
    vessel_path = None

    for filename in files_in_folder:

        lower_name = filename.lower()
        full_path = os.path.join(patient_path, filename)

        if "art" in lower_name:
            artery_path = full_path
        elif "vei" in lower_name:
            vein_path = full_path
        elif "vess" in lower_name:
            vessel_path = full_path
        elif filename.lower().endswith((".jpg", ".jpeg", ".png")) and "--" not in filename:
            main_image_path = full_path
        else:
            unmatched_files.append(filename)

    if main_image_path and artery_path and vein_path and vessel_path:
        avrdb_triplets.append((main_image_path, artery_path, vein_path, vessel_path))

print("Complete triplets found (image + artery + vein + vessel):", len(avrdb_triplets))
print()
print("First triplet:", avrdb_triplets[0] if avrdb_triplets else "NONE FOUND")
print()
print("Unmatched files (need manual check):", unmatched_files[:20])
if len(unmatched_files) > 20:
    print("... and", len(unmatched_files) - 20, "more")

In [ ]:
sample_image_path, sample_artery_path, sample_vein_path, sample_vessel_path = avrdb_triplets[0]

sample_image = Image.open(sample_image_path)
sample_artery = Image.open(sample_artery_path)
sample_vein = Image.open(sample_vein_path)
sample_vessel = Image.open(sample_vessel_path)

print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("Artery mask size:", sample_artery.size, "mode:", sample_artery.mode)
print("Vein mask size:", sample_vein.size, "mode:", sample_vein.mode)
print("Vessel mask size:", sample_vessel.size, "mode:", sample_vessel.mode)

artery_array = np.array(sample_artery.convert("L"))
vein_array = np.array(sample_vein.convert("L"))

print()
print("Artery mask unique values (first 20):", np.unique(artery_array)[:20])
print("Vein mask unique values (first 20):", np.unique(vein_array)[:20])

# COMPARE ANNOTATION IMAGES AGAINST ORIGINAL TO FIND OVERLAY COLOR

In [ ]:
# ============================================================
# COMPARE ANNOTATION IMAGES AGAINST ORIGINAL TO FIND OVERLAY COLOR
# ============================================================

original_array = np.array(sample_image.convert("RGB"), dtype=np.int32)
artery_array_rgb = np.array(sample_artery.convert("RGB"), dtype=np.int32)
vein_array_rgb = np.array(sample_vein.convert("RGB"), dtype=np.int32)

difference_artery = np.abs(artery_array_rgb - original_array).sum(axis=2)
difference_vein = np.abs(vein_array_rgb - original_array).sum(axis=2)

print("Artery difference - max:", difference_artery.max(), "mean:", difference_artery.mean())
print("Pixels with artery difference > 30:", (difference_artery > 30).sum())
print()
print("Vein difference - max:", difference_vein.max(), "mean:", difference_vein.mean())
print("Pixels with vein difference > 30:", (difference_vein > 30).sum())

# Check what color the marked artery pixels actually are
marked_artery_pixels = artery_array_rgb[difference_artery > 30]
if len(marked_artery_pixels) > 0:
    print()
    print("Sample marked artery pixel colors (first 5):", marked_artery_pixels[:5])
    print("Average marked artery color (R,G,B):", marked_artery_pixels.mean(axis=0))

marked_vein_pixels = vein_array_rgb[difference_vein > 30]
if len(marked_vein_pixels) > 0:
    print()
    print("Sample marked vein pixel colors (first 5):", marked_vein_pixels[:5])
    print("Average marked vein color (R,G,B):", marked_vein_pixels.mean(axis=0))

In [ ]:
# ============================================================
# FIND THE ACTUAL DRAWN LINE COLOR (NON-WHITE PIXELS)
# ============================================================

artery_array_rgb = np.array(sample_artery.convert("RGB"), dtype=np.int32)
vein_array_rgb = np.array(sample_vein.convert("RGB"), dtype=np.int32)
vessel_array_rgb = np.array(sample_vessel.convert("RGB"), dtype=np.int32)

# A pixel is "background" if it's close to pure white (all channels near 255)

artery_is_drawn = (artery_array_rgb.sum(axis=2) < 700)
vein_is_drawn = (vein_array_rgb.sum(axis=2) < 700)
vessel_is_drawn = (vessel_array_rgb.sum(axis=2) < 700)

print("Artery drawn pixels (non-white):", artery_is_drawn.sum(), "out of", artery_is_drawn.size)
print("Vein drawn pixels (non-white):", vein_is_drawn.sum(), "out of", vein_is_drawn.size)
print("Vessel drawn pixels (non-white):", vessel_is_drawn.sum(), "out of", vessel_is_drawn.size)

if artery_is_drawn.sum() > 0:
    print()
    print("Sample artery drawn pixel colors:", artery_array_rgb[artery_is_drawn][:5])
    print("Average artery drawn color (R,G,B):", artery_array_rgb[artery_is_drawn].mean(axis=0))

if vein_is_drawn.sum() > 0:
    print()
    print("Sample vein drawn pixel colors:", vein_array_rgb[vein_is_drawn][:5])
    print("Average vein drawn color (R,G,B):", vein_array_rgb[vein_is_drawn].mean(axis=0))

# AVRDB DATASET - COLOR-BASED ARTERY/VEIN EXTRACTION

In [ ]:

random.shuffle(avrdb_triplets)

num_triplets = len(avrdb_triplets)
num_train = int(num_triplets * 0.70)
num_val = int(num_triplets * 0.15)

avrdb_train_triplets = avrdb_triplets[:num_train]
avrdb_val_triplets = avrdb_triplets[num_train:num_train + num_val]
avrdb_test_triplets = avrdb_triplets[num_train + num_val:]

print("Training:", len(avrdb_train_triplets))
print("Validation:", len(avrdb_val_triplets))
print("Testing:", len(avrdb_test_triplets))


class AvrdbDataset(Dataset):

    def __init__(self, triplets, image_size=256, augment=False):
        self.triplets = triplets
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):

        image_path, artery_path, vein_path, vessel_path = self.triplets[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        artery_img = np.array(Image.open(artery_path).convert("RGB"), dtype=np.int32)
        vein_img = np.array(Image.open(vein_path).convert("RGB"), dtype=np.int32)

        artery_is_drawn = (artery_img.sum(axis=2) < 700)
        vein_is_drawn = (vein_img.sum(axis=2) < 700)

        red_channel_artery = artery_img[:, :, 0]
        green_channel_artery = artery_img[:, :, 1]
        blue_channel_artery = artery_img[:, :, 2]
        artery_mask = artery_is_drawn & (red_channel_artery > green_channel_artery) & (red_channel_artery > blue_channel_artery)

        red_channel_vein = vein_img[:, :, 0]
        green_channel_vein = vein_img[:, :, 1]
        blue_channel_vein = vein_img[:, :, 2]
        vein_mask = vein_is_drawn & (blue_channel_vein > red_channel_vein) & (blue_channel_vein > green_channel_vein)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


avrdb_train_dataset = AvrdbDataset(avrdb_train_triplets, image_size=384, augment=True)
avrdb_val_dataset = AvrdbDataset(avrdb_val_triplets, image_size=384, augment=False)
avrdb_test_dataset = AvrdbDataset(avrdb_test_triplets, image_size=384, augment=False)

avrdb_train_loader = DataLoader(avrdb_train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
avrdb_val_loader = DataLoader(avrdb_val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
avrdb_test_loader = DataLoader(avrdb_test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("AVRDB DataLoaders created!")
print("Training batches:", len(avrdb_train_loader))
print("Validation batches:", len(avrdb_val_loader))
print("Testing batches:", len(avrdb_test_loader))

In [ ]:
class AvrdbDatasetFullRes(Dataset):

    def __init__(self, triplets, target_width=1504, target_height=992, augment=False):
        self.triplets = triplets
        self.target_width = target_width
        self.target_height = target_height
        self.augment = augment

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):

        image_path, artery_path, vein_path, vessel_path = self.triplets[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        artery_img = np.array(Image.open(artery_path).convert("RGB"), dtype=np.int32)
        vein_img = np.array(Image.open(vein_path).convert("RGB"), dtype=np.int32)

        artery_is_drawn = (artery_img.sum(axis=2) < 740)
        vein_is_drawn = (vein_img.sum(axis=2) < 740)

        red_channel_artery = artery_img[:, :, 0]
        green_channel_artery = artery_img[:, :, 1]
        blue_channel_artery = artery_img[:, :, 2]
        artery_mask = artery_is_drawn & ((red_channel_artery - green_channel_artery) > 10) & ((red_channel_artery - blue_channel_artery) > 10)

        red_channel_vein = vein_img[:, :, 0]
        green_channel_vein = vein_img[:, :, 1]
        blue_channel_vein = vein_img[:, :, 2]
        vein_mask = vein_is_drawn & ((blue_channel_vein - red_channel_vein) > 10) & ((blue_channel_vein - green_channel_vein) > 10)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.target_height, self.target_width), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.target_height, self.target_width), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


avrdb_train_dataset = AvrdbDatasetFullRes(avrdb_train_triplets, target_width=1504, target_height=992, augment=True)
avrdb_val_dataset = AvrdbDatasetFullRes(avrdb_val_triplets, target_width=1504, target_height=992, augment=False)
avrdb_test_dataset = AvrdbDatasetFullRes(avrdb_test_triplets, target_width=1504, target_height=992, augment=False)

avrdb_train_loader = DataLoader(avrdb_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
avrdb_val_loader = DataLoader(avrdb_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
avrdb_test_loader = DataLoader(avrdb_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("AVRDB DataLoaders rebuilt at FULL resolution (1504x992)!")
print("Training batches:", len(avrdb_train_loader))
print("Validation batches:", len(avrdb_val_loader))
print("Testing batches:", len(avrdb_test_loader))

In [ ]:
avrdb_artery_train_dataset = AvrdbArteryOnlyDataset(avrdb_train_triplets, target_width=512, target_height=512, augment=True)
avrdb_artery_val_dataset = AvrdbArteryOnlyDataset(avrdb_val_triplets, target_width=512, target_height=512, augment=False)
avrdb_artery_test_dataset = AvrdbArteryOnlyDataset(avrdb_test_triplets, target_width=512, target_height=512, augment=False)

avrdb_artery_train_loader = DataLoader(avrdb_artery_train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
avrdb_artery_val_loader = DataLoader(avrdb_artery_val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
avrdb_artery_test_loader = DataLoader(avrdb_artery_test_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("AVRDB artery-only DataLoaders rebuilt at 512x512!")


avrdb_artery_model = ResNetTransUNet512(out_channels=2, pretrained=True).to(device)

print("AVRDB artery-only model (ResNet34, 512x512) created!")
print("Total parameters:", sum(p.numel() for p in avrdb_artery_model.parameters()))

avrdb_artery_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

avrdb_artery_encoder_params = list(avrdb_artery_model.stem.parameters()) + \
    list(avrdb_artery_model.layer1.parameters()) + \
    list(avrdb_artery_model.layer2.parameters()) + \
    list(avrdb_artery_model.layer3.parameters()) + \
    list(avrdb_artery_model.layer4.parameters())

avrdb_artery_new_params = list(avrdb_artery_model.bottleneck.parameters()) + \
    list(avrdb_artery_model.transformer.parameters()) + \
    list(avrdb_artery_model.transformer_refine.parameters()) + \
    list(avrdb_artery_model.decoder4.parameters()) + \
    list(avrdb_artery_model.decoder3.parameters()) + \
    list(avrdb_artery_model.decoder2.parameters()) + \
    list(avrdb_artery_model.decoder1.parameters()) + \
    list(avrdb_artery_model.final_refine.parameters()) + \
    list(avrdb_artery_model.output_layer.parameters())

avrdb_artery_optimizer = optim.AdamW([
    {"params": avrdb_artery_encoder_params, "lr": 2e-5},
    {"params": avrdb_artery_new_params, "lr": 1e-4}
], weight_decay=1e-4)

avrdb_artery_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    avrdb_artery_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_avrdb_artery_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_avrdb_artery_path = "/content/best_avrdb_artery_only.pth"

print()
print("Starting AVRDB artery-only training...")
print()

for epoch in range(num_epochs):

    avrdb_artery_model.train()

    total_train_loss = 0.0

    for images, masks in avrdb_artery_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        avrdb_artery_optimizer.zero_grad(set_to_none=True)

        outputs = avrdb_artery_model(images)
        loss = avrdb_artery_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(avrdb_artery_model.parameters(), max_norm=1.0)

        avrdb_artery_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(avrdb_artery_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(avrdb_artery_model, avrdb_artery_val_loader, avrdb_artery_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_avrdb_artery_dice:
        best_avrdb_artery_dice = current_dice
        epochs_without_improvement = 0
        torch.save(avrdb_artery_model.state_dict(), best_avrdb_artery_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    avrdb_artery_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Artery Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("AVRDB ARTERY-ONLY TRAINING COMPLETE")
print("Best Validation Artery Dice:", f"{best_avrdb_artery_dice:.4f}")


avrdb_artery_model.load_state_dict(torch.load(best_avrdb_artery_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(avrdb_artery_model, avrdb_artery_test_loader, avrdb_artery_loss_fn, device, num_classes=2)

print()
print("AVRDB ARTERY-ONLY FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

In [ ]:
# ============================================================
# THOROUGH MEMORY CLEANUP - DELETE OLD MODELS
# ============================================================

import gc

variable_names_to_clear = [
    "avrdb_model", "avrdb_graph_model", "avrdb_combined_model",
    "drive_model", "drive_graph_model", "drive_combined_model",
    "ravir_model", "ravir_resnet_model",
]

for name in variable_names_to_clear:
    if name in globals():
        del globals()[name]
        print(f"Deleted: {name}")

gc.collect()
torch.cuda.empty_cache()

print()
print("Memory after cleanup:")
print("Allocated:", torch.cuda.memory_allocated() / (1024**3), "GB")
print("Reserved:", torch.cuda.memory_reserved() / (1024**3), "GB")

In [ ]:
avrdb_artery_model = TransUNet512(in_channels=3, out_channels=2).to(device)

print("AVRDB artery-only model (from-scratch TransUNet, 512x512) created!")
print("Total parameters:", sum(p.numel() for p in avrdb_artery_model.parameters()))

avrdb_artery_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

avrdb_artery_optimizer = optim.AdamW(avrdb_artery_model.parameters(), lr=1e-4, weight_decay=1e-4)

avrdb_artery_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    avrdb_artery_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_avrdb_artery_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_avrdb_artery_path = "/content/best_avrdb_artery_only.pth"

print()
print("Starting AVRDB artery-only training...")
print()

for epoch in range(num_epochs):

    avrdb_artery_model.train()

    total_train_loss = 0.0

    for images, masks in avrdb_artery_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        avrdb_artery_optimizer.zero_grad(set_to_none=True)

        outputs = avrdb_artery_model(images)
        loss = avrdb_artery_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(avrdb_artery_model.parameters(), max_norm=1.0)

        avrdb_artery_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(avrdb_artery_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(avrdb_artery_model, avrdb_artery_val_loader, avrdb_artery_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_avrdb_artery_dice:
        best_avrdb_artery_dice = current_dice
        epochs_without_improvement = 0
        torch.save(avrdb_artery_model.state_dict(), best_avrdb_artery_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    avrdb_artery_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Artery Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("AVRDB ARTERY-ONLY TRAINING COMPLETE")
print("Best Validation Artery Dice:", f"{best_avrdb_artery_dice:.4f}")


avrdb_artery_model.load_state_dict(torch.load(best_avrdb_artery_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(avrdb_artery_model, avrdb_artery_test_loader, avrdb_artery_loss_fn, device, num_classes=2)

print()
print("AVRDB ARTERY-ONLY FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

# DOWNLOAD DRIVE DATASET (BINARY VESSEL SEGMENTATION)

In [ ]:
# ============================================================
# DOWNLOAD DRIVE DATASET (BINARY VESSEL SEGMENTATION)
# ============================================================

drive_dataset_path = kagglehub.dataset_download(
    "andrewmvd/drive-digital-retinal-images-for-vessel-extraction"
)

print("DRIVE dataset downloaded!")
print("Dataset path:", drive_dataset_path)

In [ ]:
for root, dirs, files in os.walk(drive_dataset_path):

    level = root.replace(drive_dataset_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
drive_images_folder = os.path.join(drive_dataset_path, "DRIVE", "training", "images")
drive_masks_folder = os.path.join(drive_dataset_path, "DRIVE", "training", "1st_manual")

image_files = sorted(os.listdir(drive_images_folder))
mask_files = sorted(os.listdir(drive_masks_folder))

print("Number of training images:", len(image_files))
print("Number of training masks:", len(mask_files))

sample_image_path = os.path.join(drive_images_folder, image_files[0])
sample_mask_path = os.path.join(drive_masks_folder, mask_files[0])

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print()
print("Sample image file:", image_files[0])
print("Sample mask file:", mask_files[0])
print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("Mask size:", sample_mask.size, "mode:", sample_mask.mode)

mask_array = np.array(sample_mask.convert("L"))
print("Mask unique values:", np.unique(mask_array))


# DRIVE DATASET - BINARY VESSEL SEGMENTATION

In [ ]:


image_files = sorted(os.listdir(drive_images_folder))
mask_files = sorted(os.listdir(drive_masks_folder))

drive_pairs = list(zip(
    [os.path.join(drive_images_folder, f) for f in image_files],
    [os.path.join(drive_masks_folder, f) for f in mask_files]
))

random.shuffle(drive_pairs)

drive_train_pairs = drive_pairs[:14]
drive_val_pairs = drive_pairs[14:17]
drive_test_pairs = drive_pairs[17:20]

print("Training:", len(drive_train_pairs))
print("Validation:", len(drive_val_pairs))
print("Testing:", len(drive_test_pairs))


class DriveDataset(Dataset):

    def __init__(self, pairs, image_size=512, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask, dtype=np.float32)
        mask = (mask > 127).astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


drive_train_dataset = DriveDataset(drive_train_pairs, image_size=512, augment=True)
drive_val_dataset = DriveDataset(drive_val_pairs, image_size=512, augment=False)
drive_test_dataset = DriveDataset(drive_test_pairs, image_size=512, augment=False)

drive_train_loader = DataLoader(drive_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
drive_val_loader = DataLoader(drive_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
drive_test_loader = DataLoader(drive_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("DRIVE DataLoaders created!")
print("Training batches:", len(drive_train_loader))
print("Validation batches:", len(drive_val_loader))
print("Testing batches:", len(drive_test_loader))


# TRANSUNET (FROM-SCRATCH) - FIXED FOR 512x512 INPUT (1024 TOKENS)

In [ ]:


class TransUNet512(nn.Module):

    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drive_model = TransUNet512(in_channels=3, out_channels=2).to(device)

print("DRIVE TransUNet (from-scratch) created!")
print("Total parameters:", sum(p.numel() for p in drive_model.parameters()))

drive_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

drive_optimizer = optim.AdamW(drive_model.parameters(), lr=1e-4, weight_decay=1e-4)

drive_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drive_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drive_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drive_path = "/content/best_drive_transunet_scratch.pth"

print()
print("Starting DRIVE TransUNet (from-scratch) training...")
print()

for epoch in range(num_epochs):

    drive_model.train()

    total_train_loss = 0.0

    for images, masks in drive_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drive_optimizer.zero_grad(set_to_none=True)

        outputs = drive_model(images)
        loss = drive_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drive_model.parameters(), max_norm=1.0)

        drive_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drive_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drive_model, drive_val_loader, drive_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drive_dice:
        best_drive_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drive_model.state_dict(), best_drive_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drive_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Vessel Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRIVE TRANSUNET (FROM-SCRATCH) TRAINING COMPLETE")
print("Best Validation Vessel Dice:", f"{best_drive_dice:.4f}")


drive_model.load_state_dict(torch.load(best_drive_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drive_model, drive_test_loader, drive_loss_fn, device, num_classes=2)

print()
print("DRIVE TRANSUNET (FROM-SCRATCH) FINAL TEST RESULTS")
print(f"Vessel Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

# DRIVE - REALLOCATE SPLIT (17/2/1) FOR MORE TRAINING DATA

In [ ]:

random.shuffle(drive_pairs)

drive_train_pairs = drive_pairs[:17]
drive_val_pairs = drive_pairs[17:19]
drive_test_pairs = drive_pairs[19:20]

print("Training:", len(drive_train_pairs))
print("Validation:", len(drive_val_pairs))
print("Testing:", len(drive_test_pairs))

drive_train_dataset = DriveDataset(drive_train_pairs, image_size=512, augment=True)
drive_val_dataset = DriveDataset(drive_val_pairs, image_size=512, augment=False)
drive_test_dataset = DriveDataset(drive_test_pairs, image_size=512, augment=False)

drive_train_loader = DataLoader(drive_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
drive_val_loader = DataLoader(drive_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
drive_test_loader = DataLoader(drive_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("DRIVE DataLoaders rebuilt with 17/2/1 split!")

In [ ]:
drive_model = TransUNet512(in_channels=3, out_channels=2).to(device)

print("DRIVE TransUNet (from-scratch, new split) created!")

drive_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

drive_optimizer = optim.AdamW(drive_model.parameters(), lr=1e-4, weight_decay=1e-4)

drive_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drive_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drive_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drive_path = "/content/best_drive_transunet_v3.pth"

print()
print("Starting DRIVE TransUNet (17/2/1 split) training...")
print()

for epoch in range(num_epochs):

    drive_model.train()

    total_train_loss = 0.0

    for images, masks in drive_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drive_optimizer.zero_grad(set_to_none=True)

        outputs = drive_model(images)
        loss = drive_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drive_model.parameters(), max_norm=1.0)

        drive_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drive_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drive_model, drive_val_loader, drive_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drive_dice:
        best_drive_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drive_model.state_dict(), best_drive_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drive_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Vessel Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRIVE TRANSUNET V3 TRAINING COMPLETE")
print("Best Validation Vessel Dice:", f"{best_drive_dice:.4f}")


drive_model.load_state_dict(torch.load(best_drive_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drive_model, drive_test_loader, drive_loss_fn, device, num_classes=2)

print()
print("DRIVE TRANSUNET V3 FINAL TEST RESULTS")
print(f"Vessel Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

Trans U Net SSIM

In [ ]:
from skimage.metrics import structural_similarity as ssim


def evaluate_ssim(model, loader, device, num_classes=2):

    model.eval()
    ssim_scores = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx].astype(np.float32)
                true = masks_cpu[batch_idx].astype(np.float32)
                score = ssim(pred, true, data_range=num_classes - 1)
                ssim_scores.append(score)

    return np.mean(ssim_scores)


mean_ssim_drive_transunet = evaluate_ssim(drive_model, drive_test_loader, device, num_classes=2)

print("DRIVE TRANSUNET - SSIM")
print(f"Mean SSIM: {mean_ssim_drive_transunet:.4f}")

ROC - AUC

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt


def plot_roc_curve(model, loader, device, num_classes=2, class_names=None, title="ROC Curve"):

    model.eval()

    if class_names is None:
        class_names = [f"Class {i}" for i in range(1, num_classes)]

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()
                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    plt.figure(figsize=(7, 6))

    for i, class_id in enumerate(range(1, num_classes)):
        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])
        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        auc_value = roc_auc_score(true_labels, probabilities)
        plt.plot(fpr, tpr, linewidth=2, label=f"{class_names[i]} (AUC = {auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


mean_auc_value = roc_auc_score(
    np.concatenate([masks.numpy().ravel() == 1 for images, masks in drive_test_loader]),
    np.concatenate([torch.softmax(drive_model(images.to(device)), dim=1)[:, 1].detach().cpu().numpy().ravel() for images, masks in drive_test_loader])
)

print("DRIVE TRANSUNET - ROC-AUC")
print(f"Vessel AUC: {mean_auc_value:.4f} ({mean_auc_value*100:.2f}%)")

plot_roc_curve(
    drive_model,
    drive_test_loader,
    device,
    num_classes=2,
    class_names=["Vessel"],
    title="DRIVE TransUNet - ROC Curve"
)

# GRAPH U-NET (FROM-SCRATCH) - FIXED FOR 512x512 INPUT

In [ ]:


class GraphUNet512(nn.Module):

    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=32, width=32)
        self.graph_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.graph_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drive_graph_model = GraphUNet512(in_channels=3, out_channels=2).to(device)

print("DRIVE Graph U-Net (from-scratch) created!")
print("Total parameters:", sum(p.numel() for p in drive_graph_model.parameters()))

drive_graph_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

drive_graph_optimizer = optim.AdamW(drive_graph_model.parameters(), lr=1e-4, weight_decay=1e-4)

drive_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drive_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drive_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drive_graph_path = "/content/best_drive_graph_unet.pth"

print()
print("Starting DRIVE Graph U-Net training...")
print()

for epoch in range(num_epochs):

    drive_graph_model.train()

    total_train_loss = 0.0

    for images, masks in drive_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drive_graph_optimizer.zero_grad(set_to_none=True)

        outputs = drive_graph_model(images)
        loss = drive_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drive_graph_model.parameters(), max_norm=1.0)

        drive_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drive_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drive_graph_model, drive_val_loader, drive_graph_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drive_graph_dice:
        best_drive_graph_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drive_graph_model.state_dict(), best_drive_graph_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drive_graph_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Vessel Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRIVE GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Vessel Dice:", f"{best_drive_graph_dice:.4f}")


drive_graph_model.load_state_dict(torch.load(best_drive_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drive_graph_model, drive_test_loader, drive_graph_loss_fn, device, num_classes=2)

print()
print("DRIVE GRAPH U-NET FINAL TEST RESULTS")
print(f"Vessel Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

# COMBINED GNN+TRANSFORMER (FROM-SCRATCH) - FIXED FOR 512x512

In [ ]:


class CombinedUNet512(nn.Module):

    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=1024)
        self.bottleneck_edge_index = create_grid_edges(height=32, width=32)
        self.combined_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.combined_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


drive_combined_model = CombinedUNet512(in_channels=3, out_channels=2).to(device)

print("DRIVE Combined GNN+Transformer (from-scratch) created!")
print("Total parameters:", sum(p.numel() for p in drive_combined_model.parameters()))

drive_combined_loss_fn = ACDCLoss(num_classes=2, class_weights=[0.05, 1.00]).to(device)

drive_combined_optimizer = optim.AdamW(drive_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

drive_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    drive_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_drive_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_drive_combined_path = "/content/best_drive_combined_unet.pth"

print()
print("Starting DRIVE Combined model training...")
print()

for epoch in range(num_epochs):

    drive_combined_model.train()

    total_train_loss = 0.0

    for images, masks in drive_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        drive_combined_optimizer.zero_grad(set_to_none=True)

        outputs = drive_combined_model(images)
        loss = drive_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(drive_combined_model.parameters(), max_norm=1.0)

        drive_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(drive_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(drive_combined_model, drive_val_loader, drive_combined_loss_fn, device, num_classes=2)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_drive_combined_dice:
        best_drive_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(drive_combined_model.state_dict(), best_drive_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    drive_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Vessel Dice: {current_dice:.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("DRIVE COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Vessel Dice:", f"{best_drive_combined_dice:.4f}")


drive_combined_model.load_state_dict(torch.load(best_drive_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(drive_combined_model, drive_test_loader, drive_combined_loss_fn, device, num_classes=2)

print()
print("DRIVE COMBINED MODEL FINAL TEST RESULTS")
print(f"Vessel Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")

# IOU AND HD95 FUNCTION

In [ ]:


from scipy.ndimage import label, binary_erosion, distance_transform_edt


def keep_largest_component(binary_mask):
    labeled_array, num_features = label(binary_mask)
    if num_features == 0:
        return binary_mask
    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0
    largest_component_label = component_sizes.argmax()
    return labeled_array == largest_component_label


def evaluate_iou_with_postprocessing(model, loader, device, num_classes=2):

    model.eval()
    intersections = torch.zeros(num_classes - 1, device=device)
    unions = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions_np = predictions.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for batch_idx in range(predictions_np.shape[0]):
                for class_id in range(1, num_classes):
                    pred_class = (predictions_np[batch_idx] == class_id)
                    target_class = (masks_np[batch_idx] == class_id)
                    pred_class_cleaned = keep_largest_component(pred_class)
                    intersection = np.logical_and(pred_class_cleaned, target_class).sum()
                    union = np.logical_or(pred_class_cleaned, target_class).sum()
                    intersections[class_id - 1] += intersection
                    unions[class_id - 1] += union

    iou_scores = (intersections + 1e-6) / (unions + 1e-6)
    mean_iou = iou_scores.mean()
    return mean_iou, iou_scores


def hd95_single_class(pred_mask, true_mask):
    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)
    if not pred_mask.any() and not true_mask.any():
        return 0.0
    if not pred_mask.any() or not true_mask.any():
        return np.nan
    pred_boundary = pred_mask ^ binary_erosion(pred_mask)
    true_boundary = true_mask ^ binary_erosion(true_mask)
    distance_to_true = distance_transform_edt(~true_boundary)
    distance_to_pred = distance_transform_edt(~pred_boundary)
    pred_to_true = distance_to_true[pred_boundary]
    true_to_pred = distance_to_pred[true_boundary]
    all_distances = np.concatenate([pred_to_true, true_to_pred])
    return np.percentile(all_distances, 95)


def evaluate_hd95(model, loader, device, num_classes=2):

    model.eval()
    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]
                for class_id in range(1, num_classes):
                    pred_class = keep_largest_component(pred == class_id)
                    true_class = (true == class_id)
                    hd = hd95_single_class(pred_class, true_class)
                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }
    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])
    return overall_mean_hd95, mean_hd95_per_class


print("Metric functions ready!")


# ============================================================
# ALL METRICS - DRIVE GRAPH U-NET
# ============================================================

mean_iou_graph, iou_scores_graph = evaluate_iou_with_postprocessing(drive_graph_model, drive_test_loader, device, num_classes=2)
overall_hd95_graph, hd95_per_class_graph = evaluate_hd95(drive_graph_model, drive_test_loader, device, num_classes=2)
mean_ssim_graph = evaluate_ssim(drive_graph_model, drive_test_loader, device, num_classes=2)

print("DRIVE GRAPH U-NET METRICS")
print(f"IoU: {mean_iou_graph.item()*100:.2f}%")
print(f"HD95: {overall_hd95_graph:.2f}px")
print(f"SSIM: {mean_ssim_graph:.4f}")
print()


# ============================================================
# ALL METRICS - DRIVE COMBINED
# ============================================================

mean_iou_combined, iou_scores_combined = evaluate_iou_with_postprocessing(drive_combined_model, drive_test_loader, device, num_classes=2)
overall_hd95_combined, hd95_per_class_combined = evaluate_hd95(drive_combined_model, drive_test_loader, device, num_classes=2)
mean_ssim_combined = evaluate_ssim(drive_combined_model, drive_test_loader, device, num_classes=2)

print("DRIVE COMBINED METRICS")
print(f"IoU: {mean_iou_combined.item()*100:.2f}%")
print(f"HD95: {overall_hd95_combined:.2f}px")
print(f"SSIM: {mean_ssim_combined:.4f}")

# VISUALIZATION FUNCTION (IF NOT ALREADY DEFINED THIS SESSION)

In [ ]:


import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def compute_single_image_dice(prediction, target, num_classes=4):
    dice_scores = []
    for class_id in range(1, num_classes):
        pred_class = (prediction == class_id)
        target_class = (target == class_id)
        intersection = np.logical_and(pred_class, target_class).sum()
        denom = pred_class.sum() + target_class.sum()
        if denom == 0:
            continue
        dice = (2.0 * intersection) / denom
        dice_scores.append(dice)
    if len(dice_scores) == 0:
        return 0.0
    return np.mean(dice_scores)


def visualize_predictions(model, dataset, device, num_images=5, num_classes=4, class_colors=None, class_names=None, dataset_name="Dataset"):

    model.eval()

    if class_colors is None:
        class_colors = {1: [1, 0, 0]}
    if class_names is None:
        class_names = {1: "Class1"}

    num_images = min(num_images, len(dataset))

    fig, axes = plt.subplots(num_images, 3, figsize=(12, num_images * 4))
    if num_images == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():

        for idx in range(num_images):

            image, true_mask = dataset[idx]
            input_image = image.unsqueeze(0).to(device)
            output = model(input_image)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            true_mask_np = true_mask.numpy()
            dice = compute_single_image_dice(prediction, true_mask_np, num_classes)

            display_image = image.permute(1, 2, 0).cpu().numpy()
            display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min() + 1e-8)

            gt_overlay = display_image.copy()
            pred_overlay = display_image.copy()

            for class_id, color in class_colors.items():
                gt_mask = (true_mask_np == class_id)
                pred_mask = (prediction == class_id)
                for c in range(3):
                    gt_overlay[..., c] = np.where(gt_mask, color[c], gt_overlay[..., c])
                    pred_overlay[..., c] = np.where(pred_mask, color[c], pred_overlay[..., c])

            axes[idx, 0].imshow(display_image)
            axes[idx, 0].set_title("Input")
            axes[idx, 0].axis("off")

            axes[idx, 1].imshow(gt_overlay)
            axes[idx, 1].set_title("Ground Truth")
            axes[idx, 1].axis("off")

            axes[idx, 2].imshow(pred_overlay)
            axes[idx, 2].set_title(f"Prediction (Dice: {dice:.3f})", fontweight="bold")
            axes[idx, 2].axis("off")

    legend_patches = [
        mpatches.Patch(color=color, label=class_names[class_id])
        for class_id, color in class_colors.items()
    ]
    fig.legend(handles=legend_patches, loc="upper center", ncol=len(legend_patches), fontsize=11, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"{dataset_name} - Predictions", fontsize=13, y=1.05)

    plt.tight_layout()
    plt.show()


print("Visualization function ready!")


# ============================================================
# VISUALIZE DRIVE TRANSUNET PREDICTIONS
# ============================================================

drive_model.load_state_dict(torch.load("/content/best_drive_transunet_v3.pth", map_location=device))

visualize_predictions(
    drive_model,
    drive_test_dataset,
    device,
    num_images=1,
    num_classes=2,
    class_colors={1: [1, 0, 0]},
    class_names={1: "Vessel"},
    dataset_name="DRIVE TransUNet"
)

In [ ]:
drive_all_dataset = DriveDataset(drive_pairs, image_size=512, augment=False)

visualize_predictions(
    drive_model,
    drive_all_dataset,
    device,
    num_images=6,
    num_classes=2,
    class_colors={1: [1, 0, 0]},
    class_names={1: "Vessel"},
    dataset_name="DRIVE TransUNet"
)

In [ ]:
# ============================================================
# FULL SETUP + DRIVE DATA + MODELS + ALL 3 TRAININGS + SSIM
# ============================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import shutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

!pip install -q kagglehub torch-geometric
import kagglehub
from torch_geometric.nn import SAGEConv


# --- Download DRIVE ---

drive_dataset_path = kagglehub.dataset_download(
    "andrewmvd/drive-digital-retinal-images-for-vessel-extraction"
)

drive_images_folder = os.path.join(drive_dataset_path, "DRIVE", "training", "images")
drive_masks_folder = os.path.join(drive_dataset_path, "DRIVE", "training", "1st_manual")

image_files = sorted(os.listdir(drive_images_folder))
mask_files = sorted(os.listdir(drive_masks_folder))

drive_pairs = list(zip(
    [os.path.join(drive_images_folder, f) for f in image_files],
    [os.path.join(drive_masks_folder, f) for f in mask_files]
))

random.shuffle(drive_pairs)

drive_train_pairs = drive_pairs[:17]
drive_val_pairs = drive_pairs[17:19]
drive_test_pairs = drive_pairs[19:20]

print("DRIVE data ready - Train:", len(drive_train_pairs), "Val:", len(drive_val_pairs), "Test:", len(drive_test_pairs))


# --- Dataset class ---

class DriveDataset(Dataset):
    def __init__(self, pairs, image_size=512, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]
        image = np.array(Image.open(image_path).convert("RGB"), dtype=np.float32)
        mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
        mask = (mask > 127).astype(np.int64)

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                image = torch.clamp(image * random.uniform(0.85, 1.15), 0.0, 1.0)

        mask = mask.squeeze(0)
        return image, mask


drive_train_dataset = DriveDataset(drive_train_pairs, image_size=512, augment=True)
drive_val_dataset = DriveDataset(drive_val_pairs, image_size=512, augment=False)
drive_test_dataset = DriveDataset(drive_test_pairs, image_size=512, augment=False)

drive_train_loader = DataLoader(drive_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
drive_val_loader = DataLoader(drive_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
drive_test_loader = DataLoader(drive_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("DRIVE DataLoaders ready!")


# --- Model components ---

class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act = nn.GELU()
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=0.1, activation="gelu", norm_first=True, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.shape[1], :]
        x = self.transformer(x)
        return self.norm(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = ResidualConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class TransUNet512(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(512, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))
        b = self.bottleneck(F.max_pool2d(e4, 2))
        bs, c, h, w = b.shape
        tokens = self.transformer(b.flatten(2).transpose(1, 2))
        b = tokens.transpose(1, 2).reshape(bs, c, h, w)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


def create_grid_edges(height, width):
    edges = []
    for row in range(height):
        for col in range(width):
            node = row * width + col
            if col < width - 1:
                right = node + 1
                edges.append([node, right])
                edges.append([right, node])
            if row < height - 1:
                down = node + width
                edges.append([node, down])
                edges.append([down, node])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index


class GraphBottleneckBlock(nn.Module):
    def __init__(self, feature_dim=512, hidden_dim=512, num_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        in_dim = feature_dim
        for _ in range(num_layers):
            self.layers.append(SAGEConv(in_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
            in_dim = hidden_dim
        self.act = nn.GELU()

    def forward(self, x, edge_index):
        batch_size = x.shape[0]
        outputs = []
        for b in range(batch_size):
            h = x[b]
            for layer, norm in zip(self.layers, self.norms):
                residual = h
                h = layer(h, edge_index)
                h = norm(h)
                h = self.act(h)
                if h.shape == residual.shape:
                    h = h + residual
            outputs.append(h)
        return torch.stack(outputs, dim=0)


class GraphUNet512(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=32, width=32)
        self.graph_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(512, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))
        b = self.bottleneck(F.max_pool2d(e4, 2))
        bs, c, h, w = b.shape
        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(bs, c, h, w)
        b = self.graph_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


class CombinedBottleneckBlock(nn.Module):
    def __init__(self, feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=1024):
        super().__init__()
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=feature_dim, hidden_dim=feature_dim, num_layers=num_graph_layers)
        self.transformer_reasoning = TransformerBlock(embed_dim=feature_dim, num_heads=num_heads, num_layers=num_transformer_layers, num_tokens=num_tokens)

    def forward(self, x, edge_index):
        x = self.graph_reasoning(x, edge_index)
        x = self.transformer_reasoning(x)
        return x


class CombinedUNet512(nn.Module):
    def __init__(self, in_channels=3, out_channels=2):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=1024)
        self.bottleneck_edge_index = create_grid_edges(height=32, width=32)
        self.combined_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(512, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))
        b = self.bottleneck(F.max_pool2d(e4, 2))
        bs, c, h, w = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(bs, c, h, w)
        b = self.combined_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


class MulticlassDiceLoss(nn.Module):
    def __init__(self, num_classes=2, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        total, n = 0.0, 0
        for c in range(1, self.num_classes):
            pred_c = probs[:, c]
            target_c = (targets == c).float()
            inter = (pred_c * target_c).sum()
            denom = pred_c.sum() + target_c.sum()
            dice = (2.0 * inter + self.smooth) / (denom + self.smooth)
            total += (1.0 - dice)
            n += 1
        return total / n


class ACDCLoss(nn.Module):
    def __init__(self, num_classes=2, gamma=2.0, class_weights=None):
        super().__init__()
        if class_weights is None:
            class_weights = [0.20] + [1.00] * (num_classes - 1)
        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.gamma = gamma
        self.dice_loss = MulticlassDiceLoss(num_classes=num_classes)

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = (((1 - pt) ** self.gamma) * ce).mean()
        dice = self.dice_loss(logits, targets)
        return 0.40 * focal + 1.00 * dice


def evaluate(model, loader, loss_fn, device, num_classes=2):
    model.eval()
    total_loss = 0.0
    intersections = torch.zeros(num_classes - 1, device=device)
    pred_sums = torch.zeros(num_classes - 1, device=device)
    target_sums = torch.zeros(num_classes - 1, device=device)
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item() * images.size(0)
            predictions = torch.argmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                pred_class = (predictions == class_id).float()
                target_class = (masks == class_id).float()
                intersections[class_id - 1] += (pred_class * target_class).sum()
                pred_sums[class_id - 1] += pred_class.sum()
                target_sums[class_id - 1] += target_class.sum()
    dice_scores = (2.0 * intersections + 1e-5) / (pred_sums + target_sums + 1e-5)
    mean_dice = dice_scores.mean()
    average_loss = total_loss / len(loader.dataset)
    return average_loss, mean_dice, dice_scores


print("All model components ready!")

In [ ]:
# ============================================================
# DOWNLOAD HRF ARTERY/VEIN GROUND TRUTH FROM GITHUB
# ============================================================

!git clone https://github.com/rubenhx/av-segmentation.git /content/hrf_av_repo

hrf_repo_path = "/content/hrf_av_repo"

for root, dirs, files in os.walk(hrf_repo_path):

    level = root.replace(hrf_repo_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

# DOWNLOAD OFFICIAL HRF FUNDUS IMAGES

In [ ]:


!wget -O /content/hrf_all.zip "https://www5.cs.fau.de/fileadmin/research/datasets/fundus-images/all.zip"

import os
print("File size:", os.path.getsize("/content/hrf_all.zip") / (1024*1024), "MB")

# EXTRACT AND EXPLORE HRF IMAGES

In [ ]:

import zipfile

with zipfile.ZipFile("/content/hrf_all.zip", "r") as zip_ref:
    zip_ref.extractall("/content/hrf_images")

hrf_images_path = "/content/hrf_images"

for root, dirs, files in os.walk(hrf_images_path):

    level = root.replace(hrf_images_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

# BUILD HRF IMAGE + AV MASK PAIRS

In [ ]:


hrf_images_folder = os.path.join(hrf_images_path, "images")
hrf_av_folder = os.path.join(hrf_repo_path, "HRF_AV_GT")

av_mask_files = os.listdir(hrf_av_folder)
image_files = os.listdir(hrf_images_folder)

hrf_pairs = []

for mask_filename in av_mask_files:

    base_id = mask_filename.replace("_AVmanual.png", "")

    matching_image = None
    for image_filename in image_files:
        if image_filename.lower().startswith(base_id.lower() + "."):
            matching_image = image_filename
            break

    if matching_image:
        image_path = os.path.join(hrf_images_folder, matching_image)
        mask_path = os.path.join(hrf_av_folder, mask_filename)
        hrf_pairs.append((image_path, mask_path))

print("Total matched pairs:", len(hrf_pairs))
print("First pair:", hrf_pairs[0])

sample_image_path, sample_mask_path = hrf_pairs[0]

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print()
print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("Mask size:", sample_mask.size, "mode:", sample_mask.mode)

mask_array = np.array(sample_mask.convert("RGB"))
pixels = mask_array.reshape(-1, 3)
unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

print()
print("Unique colors (top 10 by count):")
sorted_indices = np.argsort(counts)[::-1][:10]
for idx in sorted_indices:
    print(f"Color {tuple(unique_colors[idx])}: {counts[idx]} pixels")

In [ ]:
# ============================================================
# HRF-AV DATASET - CLEAN EXACT-COLOR EXTRACTION
# ============================================================

random.shuffle(hrf_pairs)

num_pairs = len(hrf_pairs)
num_train = int(num_pairs * 0.70)
num_val = int(num_pairs * 0.15)

hrf_train_pairs = hrf_pairs[:num_train]
hrf_val_pairs = hrf_pairs[num_train:num_train + num_val]
hrf_test_pairs = hrf_pairs[num_train + num_val:]

print("Training:", len(hrf_train_pairs))
print("Validation:", len(hrf_val_pairs))
print("Testing:", len(hrf_test_pairs))


class HrfAvDataset(Dataset):

    def __init__(self, pairs, image_size=512, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

        artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
        vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=512, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=512, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=512, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders created!")
print("Training batches:", len(hrf_train_loader))
print("Validation batches:", len(hrf_val_loader))
print("Testing batches:", len(hrf_test_loader))

# HRF-AV TRANSUNET (FROM-SCRATCH, 512x512)

In [ ]:
# ============================================================
# HRF-AV TRANSUNET (FROM-SCRATCH, 512x512)
# ============================================================

hrf_model = TransUNet512(in_channels=3, out_channels=3).to(device)

print("HRF-AV TransUNet (from-scratch) created!")
print("Total parameters:", sum(p.numel() for p in hrf_model.parameters()))

hrf_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_optimizer = optim.AdamW(hrf_model.parameters(), lr=1e-4, weight_decay=1e-4)

hrf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_path = "/content/best_hrf_av_transunet.pth"

print()
print("Starting HRF-AV TransUNet training...")
print()

for epoch in range(num_epochs):

    hrf_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_model(images)
        loss = hrf_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_model.parameters(), max_norm=1.0)

        hrf_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_model, hrf_val_loader, hrf_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_dice:
        best_hrf_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_model.state_dict(), best_hrf_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_dice:.4f}")


hrf_model.load_state_dict(torch.load(best_hrf_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_model, hrf_test_loader, hrf_loss_fn, device, num_classes=3)

print()
print("HRF-AV TRANSUNET FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# HRF-AV DATASET REBUILT AT 1024x1024

In [ ]:


hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=1024, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=1024, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=1024, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt at 1024x1024!")


# ============================================================
# RESNET TRANSUNET - FIXED FOR 1024x1024 INPUT (32x32=1024 TOKENS)
# ============================================================

class ResNetTransUNet1024(nn.Module):

    def __init__(self, out_channels=3, pretrained=True):
        super().__init__()
        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)
        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        b = self.bottleneck(e4)
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)
        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


hrf_model = ResNetTransUNet1024(out_channels=3, pretrained=True).to(device)

print("HRF-AV TransUNet (ResNet34, 1024x1024) created!")
print("Total parameters:", sum(p.numel() for p in hrf_model.parameters()))

hrf_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_encoder_params = list(hrf_model.stem.parameters()) + \
    list(hrf_model.layer1.parameters()) + \
    list(hrf_model.layer2.parameters()) + \
    list(hrf_model.layer3.parameters()) + \
    list(hrf_model.layer4.parameters())

hrf_new_params = list(hrf_model.bottleneck.parameters()) + \
    list(hrf_model.transformer.parameters()) + \
    list(hrf_model.transformer_refine.parameters()) + \
    list(hrf_model.decoder4.parameters()) + \
    list(hrf_model.decoder3.parameters()) + \
    list(hrf_model.decoder2.parameters()) + \
    list(hrf_model.decoder1.parameters()) + \
    list(hrf_model.final_refine.parameters()) + \
    list(hrf_model.output_layer.parameters())

hrf_optimizer = optim.AdamW([
    {"params": hrf_encoder_params, "lr": 2e-5},
    {"params": hrf_new_params, "lr": 1e-4}
], weight_decay=1e-4)

hrf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_path = "/content/best_hrf_av_transunet_v3.pth"

print()
print("Starting HRF-AV TransUNet v3 (1024x1024, ResNet34) training...")
print()

for epoch in range(num_epochs):

    hrf_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_model(images)
        loss = hrf_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_model.parameters(), max_norm=1.0)

        hrf_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_model, hrf_val_loader, hrf_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_dice:
        best_hrf_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_model.state_dict(), best_hrf_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV TRANSUNET V3 TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_dice:.4f}")


hrf_model.load_state_dict(torch.load(best_hrf_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_model, hrf_test_loader, hrf_loss_fn, device, num_classes=3)

print()
print("HRF-AV TRANSUNET V3 FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")


hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=1024, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=1024, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=1024, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=1, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt at 1024x1024!")
# RESNET TRANSUNET - FIXED FOR 1024x1024 INPUT (32x32=1024 TOKENS)

class ResNetTransUNet1024(nn.Module):

    def __init__(self, out_channels=3, pretrained=True):
        super().__init__()
        resnet = models.resnet34(weights="IMAGENET1K_V1" if pretrained else None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64, skip_channels=64, out_channels=64)
        self.final_refine = ResidualConvBlock(64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e0 = self.stem(x)
        e1 = self.layer1(e0)
        e2 = self.layer2(e1)
        e3 = self.layer3(e2)
        e4 = self.layer4(e3)
        b = self.bottleneck(e4)
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)
        d1 = self.final_refine(d1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


hrf_model = ResNetTransUNet1024(out_channels=3, pretrained=True).to(device)

print("HRF-AV TransUNet (ResNet34, 1024x1024) created!")
print("Total parameters:", sum(p.numel() for p in hrf_model.parameters()))

hrf_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_encoder_params = list(hrf_model.stem.parameters()) + \
    list(hrf_model.layer1.parameters()) + \
    list(hrf_model.layer2.parameters()) + \
    list(hrf_model.layer3.parameters()) + \
    list(hrf_model.layer4.parameters())

hrf_new_params = list(hrf_model.bottleneck.parameters()) + \
    list(hrf_model.transformer.parameters()) + \
    list(hrf_model.transformer_refine.parameters()) + \
    list(hrf_model.decoder4.parameters()) + \
    list(hrf_model.decoder3.parameters()) + \
    list(hrf_model.decoder2.parameters()) + \
    list(hrf_model.decoder1.parameters()) + \
    list(hrf_model.final_refine.parameters()) + \
    list(hrf_model.output_layer.parameters())

hrf_optimizer = optim.AdamW([
    {"params": hrf_encoder_params, "lr": 2e-5},
    {"params": hrf_new_params, "lr": 1e-4}
], weight_decay=1e-4)

hrf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_path = "/content/best_hrf_av_transunet_v3.pth"

print()
print("Starting HRF-AV TransUNet v3 (1024x1024, ResNet34) training...")
print()

for epoch in range(num_epochs):

    hrf_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_model(images)
        loss = hrf_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_model.parameters(), max_norm=1.0)

        hrf_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_model, hrf_val_loader, hrf_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_dice:
        best_hrf_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_model.state_dict(), best_hrf_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV TRANSUNET V3 TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_dice:.4f}")


hrf_model.load_state_dict(torch.load(best_hrf_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_model, hrf_test_loader, hrf_loss_fn, device, num_classes=3)

print()
print("HRF-AV TRANSUNET V3 FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# ============================================================
# HRF-AV DATASET AT 768x768
# ============================================================

hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=768, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=768, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=768, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt at 768x768!")


# ============================================================
# TRANSUNET (FROM-SCRATCH, GROUPNORM) - FIXED FOR 768x768 (48x48=2304 TOKENS)
# ============================================================

class TransUNet768(nn.Module):

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=2304)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


hrf_model = TransUNet768(in_channels=3, out_channels=3).to(device)

print("HRF-AV TransUNet (from-scratch, 768x768) created!")
print("Total parameters:", sum(p.numel() for p in hrf_model.parameters()))

hrf_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_optimizer = optim.AdamW(hrf_model.parameters(), lr=1e-4, weight_decay=1e-4)

hrf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_path = "/content/best_hrf_av_transunet_768.pth"

print()
print("Starting HRF-AV TransUNet (768x768, from-scratch) training...")
print()

for epoch in range(num_epochs):

    hrf_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_model(images)
        loss = hrf_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_model.parameters(), max_norm=1.0)

        hrf_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_model, hrf_val_loader, hrf_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_dice:
        best_hrf_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_model.state_dict(), best_hrf_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV TRANSUNET (768x768) TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_dice:.4f}")


hrf_model.load_state_dict(torch.load(best_hrf_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_model, hrf_test_loader, hrf_loss_fn, device, num_classes=3)

print()
print("HRF-AV TRANSUNET (768x768) FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# ============================================================
# HRF-AV DATASET WITH ELASTIC DEFORMATION AUGMENTATION
# ============================================================

class HrfAvDatasetElastic(Dataset):

    def __init__(self, pairs, image_size=768, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

        artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
        vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

            if random.random() < 0.5:
                elastic_transform = TF.elastic_transform
                displacement = torch.rand(2, 1, self.image_size // 32, self.image_size // 32) * 2 - 1
                displacement = F.interpolate(displacement.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
                displacement = displacement.permute(1, 2, 0) * 30.0

                image = TF.elastic_transform(image, displacement, interpolation=InterpolationMode.BILINEAR, fill=[0.0])
                mask = TF.elastic_transform(mask.float(), displacement, interpolation=InterpolationMode.NEAREST, fill=[0.0]).long()

        mask = mask.squeeze(0)

        return image, mask


hrf_train_dataset = HrfAvDatasetElastic(hrf_train_pairs, image_size=768, augment=True)
hrf_val_dataset = HrfAvDatasetElastic(hrf_val_pairs, image_size=768, augment=False)
hrf_test_dataset = HrfAvDatasetElastic(hrf_test_pairs, image_size=768, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt with elastic deformation augmentation!")

In [ ]:
# ============================================================
# HRF-AV DATASET WITH ELASTIC DEFORMATION - FIXED (BATCH DIM)
# ============================================================

class HrfAvDatasetElastic(Dataset):

    def __init__(self, pairs, image_size=768, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

        artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
        vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:

            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)

            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)

            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

            if random.random() < 0.5:

                small_size = self.image_size // 32

                displacement_small = (torch.rand(1, 2, small_size, small_size) * 2 - 1) * 30.0

                displacement_full = F.interpolate(
                    displacement_small, size=(self.image_size, self.image_size),
                    mode="bilinear", align_corners=False
                )

                displacement_field = displacement_full.permute(0, 2, 3, 1)

                image = TF.elastic_transform(image.unsqueeze(0), displacement_field, interpolation=InterpolationMode.BILINEAR, fill=[0.0]).squeeze(0)
                mask = TF.elastic_transform(mask.unsqueeze(0).float(), displacement_field, interpolation=InterpolationMode.NEAREST, fill=[0.0]).squeeze(0).long()

        mask = mask.squeeze(0)

        return image, mask


hrf_train_dataset = HrfAvDatasetElastic(hrf_train_pairs, image_size=768, augment=True)
hrf_val_dataset = HrfAvDatasetElastic(hrf_val_pairs, image_size=768, augment=False)
hrf_test_dataset = HrfAvDatasetElastic(hrf_test_pairs, image_size=768, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt with fixed elastic deformation (batch dim)!")

In [ ]:
hrf_model = TransUNet768(in_channels=3, out_channels=3).to(device)

print("HRF-AV TransUNet (elastic augmentation) created!")

hrf_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_optimizer = optim.AdamW(hrf_model.parameters(), lr=1e-4, weight_decay=1e-4)

hrf_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_path = "/content/best_hrf_av_elastic.pth"

print()
print("Starting HRF-AV TransUNet (elastic augmentation) training...")
print()

for epoch in range(num_epochs):

    hrf_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_model(images)
        loss = hrf_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_model.parameters(), max_norm=1.0)

        hrf_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_model, hrf_val_loader, hrf_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_dice:
        best_hrf_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_model.state_dict(), best_hrf_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV ELASTIC AUGMENTATION TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_dice:.4f}")


hrf_model.load_state_dict(torch.load(best_hrf_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_model, hrf_test_loader, hrf_loss_fn, device, num_classes=3)

print()
print("HRF-AV ELASTIC AUGMENTATION FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

GNN + Transfomer

In [ ]:
# ============================================================
# HRF-AV DATASET AT 768x768 (PLAIN, NO ELASTIC) - REBUILD
# ============================================================

hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=768, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=768, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=768, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("HRF-AV DataLoaders rebuilt at 768x768 (plain augmentation)!")


# ============================================================
# COMBINED GNN+TRANSFORMER (FROM-SCRATCH) - FIXED FOR 768x768
# ============================================================

class CombinedUNet768(nn.Module):

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=2304)
        self.bottleneck_edge_index = create_grid_edges(height=48, width=48)
        self.combined_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.combined_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


hrf_combined_model = CombinedUNet768(in_channels=3, out_channels=3).to(device)

print("HRF-AV Combined GNN+Transformer created!")
print("Total parameters:", sum(p.numel() for p in hrf_combined_model.parameters()))

hrf_combined_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_combined_optimizer = optim.AdamW(hrf_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

hrf_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_combined_path = "/content/best_hrf_av_combined.pth"

print()
print("Starting HRF-AV Combined model training...")
print()

for epoch in range(num_epochs):

    hrf_combined_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_combined_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_combined_model(images)
        loss = hrf_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_combined_model.parameters(), max_norm=1.0)

        hrf_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(hrf_combined_model, hrf_val_loader, hrf_combined_loss_fn, device, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_combined_dice:
        best_hrf_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_combined_model.state_dict(), best_hrf_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    hrf_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_combined_dice:.4f}")


hrf_combined_model.load_state_dict(torch.load(best_hrf_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(hrf_combined_model, hrf_test_loader, hrf_combined_loss_fn, device, num_classes=3)

print()
print("HRF-AV COMBINED MODEL FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
# ============================================================
# ALL METRIC FUNCTIONS (FRESH DEFINITIONS)
# ============================================================

from scipy.ndimage import label, binary_erosion, distance_transform_edt
from sklearn.metrics import roc_auc_score, roc_curve
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt


def keep_largest_component(binary_mask):
    labeled_array, num_features = label(binary_mask)
    if num_features == 0:
        return binary_mask
    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0
    largest_component_label = component_sizes.argmax()
    return labeled_array == largest_component_label


def evaluate_iou_with_postprocessing(model, loader, device, num_classes=3):

    model.eval()
    intersections = torch.zeros(num_classes - 1, device=device)
    unions = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions_np = predictions.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for batch_idx in range(predictions_np.shape[0]):
                for class_id in range(1, num_classes):
                    pred_class = (predictions_np[batch_idx] == class_id)
                    target_class = (masks_np[batch_idx] == class_id)
                    pred_class_cleaned = keep_largest_component(pred_class)
                    intersection = np.logical_and(pred_class_cleaned, target_class).sum()
                    union = np.logical_or(pred_class_cleaned, target_class).sum()
                    intersections[class_id - 1] += intersection
                    unions[class_id - 1] += union

    iou_scores = (intersections + 1e-6) / (unions + 1e-6)
    mean_iou = iou_scores.mean()
    return mean_iou, iou_scores


def hd95_single_class(pred_mask, true_mask):
    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)
    if not pred_mask.any() and not true_mask.any():
        return 0.0
    if not pred_mask.any() or not true_mask.any():
        return np.nan
    pred_boundary = pred_mask ^ binary_erosion(pred_mask)
    true_boundary = true_mask ^ binary_erosion(true_mask)
    distance_to_true = distance_transform_edt(~true_boundary)
    distance_to_pred = distance_transform_edt(~pred_boundary)
    pred_to_true = distance_to_true[pred_boundary]
    true_to_pred = distance_to_pred[true_boundary]
    all_distances = np.concatenate([pred_to_true, true_to_pred])
    return np.percentile(all_distances, 95)


def evaluate_hd95(model, loader, device, num_classes=3):

    model.eval()
    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]
                for class_id in range(1, num_classes):
                    pred_class = keep_largest_component(pred == class_id)
                    true_class = (true == class_id)
                    hd = hd95_single_class(pred_class, true_class)
                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }
    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])
    return overall_mean_hd95, mean_hd95_per_class


def evaluate_roc_auc(model, loader, device, num_classes=3):

    model.eval()
    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()
                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    auc_scores = {}
    for class_id in range(1, num_classes):
        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])
        auc_scores[class_id] = roc_auc_score(true_labels, probabilities)

    mean_auc = np.mean(list(auc_scores.values()))
    return mean_auc, auc_scores


def plot_roc_curve(model, loader, device, num_classes=3, class_names=None, title="ROC Curve"):

    model.eval()
    if class_names is None:
        class_names = [f"Class {i}" for i in range(1, num_classes)]

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()
                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    plt.figure(figsize=(7, 6))
    for i, class_id in enumerate(range(1, num_classes)):
        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])
        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        auc_value = roc_auc_score(true_labels, probabilities)
        plt.plot(fpr, tpr, linewidth=2, label=f"{class_names[i]} (AUC = {auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_ssim(model, loader, device, num_classes=3):

    model.eval()
    ssim_scores = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx].astype(np.float32)
                true = masks_cpu[batch_idx].astype(np.float32)
                score = ssim(pred, true, data_range=num_classes - 1)
                ssim_scores.append(score)

    return np.mean(ssim_scores)


print("All metric functions ready!")


# ============================================================
# FULL METRICS - HRF-AV COMBINED MODEL
# ============================================================

hrf_combined_model.load_state_dict(torch.load(best_hrf_combined_path, map_location=device))

mean_iou, iou_scores = evaluate_iou_with_postprocessing(hrf_combined_model, hrf_test_loader, device, num_classes=3)

print("HRF-AV COMBINED - IOU (with post-processing)")
print(f"Artery IoU: {iou_scores[0].item():.4f} ({iou_scores[0].item() * 100:.2f}%)")
print(f"Vein IoU: {iou_scores[1].item():.4f} ({iou_scores[1].item() * 100:.2f}%)")
print(f"Mean IoU: {mean_iou.item():.4f} ({mean_iou.item() * 100:.2f}%)")
print()

overall_hd95, hd95_per_class = evaluate_hd95(hrf_combined_model, hrf_test_loader, device, num_classes=3)

print("HRF-AV COMBINED - HD95 (with post-processing, in pixels)")
print(f"Artery HD95: {hd95_per_class[1]:.4f} pixels")
print(f"Vein HD95: {hd95_per_class[2]:.4f} pixels")
print(f"Mean HD95: {overall_hd95:.4f} pixels")
print()

mean_auc, auc_scores = evaluate_roc_auc(hrf_combined_model, hrf_test_loader, device, num_classes=3)

print("HRF-AV COMBINED - ROC-AUC")
print(f"Artery AUC: {auc_scores[1]:.4f} ({auc_scores[1] * 100:.2f}%)")
print(f"Vein AUC: {auc_scores[2]:.4f} ({auc_scores[2] * 100:.2f}%)")
print(f"Mean AUC: {mean_auc:.4f} ({mean_auc * 100:.2f}%)")

plot_roc_curve(
    hrf_combined_model, hrf_test_loader, device, num_classes=3,
    class_names=["Artery", "Vein"], title="HRF-AV Combined - ROC Curve"
)

mean_ssim_value = evaluate_ssim(hrf_combined_model, hrf_test_loader, device, num_classes=3)

print()
print("HRF-AV COMBINED - SSIM")
print(f"Mean SSIM: {mean_ssim_value:.4f}")

In [ ]:
checkpoint_paths = {
    "HRF-AV TransUNet (512x512, 71.83%)": "/content/best_hrf_av_transunet.pth",
    "HRF-AV TransUNet v2 (ResNet34, 1024x1024)": "/content/best_hrf_av_transunet_v3.pth",
    "HRF-AV TransUNet (768x768, 73.21%, BEST)": "/content/best_hrf_av_transunet_768.pth",
    "HRF-AV Elastic": "/content/best_hrf_av_elastic.pth",
    "HRF-AV Combined": "/content/best_hrf_av_combined.pth",
}

for name, path in checkpoint_paths.items():
    exists = os.path.exists(path)
    print(f"{'EXISTS' if exists else 'MISSING'} - {name}")

In [ ]:
# ============================================================
# BASIC SETUP (FRESH RUNTIME)
# ============================================================

!pip install -q kagglehub torch-geometric

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torch_geometric.nn import SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

In [ ]:
# ============================================================
# RE-DOWNLOAD HRF IMAGES AND AV MASKS
# ============================================================

!wget -q -O /content/hrf_all.zip "https://www5.cs.fau.de/fileadmin/research/datasets/fundus-images/all.zip"
!git clone -q https://github.com/rubenhx/av-segmentation.git /content/hrf_av_repo

import zipfile

with zipfile.ZipFile("/content/hrf_all.zip", "r") as zip_ref:
    zip_ref.extractall("/content/hrf_images")

hrf_images_path = "/content/hrf_images"
hrf_repo_path = "/content/hrf_av_repo"

hrf_images_folder = os.path.join(hrf_images_path, "images")
hrf_av_folder = os.path.join(hrf_repo_path, "HRF_AV_GT")

av_mask_files = os.listdir(hrf_av_folder)
image_files = os.listdir(hrf_images_folder)

hrf_pairs = []

for mask_filename in av_mask_files:
    base_id = mask_filename.replace("_AVmanual.png", "")
    matching_image = None
    for image_filename in image_files:
        if image_filename.lower().startswith(base_id.lower() + "."):
            matching_image = image_filename
            break
    if matching_image:
        image_path = os.path.join(hrf_images_folder, matching_image)
        mask_path = os.path.join(hrf_av_folder, mask_filename)
        hrf_pairs.append((image_path, mask_path))

print("Total matched pairs:", len(hrf_pairs))


# Recreate the SAME split as before (same seed, same shuffle order)

random.shuffle(hrf_pairs)

num_pairs = len(hrf_pairs)
num_train = int(num_pairs * 0.70)
num_val = int(num_pairs * 0.15)

hrf_train_pairs = hrf_pairs[:num_train]
hrf_val_pairs = hrf_pairs[num_train:num_train + num_val]
hrf_test_pairs = hrf_pairs[num_train + num_val:]

print("Training:", len(hrf_train_pairs))
print("Validation:", len(hrf_val_pairs))
print("Testing:", len(hrf_test_pairs))

# REBUILD SHARED MODEL COMPONENTS

In [ ]:


class ResidualConvBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.act = nn.GELU()
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        residual = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + residual)


class TransformerBlock(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, num_tokens=256):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=0.1, activation="gelu", norm_first=True, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.shape[1], :]
        x = self.transformer(x)
        return self.norm(x)


class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.conv = ResidualConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class TransUNet768(nn.Module):

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=2304)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


class MulticlassDiceLoss(nn.Module):

    def __init__(self, num_classes=3, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)
        total_dice_loss = 0.0
        foreground_classes = 0
        for class_idx in range(1, self.num_classes):
            pred_class = probabilities[:, class_idx]
            target_class = (targets == class_idx).float()
            intersection = (pred_class * target_class).sum()
            denominator = pred_class.sum() + target_class.sum()
            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
            total_dice_loss += (1.0 - dice)
            foreground_classes += 1
        return total_dice_loss / foreground_classes


class ACDCLoss(nn.Module):

    def __init__(self, num_classes=3, gamma=2.0, smooth=1e-5, class_weights=None):
        super().__init__()
        self.num_classes = num_classes
        self.gamma = gamma
        self.smooth = smooth
        if class_weights is None:
            class_weights = [0.20] + [1.00] * (num_classes - 1)
        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.dice_loss = MulticlassDiceLoss(num_classes=num_classes)

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = (((1 - pt) ** self.gamma) * ce_loss).mean()
        dice_loss = self.dice_loss(logits, targets)
        return (0.40 * focal_loss) + (1.00 * dice_loss)


def evaluate(model, loader, loss_fn, device, num_classes=3):

    model.eval()
    total_loss = 0.0
    intersections = torch.zeros(num_classes - 1, device=device)
    pred_sums = torch.zeros(num_classes - 1, device=device)
    target_sums = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            loss = loss_fn(outputs, masks)
            total_loss += loss.item() * images.size(0)
            predictions = torch.argmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                pred_class = (predictions == class_id).float()
                target_class = (masks == class_id).float()
                intersections[class_id - 1] += (pred_class * target_class).sum()
                pred_sums[class_id - 1] += pred_class.sum()
                target_sums[class_id - 1] += target_class.sum()

    dice_scores = (2.0 * intersections + 1e-5) / (pred_sums + target_sums + 1e-5)
    mean_dice = dice_scores.mean()
    average_loss = total_loss / len(loader.dataset)
    return average_loss, mean_dice, dice_scores


# ============================================================
# HRF-AV DATASET CLASS
# ============================================================

class HrfAvDataset(Dataset):

    def __init__(self, pairs, image_size=768, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

        artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
        vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

        combined_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        combined_mask[artery_mask] = 1
        combined_mask[vein_mask] = 2

        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1)
        mask = torch.tensor(combined_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        mask = mask.squeeze(0)

        return image, mask


hrf_train_dataset = HrfAvDataset(hrf_train_pairs, image_size=768, augment=True)
hrf_val_dataset = HrfAvDataset(hrf_val_pairs, image_size=768, augment=False)
hrf_test_dataset = HrfAvDataset(hrf_test_pairs, image_size=768, augment=False)

hrf_train_loader = DataLoader(hrf_train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
hrf_val_loader = DataLoader(hrf_val_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
hrf_test_loader = DataLoader(hrf_test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

print("Everything rebuilt! Ready to train.")

In [ ]:
# ============================================================
# PATCH-BASED HRF-AV DATASET (TRAINING: RANDOM PATCHES)
# ============================================================

class HrfAvPatchTrainDataset(Dataset):

    def __init__(self, pairs, patch_size=512, patches_per_image=8, augment=True):
        self.pairs = pairs
        self.patch_size = patch_size
        self.patches_per_image = patches_per_image
        self.augment = augment

    def __len__(self):
        return len(self.pairs) * self.patches_per_image

    def __getitem__(self, idx):

        pair_idx = idx // self.patches_per_image
        image_path, mask_path = self.pairs[pair_idx]

        image = Image.open(image_path).convert("RGB")
        image = np.array(image, dtype=np.float32)

        mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

        artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
        vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

        full_mask = np.zeros(artery_mask.shape, dtype=np.int64)
        full_mask[artery_mask] = 1
        full_mask[vein_mask] = 2

        height, width = full_mask.shape

        max_y = height - self.patch_size
        max_x = width - self.patch_size

        top = random.randint(0, max_y)
        left = random.randint(0, max_x)

        image_patch = image[top:top + self.patch_size, left:left + self.patch_size, :]
        mask_patch = full_mask[top:top + self.patch_size, left:left + self.patch_size]

        image_patch = torch.tensor(image_patch, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask_patch = torch.tensor(mask_patch, dtype=torch.long)

        if self.augment:
            if random.random() < 0.5:
                image_patch = TF.hflip(image_patch)
                mask_patch = TF.hflip(mask_patch)
            if random.random() < 0.5:
                image_patch = TF.vflip(image_patch)
                mask_patch = TF.vflip(mask_patch)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image_patch = torch.clamp(image_patch * factor, 0.0, 1.0)

        return image_patch, mask_patch


# ============================================================
# PATCH-BASED HRF-AV DATASET (VAL/TEST: FULL GRID, NO OVERLAP)
# ============================================================

def extract_full_image_and_mask(image_path, mask_path):

    image = Image.open(image_path).convert("RGB")
    image = np.array(image, dtype=np.float32)

    mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)

    artery_mask = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
    vein_mask = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)

    full_mask = np.zeros(artery_mask.shape, dtype=np.int64)
    full_mask[artery_mask] = 1
    full_mask[vein_mask] = 2

    return image, full_mask


def evaluate_with_patches(model, pairs, device, patch_size=512, num_classes=3):

    model.eval()

    intersections = torch.zeros(num_classes - 1, device=device)
    pred_sums = torch.zeros(num_classes - 1, device=device)
    target_sums = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():

        for image_path, mask_path in pairs:

            image, full_mask = extract_full_image_and_mask(image_path, mask_path)

            height, width, _ = image.shape

            full_prediction = np.zeros((height, width), dtype=np.int64)

            for top in range(0, height, patch_size):
                for left in range(0, width, patch_size):

                    bottom = min(top + patch_size, height)
                    right = min(left + patch_size, width)

                    patch_top = max(0, bottom - patch_size)
                    patch_left = max(0, right - patch_size)

                    image_patch = image[patch_top:patch_top + patch_size, patch_left:patch_left + patch_size, :]

                    image_tensor = torch.tensor(image_patch, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
                    image_tensor = image_tensor.to(device)

                    output = model(image_tensor)
                    prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

                    full_prediction[patch_top:patch_top + patch_size, patch_left:patch_left + patch_size] = prediction

            for class_id in range(1, num_classes):
                pred_class = (full_prediction == class_id)
                target_class = (full_mask == class_id)
                intersections[class_id - 1] += np.logical_and(pred_class, target_class).sum()
                pred_sums[class_id - 1] += pred_class.sum()
                target_sums[class_id - 1] += target_class.sum()

    dice_scores = (2.0 * intersections + 1e-5) / (pred_sums + target_sums + 1e-5)
    mean_dice = dice_scores.mean()

    return mean_dice, dice_scores


print("Patch-based dataset and evaluation functions ready!")


# ============================================================
# BUILD PATCH TRAINING DATASET
# ============================================================

hrf_patch_train_dataset = HrfAvPatchTrainDataset(hrf_train_pairs, patch_size=512, patches_per_image=8, augment=True)
hrf_patch_train_loader = DataLoader(hrf_patch_train_dataset, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

print("Patch training dataset created!")
print("Total patches per epoch:", len(hrf_patch_train_dataset))
print("Training batches:", len(hrf_patch_train_loader))

# TRANSUNET (FROM-SCRATCH) FOR 512x512 PATCHES (1024 TOKENS)

In [ ]:


class TransUNet512Patch(nn.Module):

    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.transformer = TransformerBlock(embed_dim=512, num_heads=8, num_layers=4, num_tokens=1024)
        self.transformer_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=512, out_channels=512)
        self.decoder3 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder2 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder1 = DecoderBlock(in_channels=128, skip_channels=64, out_channels=64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, kernel_size=2))
        e3 = self.encoder3(F.max_pool2d(e2, kernel_size=2))
        e4 = self.encoder4(F.max_pool2d(e3, kernel_size=2))
        b = self.bottleneck(F.max_pool2d(e4, kernel_size=2))
        batch_size, channels, height, width = b.shape
        tokens = b.flatten(2).transpose(1, 2)
        tokens = self.transformer(tokens)
        b = tokens.transpose(1, 2).reshape(batch_size, channels, height, width)
        b = self.transformer_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


hrf_patch_model = TransUNet512Patch(in_channels=3, out_channels=3).to(device)

print("HRF-AV Patch-based TransUNet created!")
print("Total parameters:", sum(p.numel() for p in hrf_patch_model.parameters()))

hrf_patch_loss_fn = ACDCLoss(num_classes=3, class_weights=[0.05, 1.00, 1.00]).to(device)

hrf_patch_optimizer = optim.AdamW(hrf_patch_model.parameters(), lr=1e-4, weight_decay=1e-4)

hrf_patch_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    hrf_patch_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_hrf_patch_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_hrf_patch_path = "/content/best_hrf_av_patch.pth"

print()
print("Starting HRF-AV patch-based training...")
print()

for epoch in range(num_epochs):

    hrf_patch_model.train()

    total_train_loss = 0.0

    for images, masks in hrf_patch_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        hrf_patch_optimizer.zero_grad(set_to_none=True)

        outputs = hrf_patch_model(images)
        loss = hrf_patch_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(hrf_patch_model.parameters(), max_norm=1.0)

        hrf_patch_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(hrf_patch_train_loader.dataset)

    val_mean_dice, val_dice = evaluate_with_patches(hrf_patch_model, hrf_val_pairs, device, patch_size=512, num_classes=3)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_hrf_patch_dice:
        best_hrf_patch_dice = current_dice
        epochs_without_improvement = 0
        torch.save(hrf_patch_model.state_dict(), best_hrf_patch_path)
        new_best = True
    else:
        epochs_without_improvement += 1
    hrf_patch_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Mean Dice (full-image): {current_dice:.4f} | "
        f"Artery: {val_dice[0].item():.4f} | "
        f"Vein: {val_dice[1].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("HRF-AV PATCH-BASED TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_hrf_patch_dice:.4f}")


hrf_patch_model.load_state_dict(torch.load(best_hrf_patch_path, map_location=device))

test_mean_dice, test_dice = evaluate_with_patches(hrf_patch_model, hrf_test_pairs, device, patch_size=512, num_classes=3)

print()
print("HRF-AV PATCH-BASED FINAL TEST RESULTS")
print(f"Artery Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Vein Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)

shutil.copy("/content/best_hrf_av_patch.pth", os.path.join(backup_folder, "best_hrf_av_patch.pth"))

print("HRF-AV patch-based model backed up to Google Drive!")

## FULL METRICS FOR PATCH-BASED MODEL (IoU, HD95, ROC-AUC, SSIM)

In [ ]:
# ============================================================
# METRIC HELPER FUNCTIONS
# ============================================================

from scipy.ndimage import label, binary_erosion, distance_transform_edt
from sklearn.metrics import roc_auc_score
from skimage.metrics import structural_similarity as ssim


def extract_full_image_and_mask(image_path, mask_path):
    image = np.array(Image.open(image_path).convert("RGB"), dtype=np.float32)
    mask_img = np.array(Image.open(mask_path).convert("RGB"), dtype=np.int32)
    artery = (mask_img[:, :, 0] == 255) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 0)
    vein = (mask_img[:, :, 0] == 0) & (mask_img[:, :, 1] == 0) & (mask_img[:, :, 2] == 255)
    full_mask = np.zeros(artery.shape, dtype=np.int64)
    full_mask[artery] = 1
    full_mask[vein] = 2
    return image, full_mask


def predict_full_image(model, image, device, patch_size=512, num_classes=3):
    h, w, _ = image.shape
    full_pred = np.zeros((h, w), dtype=np.int64)
    full_probs = np.zeros((num_classes, h, w), dtype=np.float32)
    with torch.no_grad():
        for top in range(0, h, patch_size):
            for left in range(0, w, patch_size):
                bottom = min(top + patch_size, h)
                right = min(left + patch_size, w)
                pt_, pl_ = max(0, bottom - patch_size), max(0, right - patch_size)
                patch = image[pt_:pt_ + patch_size, pl_:pl_ + patch_size, :]
                tensor = torch.tensor(patch, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device) / 255.0
                probs = torch.softmax(model(tensor), dim=1).squeeze(0).cpu().numpy()
                full_pred[pt_:pt_ + patch_size, pl_:pl_ + patch_size] = np.argmax(probs, axis=0)
                full_probs[:, pt_:pt_ + patch_size, pl_:pl_ + patch_size] = probs
    return full_pred, full_probs


def keep_largest_component(mask):
    labeled, n = label(mask)
    if n == 0:
        return mask
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    return labeled == sizes.argmax()


def hd95_single_class(pred, true):
    pred, true = pred.astype(bool), true.astype(bool)
    if not pred.any() and not true.any():
        return 0.0
    if not pred.any() or not true.any():
        return np.nan
    pb = pred ^ binary_erosion(pred)
    tb = true ^ binary_erosion(true)
    d_true = distance_transform_edt(~tb)
    d_pred = distance_transform_edt(~pb)
    dists = np.concatenate([d_true[pb], d_pred[tb]])
    return np.percentile(dists, 95)


def evaluate_all_metrics(model, pairs, device, patch_size=512, num_classes=3):
    model.eval()
    iou_i = np.zeros(num_classes - 1)
    iou_u = np.zeros(num_classes - 1)
    hd_vals = {c: [] for c in range(1, num_classes)}
    true_labels = {c: [] for c in range(1, num_classes)}
    probs_all = {c: [] for c in range(1, num_classes)}
    ssim_scores = []

    for image_path, mask_path in pairs:
        image, full_mask = extract_full_image_and_mask(image_path, mask_path)
        full_pred, full_probs = predict_full_image(model, image, device, patch_size, num_classes)

        for c in range(1, num_classes):
            pc = keep_largest_component(full_pred == c)
            tc = (full_mask == c)
            iou_i[c - 1] += np.logical_and(pc, tc).sum()
            iou_u[c - 1] += np.logical_or(pc, tc).sum()
            hd = hd95_single_class(pc, tc)
            if not np.isnan(hd):
                hd_vals[c].append(hd)
            true_labels[c].append(tc.ravel())
            probs_all[c].append(full_probs[c].ravel())

        ssim_scores.append(ssim(full_pred.astype(np.float32), full_mask.astype(np.float32), data_range=num_classes - 1))

    iou_scores = (iou_i + 1e-6) / (iou_u + 1e-6)
    hd95_per_class = {c: (np.mean(v) if v else np.nan) for c, v in hd_vals.items()}
    auc_scores = {c: roc_auc_score(np.concatenate(true_labels[c]), np.concatenate(probs_all[c])) for c in range(1, num_classes)}

    return {
        "iou_scores": iou_scores, "mean_iou": iou_scores.mean(),
        "hd95_per_class": hd95_per_class, "mean_hd95": np.mean([v for v in hd95_per_class.values() if not np.isnan(v)]),
        "auc_scores": auc_scores, "mean_auc": np.mean(list(auc_scores.values())),
        "mean_ssim": np.mean(ssim_scores)
    }


print("Metric functions ready!")


# ============================================================
# RUN METRICS
# ============================================================

results = evaluate_all_metrics(hrf_patch_model, hrf_test_pairs, device)

print()
print("HRF-AV PATCH-BASED MODEL - FULL METRICS")
print(f"IoU - Artery: {results['iou_scores'][0]*100:.2f}%, Vein: {results['iou_scores'][1]*100:.2f}%, Mean: {results['mean_iou']*100:.2f}%")
print(f"HD95 - Artery: {results['hd95_per_class'][1]:.2f}px, Vein: {results['hd95_per_class'][2]:.2f}px, Mean: {results['mean_hd95']:.2f}px")
print(f"ROC-AUC - Artery: {results['auc_scores'][1]*100:.2f}%, Vein: {results['auc_scores'][2]*100:.2f}%, Mean: {results['mean_auc']*100:.2f}%")
print(f"SSIM: {results['mean_ssim']:.4f}")

In [ ]:
# ============================================================
# DOWNLOAD TEETH SEGMENTATION DATASET
# ============================================================

import kagglehub

teeth_path = kagglehub.dataset_download(
    "humansintheloop/teeth-segmentation-on-dental-x-ray-images"
)

print("Teeth dataset downloaded!")
print("Dataset path:", teeth_path)

In [ ]:
for root, dirs, files in os.walk(teeth_path):

    level = root.replace(teeth_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    subindent = "  " * (level + 1)

    for f in files[:5]:
        print(f"{subindent}{f}")

    if len(files) > 5:
        print(f"{subindent}... and {len(files) - 5} more files")

In [ ]:
teeth_images_folder = os.path.join(teeth_path, "Teeth Segmentation PNG", "d2", "img")
teeth_masks_folder = os.path.join(teeth_path, "Teeth Segmentation PNG", "d2", "masks_machine")

image_files = sorted(os.listdir(teeth_images_folder))
mask_files = sorted(os.listdir(teeth_masks_folder))

print("Total images:", len(image_files))
print("Total masks:", len(mask_files))

sample_image_name = image_files[0]
sample_mask_name = sample_image_name.replace(".jpg", ".png")

sample_image_path = os.path.join(teeth_images_folder, sample_image_name)
sample_mask_path = os.path.join(teeth_masks_folder, sample_mask_name)

print()
print("Sample image:", sample_image_name)
print("Sample mask exists:", os.path.exists(sample_mask_path))

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print("Image size:", sample_image.size, "mode:", sample_image.mode)
print("Mask size:", sample_mask.size, "mode:", sample_mask.mode)

mask_array = np.array(sample_mask)
print("Mask array shape:", mask_array.shape)
print("Unique values in mask:", np.unique(mask_array))

In [ ]:
mask_array = np.array(sample_mask)

pixels = mask_array.reshape(-1, 3)
unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)

print("Total distinct colors found:", len(unique_colors))
print()
print("Colors and pixel counts:")
for color, count in zip(unique_colors, counts):
    print(f"Color {tuple(color)}: {count} pixels")

In [ ]:
import json

color_map_path = os.path.join(teeth_path, "Teeth Segmentation JSON", "obj_class_to_machine_color.json")

with open(color_map_path, "r") as f:
    color_mapping = json.load(f)

print("Tooth number to color mapping:")
for tooth_id, color in color_mapping.items():
    print(f"Tooth {tooth_id}: {color}")

In [ ]:
json_masks_folder = os.path.join(teeth_path, "Teeth Segmentation JSON", "d2", "masks_machine")

sample_json_mask_path = os.path.join(json_masks_folder, sample_mask_name)

sample_json_mask = Image.open(sample_json_mask_path)

print("JSON folder mask size:", sample_json_mask.size, "mode:", sample_json_mask.mode)

json_mask_array = np.array(sample_json_mask)
print("JSON mask array shape:", json_mask_array.shape)
print("Unique values in JSON mask:", np.unique(json_mask_array))

In [ ]:
# ============================================================
# TEETH DATASET - 4-QUADRANT GROUPING
# ============================================================

import json

color_map_path = os.path.join(teeth_path, "Teeth Segmentation JSON", "obj_class_to_machine_color.json")

with open(color_map_path, "r") as f:
    color_mapping = json.load(f)

# Build tooth_number -> pixel_value lookup (ignore the odd "13(polygon)" duplicate)

tooth_to_pixel_value = {}

for key, value in color_mapping.items():
    if "(polygon)" in key:
        continue
    tooth_number = int(key.replace("Tooth ", ""))
    pixel_value = value[0]
    tooth_to_pixel_value[tooth_number] = pixel_value


def get_quadrant(tooth_number):
    if 1 <= tooth_number <= 8:
        return 1  # upper right
    elif 9 <= tooth_number <= 16:
        return 2  # upper left
    elif 17 <= tooth_number <= 24:
        return 3  # lower left
    elif 25 <= tooth_number <= 32:
        return 4  # lower right
    return 0


# Build pixel_value -> quadrant lookup table (fast remapping)

pixel_value_to_quadrant = np.zeros(256, dtype=np.int64)

for tooth_number, pixel_value in tooth_to_pixel_value.items():
    pixel_value_to_quadrant[pixel_value] = get_quadrant(tooth_number)


json_images_folder = os.path.join(teeth_path, "Teeth Segmentation JSON", "d2", "img")
json_masks_folder = os.path.join(teeth_path, "Teeth Segmentation JSON", "d2", "masks_machine")

teeth_image_files = sorted(os.listdir(json_images_folder))

teeth_pairs = []

for image_filename in teeth_image_files:
    mask_filename = image_filename.replace(".jpg", ".png")
    image_path = os.path.join(json_images_folder, image_filename)
    mask_path = os.path.join(json_masks_folder, mask_filename)
    if os.path.exists(mask_path):
        teeth_pairs.append((image_path, mask_path))

print("Total teeth image/mask pairs found:", len(teeth_pairs))


random.shuffle(teeth_pairs)

num_pairs = len(teeth_pairs)
num_train = int(num_pairs * 0.70)
num_val = int(num_pairs * 0.15)

teeth_train_pairs = teeth_pairs[:num_train]
teeth_val_pairs = teeth_pairs[num_train:num_train + num_val]
teeth_test_pairs = teeth_pairs[num_train + num_val:]

print("Training:", len(teeth_train_pairs))
print("Validation:", len(teeth_val_pairs))
print("Testing:", len(teeth_test_pairs))


class TeethDataset(Dataset):

    def __init__(self, pairs, image_size=256, augment=False):
        self.pairs = pairs
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):

        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("L")
        image = np.array(image, dtype=np.float32)

        mask = Image.open(mask_path).convert("L")
        mask = np.array(mask, dtype=np.int64)

        quadrant_mask = pixel_value_to_quadrant[mask]

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)
        mask = torch.tensor(quadrant_mask, dtype=torch.long).unsqueeze(0)

        image = F.interpolate(image.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.float().unsqueeze(0), size=(self.image_size, self.image_size), mode="nearest").squeeze(0).long()

        image = image / 255.0

        if self.augment:
            if random.random() < 0.5:
                angle = random.uniform(-10, 10)
                image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR, fill=0)
                mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST, fill=0)
            if random.random() < 0.3:
                factor = random.uniform(0.85, 1.15)
                image = torch.clamp(image * factor, 0.0, 1.0)
            if random.random() < 0.25:
                noise = torch.randn_like(image) * 0.01
                image = torch.clamp(image + noise, 0.0, 1.0)

        image = image.repeat(3, 1, 1)
        mask = mask.squeeze(0)

        return image, mask


teeth_train_dataset = TeethDataset(teeth_train_pairs, image_size=256, augment=True)
teeth_val_dataset = TeethDataset(teeth_val_pairs, image_size=256, augment=False)
teeth_test_dataset = TeethDataset(teeth_test_pairs, image_size=256, augment=False)

teeth_train_loader = DataLoader(teeth_train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
teeth_val_loader = DataLoader(teeth_val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
teeth_test_loader = DataLoader(teeth_test_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print("Teeth DataLoaders created!")
print("Training batches:", len(teeth_train_loader))
print("Validation batches:", len(teeth_val_loader))
print("Testing batches:", len(teeth_test_loader))

In [ ]:
# ============================================================
# TEETH TRANSUNET (FROM-SCRATCH, 5 CLASSES)
# ============================================================

teeth_model = TransUNet(in_channels=3, out_channels=5).to(device)

print("Teeth TransUNet created!")
print("Total parameters:", sum(p.numel() for p in teeth_model.parameters()))

teeth_loss_fn = ACDCLoss(num_classes=5, class_weights=[0.10, 1.00, 1.00, 1.00, 1.00]).to(device)

teeth_optimizer = optim.AdamW(teeth_model.parameters(), lr=1e-4, weight_decay=1e-4)

teeth_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    teeth_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_teeth_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_teeth_path = "/content/best_teeth_transunet.pth"

print()
print("Starting Teeth TransUNet training...")
print()

for epoch in range(num_epochs):

    teeth_model.train()

    total_train_loss = 0.0

    for images, masks in teeth_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        teeth_optimizer.zero_grad(set_to_none=True)

        outputs = teeth_model(images)
        loss = teeth_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(teeth_model.parameters(), max_norm=1.0)

        teeth_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(teeth_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(teeth_model, teeth_val_loader, teeth_loss_fn, device, num_classes=5)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_teeth_dice:
        best_teeth_dice = current_dice
        epochs_without_improvement = 0
        torch.save(teeth_model.state_dict(), best_teeth_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    teeth_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"UR: {val_dice[0].item():.4f} | "
        f"UL: {val_dice[1].item():.4f} | "
        f"LL: {val_dice[2].item():.4f} | "
        f"LR: {val_dice[3].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("TEETH TRANSUNET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_teeth_dice:.4f}")


teeth_model.load_state_dict(torch.load(best_teeth_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(teeth_model, teeth_test_loader, teeth_loss_fn, device, num_classes=5)

print()
print("TEETH TRANSUNET FINAL TEST RESULTS")
print(f"Upper Right Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Upper Left Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Lower Left Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Lower Right Dice: {test_dice[3].item():.4f} ({test_dice[3].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)

shutil.copy("/content/best_teeth_transunet.pth", os.path.join(backup_folder, "best_teeth_transunet.pth"))

print("Teeth TransUNet backed up to Google Drive!")

In [ ]:
# ============================================================
# GRAPH COMPONENTS (REDEFINED FRESH)
# ============================================================

!pip install -q torch-geometric
from torch_geometric.nn import SAGEConv


def create_grid_edges(height, width):
    edges = []
    for row in range(height):
        for col in range(width):
            node = row * width + col
            if col < width - 1:
                right = node + 1
                edges.append([node, right])
                edges.append([right, node])
            if row < height - 1:
                down = node + width
                edges.append([node, down])
                edges.append([down, node])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index


class GraphBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, hidden_dim=512, num_layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        in_dim = feature_dim
        for _ in range(num_layers):
            self.layers.append(SAGEConv(in_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
            in_dim = hidden_dim
        self.act = nn.GELU()

    def forward(self, x, edge_index):
        batch_size = x.shape[0]
        outputs = []
        for b in range(batch_size):
            h = x[b]
            for layer, norm in zip(self.layers, self.norms):
                residual = h
                h = layer(h, edge_index)
                h = norm(h)
                h = self.act(h)
                if h.shape == residual.shape:
                    h = h + residual
            outputs.append(h)
        return torch.stack(outputs, dim=0)


class MSCMRGraphUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=5):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=512, hidden_dim=512, num_layers=4)
        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)
        self.graph_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(512, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))
        b = self.bottleneck(F.max_pool2d(e4, 2))
        bs, c, h, w = b.shape
        graph_input = b.flatten(2).transpose(1, 2)
        graph_output = self.graph_reasoning(graph_input, edge_index)
        b = graph_output.transpose(1, 2).reshape(bs, c, h, w)
        b = self.graph_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


print("Graph components ready!")


# ============================================================
# TEETH GRAPH U-NET (FROM-SCRATCH, 5 CLASSES)
# ============================================================

teeth_graph_model = MSCMRGraphUNet(in_channels=3, out_channels=5).to(device)

print("Teeth Graph U-Net created!")
print("Total parameters:", sum(p.numel() for p in teeth_graph_model.parameters()))

teeth_graph_loss_fn = ACDCLoss(num_classes=5, class_weights=[0.10, 1.00, 1.00, 1.00, 1.00]).to(device)

teeth_graph_optimizer = optim.AdamW(teeth_graph_model.parameters(), lr=1e-4, weight_decay=1e-4)

teeth_graph_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    teeth_graph_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_teeth_graph_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_teeth_graph_path = "/content/best_teeth_graph_unet.pth"

print()
print("Starting Teeth Graph U-Net training...")
print()

for epoch in range(num_epochs):

    teeth_graph_model.train()

    total_train_loss = 0.0

    for images, masks in teeth_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        teeth_graph_optimizer.zero_grad(set_to_none=True)

        outputs = teeth_graph_model(images)
        loss = teeth_graph_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(teeth_graph_model.parameters(), max_norm=1.0)

        teeth_graph_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(teeth_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(teeth_graph_model, teeth_val_loader, teeth_graph_loss_fn, device, num_classes=5)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_teeth_graph_dice:
        best_teeth_graph_dice = current_dice
        epochs_without_improvement = 0
        torch.save(teeth_graph_model.state_dict(), best_teeth_graph_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    teeth_graph_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"UR: {val_dice[0].item():.4f} | "
        f"UL: {val_dice[1].item():.4f} | "
        f"LL: {val_dice[2].item():.4f} | "
        f"LR: {val_dice[3].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("TEETH GRAPH U-NET TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_teeth_graph_dice:.4f}")


teeth_graph_model.load_state_dict(torch.load(best_teeth_graph_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(teeth_graph_model, teeth_test_loader, teeth_graph_loss_fn, device, num_classes=5)

print()
print("TEETH GRAPH U-NET FINAL TEST RESULTS")
print(f"Upper Right Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Upper Left Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Lower Left Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Lower Right Dice: {test_dice[3].item():.4f} ({test_dice[3].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")


# ============================================================
# BACKUP IMMEDIATELY
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)
shutil.copy(best_teeth_graph_path, os.path.join(backup_folder, "best_teeth_graph_unet.pth"))

print("Backed up to Google Drive!")

In [ ]:
# ============================================================
# COMBINED GNN+TRANSFORMER COMPONENTS
# ============================================================

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, num_tokens=256):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_tokens, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim * 4,
            dropout=0.1, activation="gelu", norm_first=True, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.pos_embedding[:, :x.shape[1], :]
        x = self.transformer(x)
        return self.norm(x)


class CombinedBottleneckBlock(nn.Module):

    def __init__(self, feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=256):
        super().__init__()
        self.graph_reasoning = GraphBottleneckBlock(feature_dim=feature_dim, hidden_dim=feature_dim, num_layers=num_graph_layers)
        self.transformer_reasoning = TransformerBlock(embed_dim=feature_dim, num_heads=num_heads, num_layers=num_transformer_layers, num_tokens=num_tokens)

    def forward(self, x, edge_index):
        x = self.graph_reasoning(x, edge_index)
        x = self.transformer_reasoning(x)
        return x


class MSCMRCombinedUNet(nn.Module):

    def __init__(self, in_channels=3, out_channels=5):
        super().__init__()
        self.encoder1 = ResidualConvBlock(in_channels, 64)
        self.encoder2 = ResidualConvBlock(64, 128)
        self.encoder3 = ResidualConvBlock(128, 256)
        self.encoder4 = ResidualConvBlock(256, 512)
        self.bottleneck = ResidualConvBlock(512, 512)
        self.combined_reasoning = CombinedBottleneckBlock(feature_dim=512, num_graph_layers=4, num_heads=8, num_transformer_layers=4, num_tokens=256)
        self.bottleneck_edge_index = create_grid_edges(height=16, width=16)
        self.combined_refine = ResidualConvBlock(512, 512)
        self.decoder4 = DecoderBlock(512, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[2:]
        device = x.device
        edge_index = self.bottleneck_edge_index.to(device)
        e1 = self.encoder1(x)
        e2 = self.encoder2(F.max_pool2d(e1, 2))
        e3 = self.encoder3(F.max_pool2d(e2, 2))
        e4 = self.encoder4(F.max_pool2d(e3, 2))
        b = self.bottleneck(F.max_pool2d(e4, 2))
        bs, c, h, w = b.shape
        combined_input = b.flatten(2).transpose(1, 2)
        combined_output = self.combined_reasoning(combined_input, edge_index)
        b = combined_output.transpose(1, 2).reshape(bs, c, h, w)
        b = self.combined_refine(b)
        d4 = self.decoder4(b, e4)
        d3 = self.decoder3(d4, e3)
        d2 = self.decoder2(d3, e2)
        d1 = self.decoder1(d2, e1)
        d1 = F.interpolate(d1, size=input_size, mode="bilinear", align_corners=False)
        return self.output_layer(d1)


print("Combined components ready!")


# ============================================================
# TEETH COMBINED GNN+TRANSFORMER (FROM-SCRATCH, 5 CLASSES)
# ============================================================

teeth_combined_model = MSCMRCombinedUNet(in_channels=3, out_channels=5).to(device)

print("Teeth Combined model created!")
print("Total parameters:", sum(p.numel() for p in teeth_combined_model.parameters()))

teeth_combined_loss_fn = ACDCLoss(num_classes=5, class_weights=[0.10, 1.00, 1.00, 1.00, 1.00]).to(device)

teeth_combined_optimizer = optim.AdamW(teeth_combined_model.parameters(), lr=1e-4, weight_decay=1e-4)

teeth_combined_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    teeth_combined_optimizer, mode="max", factor=0.5, patience=8
)

print("Optimizer ready!")


num_epochs = 150

best_teeth_combined_dice = 0.0
epochs_without_improvement = 0
early_stop_patience = 15

best_teeth_combined_path = "/content/best_teeth_combined_unet.pth"

print()
print("Starting Teeth Combined model training...")
print()

for epoch in range(num_epochs):

    teeth_combined_model.train()

    total_train_loss = 0.0

    for images, masks in teeth_train_loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        teeth_combined_optimizer.zero_grad(set_to_none=True)

        outputs = teeth_combined_model(images)
        loss = teeth_combined_loss_fn(outputs, masks)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(teeth_combined_model.parameters(), max_norm=1.0)

        teeth_combined_optimizer.step()

        total_train_loss += loss.item() * images.size(0)

    train_loss = total_train_loss / len(teeth_train_loader.dataset)

    val_loss, val_mean_dice, val_dice = evaluate(teeth_combined_model, teeth_val_loader, teeth_combined_loss_fn, device, num_classes=5)

    current_dice = val_mean_dice.item()

    new_best = False

    if current_dice > best_teeth_combined_dice:
        best_teeth_combined_dice = current_dice
        epochs_without_improvement = 0
        torch.save(teeth_combined_model.state_dict(), best_teeth_combined_path)
        new_best = True
    else:
        epochs_without_improvement += 1

    teeth_combined_scheduler.step(current_dice)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Mean Dice: {current_dice:.4f} | "
        f"UR: {val_dice[0].item():.4f} | "
        f"UL: {val_dice[1].item():.4f} | "
        f"LL: {val_dice[2].item():.4f} | "
        f"LR: {val_dice[3].item():.4f}"
        + (" NEW BEST!" if new_best else "")
    )

    if epochs_without_improvement >= early_stop_patience:
        print("Early stopping triggered!")
        break

print()
print("TEETH COMBINED MODEL TRAINING COMPLETE")
print("Best Validation Mean Dice:", f"{best_teeth_combined_dice:.4f}")


teeth_combined_model.load_state_dict(torch.load(best_teeth_combined_path, map_location=device))

test_loss, test_mean_dice, test_dice = evaluate(teeth_combined_model, teeth_test_loader, teeth_combined_loss_fn, device, num_classes=5)

print()
print("TEETH COMBINED MODEL FINAL TEST RESULTS")
print(f"Upper Right Dice: {test_dice[0].item():.4f} ({test_dice[0].item() * 100:.2f}%)")
print(f"Upper Left Dice: {test_dice[1].item():.4f} ({test_dice[1].item() * 100:.2f}%)")
print(f"Lower Left Dice: {test_dice[2].item():.4f} ({test_dice[2].item() * 100:.2f}%)")
print(f"Lower Right Dice: {test_dice[3].item():.4f} ({test_dice[3].item() * 100:.2f}%)")
print(f"Mean Dice: {test_mean_dice.item():.4f} ({test_mean_dice.item() * 100:.2f}%)")


# ============================================================
# BACKUP IMMEDIATELY
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil

backup_folder = "/content/drive/MyDrive/model_checkpoints_backup"
os.makedirs(backup_folder, exist_ok=True)
shutil.copy(best_teeth_combined_path, os.path.join(backup_folder, "best_teeth_combined_unet.pth"))

print("Backed up to Google Drive!")

# ALL METRIC FUNCTIONS

In [ ]:


from scipy.ndimage import label, binary_erosion, distance_transform_edt
from sklearn.metrics import roc_auc_score, roc_curve
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt


def keep_largest_component(binary_mask):
    labeled_array, num_features = label(binary_mask)
    if num_features == 0:
        return binary_mask
    component_sizes = np.bincount(labeled_array.ravel())
    component_sizes[0] = 0
    largest_component_label = component_sizes.argmax()
    return labeled_array == largest_component_label


def evaluate_iou_with_postprocessing(model, loader, device, num_classes=5):

    model.eval()
    intersections = torch.zeros(num_classes - 1, device=device)
    unions = torch.zeros(num_classes - 1, device=device)

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions_np = predictions.cpu().numpy()
            masks_np = masks.cpu().numpy()

            for batch_idx in range(predictions_np.shape[0]):
                for class_id in range(1, num_classes):
                    pred_class = (predictions_np[batch_idx] == class_id)
                    target_class = (masks_np[batch_idx] == class_id)
                    pred_class_cleaned = keep_largest_component(pred_class)
                    intersection = np.logical_and(pred_class_cleaned, target_class).sum()
                    union = np.logical_or(pred_class_cleaned, target_class).sum()
                    intersections[class_id - 1] += intersection
                    unions[class_id - 1] += union

    iou_scores = (intersections + 1e-6) / (unions + 1e-6)
    mean_iou = iou_scores.mean()
    return mean_iou, iou_scores


def hd95_single_class(pred_mask, true_mask):
    pred_mask = pred_mask.astype(bool)
    true_mask = true_mask.astype(bool)
    if not pred_mask.any() and not true_mask.any():
        return 0.0
    if not pred_mask.any() or not true_mask.any():
        return np.nan
    pred_boundary = pred_mask ^ binary_erosion(pred_mask)
    true_boundary = true_mask ^ binary_erosion(true_mask)
    distance_to_true = distance_transform_edt(~true_boundary)
    distance_to_pred = distance_transform_edt(~pred_boundary)
    pred_to_true = distance_to_true[pred_boundary]
    true_to_pred = distance_to_pred[true_boundary]
    all_distances = np.concatenate([pred_to_true, true_to_pred])
    return np.percentile(all_distances, 95)


def evaluate_hd95(model, loader, device, num_classes=5):

    model.eval()
    class_hd95 = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx]
                true = masks_cpu[batch_idx]
                for class_id in range(1, num_classes):
                    pred_class = keep_largest_component(pred == class_id)
                    true_class = (true == class_id)
                    hd = hd95_single_class(pred_class, true_class)
                    if not np.isnan(hd):
                        class_hd95[class_id].append(hd)

    mean_hd95_per_class = {
        class_id: np.mean(values) if len(values) > 0 else np.nan
        for class_id, values in class_hd95.items()
    }
    overall_mean_hd95 = np.mean([v for v in mean_hd95_per_class.values() if not np.isnan(v)])
    return overall_mean_hd95, mean_hd95_per_class


def evaluate_roc_auc(model, loader, device, num_classes=5):

    model.eval()
    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()
                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    auc_scores = {}
    for class_id in range(1, num_classes):
        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])
        auc_scores[class_id] = roc_auc_score(true_labels, probabilities)

    mean_auc = np.mean(list(auc_scores.values()))
    return mean_auc, auc_scores


def evaluate_ssim(model, loader, device, num_classes=5):

    model.eval()
    ssim_scores = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)
            predictions = predictions.cpu().numpy()
            masks_cpu = masks.cpu().numpy()

            for batch_idx in range(predictions.shape[0]):
                pred = predictions[batch_idx].astype(np.float32)
                true = masks_cpu[batch_idx].astype(np.float32)
                score = ssim(pred, true, data_range=num_classes - 1)
                ssim_scores.append(score)

    return np.mean(ssim_scores)


print("All metric functions ready!")


# ============================================================
# FULL METRICS - TEETH COMBINED MODEL
# ============================================================

mean_iou, iou_scores = evaluate_iou_with_postprocessing(teeth_combined_model, teeth_test_loader, device, num_classes=5)

print("TEETH COMBINED - IOU (with post-processing)")
print(f"UR: {iou_scores[0].item()*100:.2f}%, UL: {iou_scores[1].item()*100:.2f}%, LL: {iou_scores[2].item()*100:.2f}%, LR: {iou_scores[3].item()*100:.2f}%")
print(f"Mean IoU: {mean_iou.item()*100:.2f}%")
print()

overall_hd95, hd95_per_class = evaluate_hd95(teeth_combined_model, teeth_test_loader, device, num_classes=5)

print("TEETH COMBINED - HD95 (with post-processing, in pixels)")
print(f"UR: {hd95_per_class[1]:.2f}px, UL: {hd95_per_class[2]:.2f}px, LL: {hd95_per_class[3]:.2f}px, LR: {hd95_per_class[4]:.2f}px")
print(f"Mean HD95: {overall_hd95:.2f}px")
print()

mean_auc, auc_scores = evaluate_roc_auc(teeth_combined_model, teeth_test_loader, device, num_classes=5)

print("TEETH COMBINED - ROC-AUC")
print(f"UR: {auc_scores[1]*100:.2f}%, UL: {auc_scores[2]*100:.2f}%, LL: {auc_scores[3]*100:.2f}%, LR: {auc_scores[4]*100:.2f}%")
print(f"Mean AUC: {mean_auc*100:.2f}%")
print()

mean_ssim_value = evaluate_ssim(teeth_combined_model, teeth_test_loader, device, num_classes=5)

print("TEETH COMBINED - SSIM")
print(f"Mean SSIM: {mean_ssim_value:.4f}")

In [ ]:
# ============================================================
# ROC-AUC CURVE PLOT FUNCTION
# ============================================================

from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt


def plot_roc_curve(model, loader, device, num_classes=5, class_names=None, title="ROC Curve"):

    model.eval()

    if class_names is None:
        class_names = [f"Class {i}" for i in range(1, num_classes)]

    all_true_labels = {class_id: [] for class_id in range(1, num_classes)}
    all_probabilities = {class_id: [] for class_id in range(1, num_classes)}

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = torch.softmax(outputs, dim=1)
            for class_id in range(1, num_classes):
                class_probs = probabilities[:, class_id].cpu().numpy().ravel()
                class_true = (masks == class_id).cpu().numpy().ravel()
                all_true_labels[class_id].append(class_true)
                all_probabilities[class_id].append(class_probs)

    plt.figure(figsize=(7, 6))

    for i, class_id in enumerate(range(1, num_classes)):
        true_labels = np.concatenate(all_true_labels[class_id])
        probabilities = np.concatenate(all_probabilities[class_id])
        fpr, tpr, _ = roc_curve(true_labels, probabilities)
        auc_value = roc_auc_score(true_labels, probabilities)
        plt.plot(fpr, tpr, linewidth=2, label=f"{class_names[i]} (AUC = {auc_value:.4f})")

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1, label="Random guess")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_roc_curve(
    teeth_combined_model,
    teeth_test_loader,
    device,
    num_classes=5,
    class_names=["Upper Right", "Upper Left", "Lower Left", "Lower Right"],
    title="Teeth Quadrant Combined GNN+Transformer - ROC Curve"
)

In [ ]:
# ============================================================
# VISUALIZATION FUNCTION
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


def compute_single_image_dice(prediction, target, num_classes=5):
    dice_scores = []
    for class_id in range(1, num_classes):
        pred_class = (prediction == class_id)
        target_class = (target == class_id)
        intersection = np.logical_and(pred_class, target_class).sum()
        denom = pred_class.sum() + target_class.sum()
        if denom == 0:
            continue
        dice = (2.0 * intersection) / denom
        dice_scores.append(dice)
    if len(dice_scores) == 0:
        return 0.0
    return np.mean(dice_scores)


def visualize_predictions(model, dataset, device, num_images=6, num_classes=5, class_colors=None, class_names=None, dataset_name="Dataset"):

    model.eval()

    if class_colors is None:
        class_colors = {1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1], 4: [1, 1, 0]}
    if class_names is None:
        class_names = {1: "Class1", 2: "Class2", 3: "Class3", 4: "Class4"}

    num_images = min(num_images, len(dataset))

    fig, axes = plt.subplots(num_images, 3, figsize=(12, num_images * 4))
    if num_images == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():

        for idx in range(num_images):

            image, true_mask = dataset[idx]
            input_image = image.unsqueeze(0).to(device)
            output = model(input_image)
            prediction = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()
            true_mask_np = true_mask.numpy()
            dice = compute_single_image_dice(prediction, true_mask_np, num_classes)

            display_image = image[0].cpu().numpy()
            display_image = (display_image - display_image.min()) / (display_image.max() - display_image.min() + 1e-8)

            gt_overlay = np.stack([display_image] * 3, axis=-1)
            pred_overlay = np.stack([display_image] * 3, axis=-1)

            for class_id, color in class_colors.items():
                gt_mask = (true_mask_np == class_id)
                pred_mask = (prediction == class_id)
                for c in range(3):
                    gt_overlay[..., c] = np.where(gt_mask, color[c], gt_overlay[..., c])
                    pred_overlay[..., c] = np.where(pred_mask, color[c], pred_overlay[..., c])

            axes[idx, 0].imshow(display_image, cmap="gray")
            axes[idx, 0].set_title("Input")
            axes[idx, 0].axis("off")

            axes[idx, 1].imshow(gt_overlay)
            axes[idx, 1].set_title("Ground Truth")
            axes[idx, 1].axis("off")

            axes[idx, 2].imshow(pred_overlay)
            axes[idx, 2].set_title(f"Prediction (Dice: {dice:.3f})", fontweight="bold")
            axes[idx, 2].axis("off")

    legend_patches = [
        mpatches.Patch(color=color, label=class_names[class_id])
        for class_id, color in class_colors.items()
    ]
    fig.legend(handles=legend_patches, loc="upper center", ncol=len(legend_patches), fontsize=11, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(f"{dataset_name} - Predictions with Per-Image Dice Score", fontsize=13, y=1.05)

    plt.tight_layout()
    plt.show()


print("Visualization function ready!")


# ============================================================
# VISUALIZE TEETH COMBINED MODEL
# ============================================================

visualize_predictions(
    teeth_combined_model,
    teeth_test_dataset,
    device,
    num_images=6,
    num_classes=5,
    class_colors={1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1], 4: [1, 1, 0]},
    class_names={1: "Upper Right", 2: "Upper Left", 3: "Lower Left", 4: "Lower Right"},
    dataset_name="Teeth Quadrant Combined GNN+Transformer"
)